# Keyed fragile watermarking for medical image tamper detection
### The synthesis study, ISIC 2019 run

This notebook assembles the best published finding for each sub-problem of
tamper detection into one system, and measures the assembly against every
constituent on a **protocol frozen before any training**.

| row | sub-problem | best answer | source |
|---|---|---|---|
| 1 | where to spend embedding energy | attention-weighted strength | PFF-WM |
| 2 | what the verifier reads | **keyed** QIM residual, integer Haar | this work |
| 3 | is anything wrong | one-class AE, authentic-only | PFF-WM |
| 4 | error map to mask | RefineNet, 3 levels + skips | PFF-WM |
| 5 | output formulation | segmentation, not a scalar | WAM |
| 6 | survive routine processing | degradation conditioning | OmniGuard |
| 7 | benign vs malicious vs unknown | residual features to classifier | post-quantum DCT |
| 8 | when evidence is weak | **calibrated abstention + reason codes** | this work |

**What this notebook does NOT claim.** The published localisation bar — IoU
0.832 — was measured on **LiTS** with *lesion-insertion* tampering. This
notebook runs on **ISIC 2019** with a wider family set. Numbers from here are
therefore not directly comparable to that bar, and the receipt labels them
`protocol: isic2019-synthesis-v1`. Any table that puts them side by side must
carry that label.

## 1. Setup

The package below is byte-identical to the module that `pytest tests/` covers. It is embedded rather than reimplemented so the two cannot drift.

In [ ]:
import base64, os, sys, json, glob, time, hashlib, random
from pathlib import Path

import numpy as np
import torch

PKG_ROOT = "/kaggle/working/_keyswm" if os.path.isdir("/kaggle/working") else "/tmp/_keyswm"
os.makedirs(os.path.join(PKG_ROOT, "keyswm"), exist_ok=True)

FILES = {'protocol': 'IiIiVGhlIGZyb3plbiBwcm90b2NvbC4gU2luZ2xlIHNvdXJjZSBvZiB0cnV0aC4KCk5vdGhpbmcgaW4gdGhpcyBmaWxlIG1heSBjaGFuZ2UgYWZ0ZXIgdGhlIGZpcnN0IHRyYWluaW5nIHJ1bi4gQW55IGNoYW5nZSBpcyBhbgphbWVuZG1lbnQ6IGFwcGVuZGVkIHRvIGRvY3MvQU1FTkRNRU5UUy5tZCwgdGhlIHN1cGVyc2VkZWQgbnVtYmVyIHJldGlyZWQsIHRoZQpuZXcgbnVtYmVyIHJlcG9ydGVkIHNlcGFyYXRlbHkuIFRoaXMgaXMgdGhlIHNhbWUgZGlzY2lwbGluZSB0aGF0IHByb2R1Y2VkIDM0CmFtZW5kbWVudHMgb24gdGhlIGNsYXNzaWNhbCBzeXN0ZW0sIGFwcGxpZWQgdG8gYSBsYXJnZXIgb25lLgoKV2h5IHRoaXMgZmlsZSBleGlzdHMgYXMgY29kZSBhbmQgbm90IGFzIHByb3NlOiBpbiB0aGUgY2xhc3NpY2FsIHBoYXNlLCBvbmUKZG9jdW1lbnQgY2xhaW1lZCAwLjk5OSBkZXRlY3Rpb24gd2hpbGUgYW5vdGhlciBjbGFpbWVkIDAuOTg3IGNvdmVyYWdlLCBhbmQgYQp0aGlyZCBjbGFpbWVkIGEgbGF0ZW5jeSBmcm9tIGEgYnJhbmNoIHRoYXQgbm8gbG9uZ2VyIGV4aXN0ZWQuIFRocmVlIGZpbGVzLCB0aHJlZQpzdG9yaWVzLCBvbmUgcmVwb3NpdG9yeS4gQSBwcm90b2NvbCB0aGF0IGlzIGltcG9ydGVkIGNhbm5vdCBkcmlmdC4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MsIGZpZWxkCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gY29ycG9yYQoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgQ29ycHVzOgogICAgbmFtZTogc3RyCiAgICByb2xlOiBzdHIgICAgICAgICAgICAjIHNlZW4gfCBiZW5jaG1hcmsgfCBzdWJzdGl0dXRlCiAgICBuX2ltYWdlczogaW50IHwgTm9uZQogICAgbm90ZTogc3RyCgoKQ09SUE9SQSA9ICgKICAgIENvcnB1cygiaXNpYzIwMTkiLCAicHJpbWFyeSIsIDI1MzMxLAogICAgICAgICAgICJLYWdnbGUsIGZyZWUuIFRIRSBjb3JwdXMgZm9yIHRoaXMgcGhhc2U6IHRyYWluLCB2YWxpZGF0ZSBhbmQgdGVzdC4iKSwKICAgIENvcnB1cygibGl0cyIsICJkZWZlcnJlZCIsIDEzMSwKICAgICAgICAgICAiREVGRVJSRUQgMjAyNi0xMC0yNC4gUEZGLVdNIG1lYXN1cmVkIDAuODMyIElvVSBoZXJlLiBUaGlzIGlzIHRoZSBvbmx5ICIKICAgICAgICAgICAiY29ycHVzIG9uIHdoaWNoIGEgbG9jYWxpc2F0aW9uIG51bWJlciBjYW4gc2l0IGJlc2lkZSB0aGVpcnMuIiksCiAgICBDb3JwdXMoImNoZXhwZXJ0IiwgImRlZmVycmVkIiwgTm9uZSwKICAgICAgICAgICAiREVGRVJSRUQgMjAyNi0xMC0yNCBieSBkZWNpc2lvbi4gUmVxdWlyZXMgYSBTdGFuZm9yZCByZXNlYXJjaC11c2UgZm9ybS4gIgogICAgICAgICAgICJQRkYtV00ncyA5Ny42JSBkZXRlY3Rpb24gaXMgcmVwb3J0ZWQgaGVyZSwgc28gd2l0aG91dCBpdCBubyBkZXRlY3Rpb24gIgogICAgICAgICAgICJudW1iZXIgb2Ygb3VycyBjYW4gYmUgdGFidWxhdGVkIGFnYWluc3QgdGhlaXJzLiIpLAogICAgQ29ycHVzKCJuaWhfY2hlc3R4cmF5MTQiLCAiZGVmZXJyZWQiLCAxMTIxMjAsCiAgICAgICAgICAgIkRFRkVSUkVELiBGcmVlIHN1YnN0aXR1dGUgZm9yIENoZVhwZXJ0LiBJZiBldmVyIHVzZWQsIG11c3QgYmUgbGFiZWxsZWQgIgogICAgICAgICAgICJhcyBhIHN1YnN0aXR1dGUgYW5kIG5ldmVyIGNhbGxlZCBDaGVYcGVydC4iKSwKKQoKIyBXaGF0IHRoZSBkZWZlcnJhbCBjb3N0cywgc3RhdGVkIG9uY2Ugc28gaXQgY2Fubm90IGJlIGRpc2NvdmVyZWQgbGF0ZXIgYXMgYQojIHN1cnByaXNlLiBUaGlzIGlzIGEgc2NvcGUgZGVjaXNpb24sIG5vdCBhIGRlZmVjdDogSVNJQyAyMDE5IGlzIG9uZSBvZiB0aGUKIyB0aHJlZSBjb3Jwb3JhIFBGRi1XTSBldmFsdWF0ZXMgb24sIHNvIHJ1bm5pbmcgb24gaXQgaXMgbGVnaXRpbWF0ZS4gV2hhdCBpdAojIGRvZXMgbm90IGJ1eSBpcyBhIHJvdyBpbiB0aGVpciBoZWFkbGluZSB0YWJsZXMuCkRFRkVSUkFMX05PVEUgPSAoCiAgICAiSVNJQyAyMDE5IGlzIG9uZSBvZiB0aGUgdGhyZWUgY29ycG9yYSBQRkYtV00gZXZhbHVhdGVzIG9uLCBzbyBhIHJ1biBoZXJlIGlzICIKICAgICJvbiB0aGVpciBncm91bmQuIEJ1dCB0aGVpciBUYWJsZSAyIChkZXRlY3Rpb24pIGlzIENoZVhwZXJ0IGFuZCB0aGVpciAiCiAgICAiVGFibGUgNCAobG9jYWxpc2F0aW9uIElvVSkgaXMgTGlUUy4gTnVtYmVycyBmcm9tIGFuIElTSUMtb25seSBwaGFzZSBjYW5ub3QgIgogICAgImJlIHBsYWNlZCBpbiBlaXRoZXIgdGFibGU7IHRoZXkgY2FuIG9ubHkgYmUgcmVwb3J0ZWQgYXMgdGhpcyB3b3JrJ3Mgb3duICIKICAgICJwcm90b2NvbCBvbiBhIGNvcnB1cyBib3RoIHN5c3RlbXMgY2FuIHJ1biBvbi4iCikKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBzcGxpdHMKCkBkYXRhY2xhc3MoZnJvemVuPVRydWUpCmNsYXNzIFNwbGl0OgogICAgdHJhaW46IGZsb2F0CiAgICB2YWw6IGZsb2F0CiAgICB0ZXN0OiBmbG9hdAogICAgZ3JvdXBlZF9ieV9wYXRpZW50OiBib29sCiAgICBub3RlOiBzdHIKCgpTUExJVCA9IFNwbGl0KAogICAgdHJhaW49MC43MCwgdmFsPTAuMTUsIHRlc3Q9MC4xNSwKICAgIGdyb3VwZWRfYnlfcGF0aWVudD1UcnVlLAogICAgbm90ZT0oIkdyb3VwIGJ5IHBhdGllbnQgd2hlcmUgdGhlIGNvcnB1cyBjYXJyaWVzIHBhdGllbnQgaWRzLiBMaVRTIHZvbHVtZXMgIgogICAgICAgICAgImFyZSBvbmUgcGF0aWVudCBlYWNoOiBhIHNsaWNlIG5ldmVyIGFwcGVhcnMgaW4gdHdvIHNwbGl0cy4gVGVzdCBpcyAiCiAgICAgICAgICAib3BlbmVkIG9uY2UsIGFmdGVyIG1vZGVsIHNlbGVjdGlvbiBvbiB2YWwuIiksCikKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSB0YW1wZXIgZmFtaWxpZXMKCkBkYXRhY2xhc3MoZnJvemVuPVRydWUpCmNsYXNzIEZhbWlseToKICAgIGZpZDogc3RyCiAgICBuYW1lOiBzdHIKICAgIGtpbmQ6IHN0ciAgICAgICAgICAgICMgdGFtcGVyIHwgYmVuaWduCiAgICBwYXJhbXM6IHR1cGxlCiAgICBleHBlY3RlZDogc3RyICAgICAgICAjIHdoYXQgYSBjb3JyZWN0IHN5c3RlbSBkb2VzCiAgICBub3RlOiBzdHIgPSAiIgoKClRBTVBFUl9GQU1JTElFUyA9ICgKICAgIEZhbWlseSgiVDEiLCAiY29weS1tb3ZlIiwgInRhbXBlciIsICg4LCAxNiwgMzIsIDY0KSwKICAgICAgICAgICAiZmxhZyB0aGUgcGFzdGVkIGJsb2NrcyIsICJjYXJyaWVkIG92ZXIgZnJvbSB0aGUgY2xhc3NpY2FsIGJhdHRlcnkiKSwKICAgIEZhbWlseSgiVDIiLCAibGVzaW9uIGluc2VydGlvbiIsICJ0YW1wZXIiLCAoInN5bnRoZXRpYyIsKSwKICAgICAgICAgICAiZmxhZyB0aGUgaW5zZXJ0ZWQgbWFzcyIsCiAgICAgICAgICAgInRoZSBQRkYtV00gTGlUUyBwcm90b2NvbDogc3ludGhldGljIHR1bW91ciBwYXN0ZWQgaW50byBsaXZlciBwYXJlbmNoeW1hIiksCiAgICBGYW1pbHkoIlQzIiwgImxlc2lvbiByZW1vdmFsIiwgInRhbXBlciIsICgidGVsZWEiLCksCiAgICAgICAgICAgImZsYWcgdGhlIGlucGFpbnRlZCByZWdpb24iLCAiY2xhc3NpY2FsIGNvbmRpdGlvbiAyMiIpLAogICAgRmFtaWx5KCJUNCIsICJidXJuZWQtaW4gdGV4dCIsICJ0YW1wZXIiLCAoIjY0eDI0IiwpLAogICAgICAgICAgICJmbGFnIHRoZSB0ZXh0IGJveCIsICJjbGFzc2ljYWwgY29uZGl0aW9uIDIxOyBvdXIgd2Vha2VzdCBmYW1pbHkiKSwKICAgIEZhbWlseSgiVDUiLCAic3BsaWNpbmcgKGNyb3NzLWltYWdlKSIsICJ0YW1wZXIiLCAoImZvcmVpZ24iLCksCiAgICAgICAgICAgImZsYWcgdGhlIHNwbGljZWQgcmVnaW9uIiwKICAgICAgICAgICAidGhlIGdhcCB0aGUgY2xhc3NpY2FsIHBoYXNlIG5ldmVyIG1lYXN1cmVkOyBzZWUgZG9jcywgbGltaXRhdGlvbiA1LjMiKSwKICAgIEZhbWlseSgiVDYiLCAiZ2VuZXJhdGl2ZSBzdWJzdGl0dXRpb24iLCAidGFtcGVyIiwgKCJkZWNsYXJlZC1vdXQtb2Ytc2NvcGUiLCksCiAgICAgICAgICAgImZsYWcgdGhlIHN1YnN0aXR1dGVkIHJlZ2lvbiIsCiAgICAgICAgICAgIk5PVCBmYWtlZC4gUmVwb3J0ZWQgYXMgdW5tZWFzdXJlZCB1bmxlc3MgYSBnZW5lcmF0b3IgaXMgYXZhaWxhYmxlLiIpLAogICAgRmFtaWx5KCJCMSIsICJiZW5pZ24gc2luZ2xlIG9wIiwgImJlbmlnbiIsICgianBlZzkwIiwgImpwZWc3NSIsICJqcGVnNTAiLCAic2lnbWExIiwgInJlc2l6ZTkwIiksCiAgICAgICAgICAgIm5vIHRhbXBlciBjbGFpbSIpLAogICAgRmFtaWx5KCJCMiIsICJiZW5pZ24gY2hhaW4iLCAiYmVuaWduIiwgKCJzY2FsZS0+anBlZy0+c2NhbGUtPmpwZWciLCksCiAgICAgICAgICAgImFic3RhaW4sIG5ldmVyIGFjY3VzZSIpLAopCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gbWV0cmljcwoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgTWV0cmljU3BlYzoKICAgIG5hbWU6IHN0cgogICAgZGlyZWN0aW9uOiBzdHIgICAgICAgIyBoaWdoZXIgfCBsb3dlcgogICAgZGVmaW5pdGlvbjogc3RyCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIHN0cmF0YQoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgU3RyYXR1bToKICAgIG5hbWU6IHN0cgogICAgc291cmNlOiBzdHIKICAgIG5vdGU6IHN0cgoKClNUUkFUQSA9ICgKICAgIFN0cmF0dW0oImRpYWdub3NpcyIsICJpc2ljMjAxOSBtZXRhZGF0YSIsCiAgICAgICAgICAgICJJU0lDIDIwMTkncyBlaWdodCBkaWFnbm9zdGljIGNhdGVnb3JpZXMuIExvY2FsaXNhdGlvbiBjYW4gZmFpbCAiCiAgICAgICAgICAgICJkaWZmZXJlbnRseSBvbiBmbGF0IGxlc2lvbnMgKE5WLCBCS0wpIHRoYW4gb24gdGV4dHVyZWQgb25lcyAoQkNDLCAiCiAgICAgICAgICAgICJTQ0MpLiBSZXBvcnQgZXZlcnkgYnVja2V0IHdpdGggaXRzIG47IGFuIGVtcHR5IGJ1Y2tldCBpcyByZXBvcnRlZCAiCiAgICAgICAgICAgICJhcyBlbXB0eSwgbmV2ZXIgb21pdHRlZC4iKSwKICAgIFN0cmF0dW0oInRhbXBlcl9hcmVhIiwgIm1lYXN1cmVkIGZyb20gdGhlIHBpeGVsIG1hc2siLAogICAgICAgICAgICAiSW9VIGlzIHN0cm9uZ2x5IGFyZWEtZGVwZW5kZW50LCBhbmQgYSBwZXJmZWN0IGJsb2NrIGRldGVjdG9yIHRvcHMgIgogICAgICAgICAgICAib3V0IGF0IDAuODA5IG9uIGEgcmFkaXVzLTMyIGxlc2lvbi4gRXZlcnkgSW9VIGlzIHJlcG9ydGVkIHdpdGggdGhlICIKICAgICAgICAgICAgImNlaWxpbmcgZm9yIGl0cyBidWNrZXQuIiksCiAgICBTdHJhdHVtKCJmYW1pbHkiLCAidGhlIHByb3RvY29sIiwKICAgICAgICAgICAgIkVpZ2h0IGZhbWlsaWVzLiBBIG1lYW4gb3ZlciB0aGVtIGlzIGRvbWluYXRlZCBieSB3aGljaGV2ZXIgaXMgbW9zdCAiCiAgICAgICAgICAgICJudW1lcm91cywgc28gdGhlIGZhbWlseSB0YWJsZSBpcyBhbHdheXMgcHJpbnRlZCBpbiBmdWxsLiIpLAogICAgU3RyYXR1bSgiY292ZXJhZ2UiLCAidGhlIGFic3RlbnRpb24gcnVsZSIsCiAgICAgICAgICAgICJTZWxlY3RpdmUgcmlzayBhZ2FpbnN0IGNvdmVyYWdlLiBBIHN5c3RlbSB0aGF0IGFuc3dlcnMgbGVzcyBtdXN0IGJlICIKICAgICAgICAgICAgInNob3duIHRvIGFuc3dlciBiZXR0ZXIsIG9yIHRoZSBhYnN0ZW50aW9uIGlzIGhpZGluZyBhIGZhaWx1cmUuIiksCikKCgpNRVRSSUNTID0gKAogICAgTWV0cmljU3BlYygiaW91IiwgImhpZ2hlciIsICJwZXItaW1hZ2UgSW9VIGJldHdlZW4gcHJlZGljdGVkIGFuZCBtZWFzdXJlZCB0YW1wZXIgbWFzayIpLAogICAgTWV0cmljU3BlYygiZGljZSIsICJoaWdoZXIiLCAiaGFybW9uaWMgbWVhbiBvZiBwcmVjaXNpb24gYW5kIHJlY2FsbCBvbiBwaXhlbHMiKSwKICAgIE1ldHJpY1NwZWMoInBpeGVsX2FjYyIsICJoaWdoZXIiLCAiZnJhY3Rpb24gb2YgYmxvY2tzIGNsYXNzaWZpZWQgY29ycmVjdGx5IiksCiAgICBNZXRyaWNTcGVjKCJwaXhlbF9mMSIsICJoaWdoZXIiLCAiRjEgb3ZlciB0YW1wZXJlZCBibG9ja3MiKSwKICAgIE1ldHJpY1NwZWMoImRldGVjdGlvbl9hY2MiLCAiaGlnaGVyIiwgImZyYWN0aW9uIG9mIGltYWdlcyB3aGVyZSB0aGUgdGFtcGVyIHZlcmRpY3QgbWF0Y2hlcyBncm91bmQgdHJ1dGgiKSwKICAgIE1ldHJpY1NwZWMoImJlbmVmaXRfYXRfMC41IiwgImhpZ2hlciIsICJkZXRlY3Rpb24gcmF0ZSBhdCBJb1UgdGhyZXNob2xkIDAuNSIpLAogICAgTWV0cmljU3BlYygiYmxvY2tfZjEiLCAiaGlnaGVyIiwgIkYxIGF0IHRoZSA4eDggdmVyZGljdCBncmlkIiksCiAgICBNZXRyaWNTcGVjKCJzZWxlY3RpdmVfcmlzayIsICJsb3dlciIsICJlcnJvciByYXRlIGFtb25nIGFuc3dlcmVkIGJsb2NrcyBhcyBjb3ZlcmFnZSBmYWxscyIpLAogICAgTWV0cmljU3BlYygibGF0ZW5jeV9tcyIsICJsb3dlciIsICJ3YWxsIGNsb2NrIHBlciA1MTJ4NTEyIGltYWdlLCBpc29sYXRlZCIpLAogICAgTWV0cmljU3BlYygicGFyYW1zX20iLCAibG93ZXIiLCAidHJhaW5hYmxlIHBhcmFtZXRlcnMsIG1pbGxpb25zIiksCikKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBiYXJzCgpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQpjbGFzcyBCYXI6CiAgICBuYW1lOiBzdHIKICAgIHZhbHVlOiBmbG9hdAogICAgdW5pdDogc3RyCiAgICBzb3VyY2U6IHN0cgogICAgY29ycHVzOiBzdHIKCgpCQVJTID0gKAogICAgQmFyKCJkZXRlY3Rpb24iLCA5Ny42LCAiJSIsICJQRkYtV00gVGFibGUgMiIsICJjaGV4cGVydCIpLAogICAgQmFyKCJpb3UiLCAwLjgzMiwgIklvVSIsICJQRkYtV00gVGFibGUgNCAoUmVmaW5lTmV0KSIsICJsaXRzIiksCiAgICBCYXIoImZhbHNlX3Bvc2l0aXZlIiwgMC41LCAiJSIsICJQRkYtV00sIGJlbmlnbiB0cmFuc2Zvcm1hdGlvbnMiLCAiYW55IiksCiAgICBCYXIoImxhdGVuY3kiLCA0NC4wLCAibXMiLCAiUEZGLVdNIFRhYmxlIDYiLCAiNTEyeDUxMiIpLAogICAgQmFyKCJwYXJhbXMiLCAxNS4xLCAiTSIsICJQRkYtV00gVGFibGUgNiIsICJmdWxsIG1vZGVsIiksCikKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBjb25maWd1cmF0aW9ucwoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgQ29uZmlnOgogICAgY2lkOiBzdHIKICAgIG5hbWU6IHN0cgogICAgY29tcG9uZW50czogdHVwbGUKICAgIGlzb2xhdGVzOiBzdHIKICAgIHJlcXVpcmVkOiBib29sID0gVHJ1ZQoKCk1BVFJJWCA9ICgKICAgIENvbmZpZygiQzAiLCAiY2xhc3NpY2FsIG9ubHkiLCAoKSwgInRoZSBjb250cm9sIOKAlCBvdXIgZXhpc3Rpbmcgc3lzdGVtIiwgVHJ1ZSksCiAgICBDb25maWcoIkMxIiwgIisgb25lLWNsYXNzIEFFIiwgKCJhZSIsKSwgIlBGRi1XTSByb3cgMyIsIFRydWUpLAogICAgQ29uZmlnKCJDMiIsICIrIFJlZmluZU5ldCIsICgicmVmaW5lIiwpLCAiUEZGLVdNIHJvdyA0IiwgVHJ1ZSksCiAgICBDb25maWcoIkMzIiwgIisgQUUgKyBSZWZpbmVOZXQiLCAoImFlIiwgInJlZmluZSIpLCAiUEZGLVdNIHJvd3MgMys0LCBubyBhdHRlbnRpb24iLCBUcnVlKSwKICAgIENvbmZpZygiQzQiLCAiKyBhdHRlbnRpb24gKGZ1bGwgUEZGLVdNKSIsICgiYWUiLCAicmVmaW5lIiwgImF0dG4iKSwgIlBGRi1XTSByb3cgMSwgZnVsbCByZXByb2R1Y3Rpb24iLCBUcnVlKSwKICAgIENvbmZpZygiQzUiLCAiKyBkZWdyYWRhdGlvbiBjb25kaXRpb25pbmciLCAoImFlIiwgInJlZmluZSIsICJhdHRuIiwgImRlZ3JhZCIpLCAiT21uaUd1YXJkIHJvdyA2IiwgVHJ1ZSksCiAgICBDb25maWcoIkM2IiwgIisgc2VnbWVudGF0aW9uIGZvcm11bGF0aW9uIiwgKCJhZSIsICJyZWZpbmUiLCAiYXR0biIsICJkZWdyYWQiLCAic2VnIiksICJXQU0gcm93IDUiLCBUcnVlKSwKICAgIENvbmZpZygiQzciLCAiKyB0aHJlZS13YXkgaGVhZCIsICgiYWUiLCAicmVmaW5lIiwgImF0dG4iLCAiZGVncmFkIiwgInNlZyIsICJ0aHJlZSIpLCAicm93IDciLCBUcnVlKSwKICAgIENvbmZpZygiQzgiLCAiZnVsbCBzeW50aGVzaXMiLCAoImFlIiwgInJlZmluZSIsICJhdHRuIiwgImRlZ3JhZCIsICJzZWciLCAidGhyZWUiLCAia2V5ZWQiKSwKICAgICAgICAgICAiYWxsIGVpZ2h0IHJvd3MiLCBUcnVlKSwKICAgIENvbmZpZygiQzkiLCAiQzggbWludXMgdGhlIGtleSIsICgiYWUiLCAicmVmaW5lIiwgImF0dG4iLCAiZGVncmFkIiwgInNlZyIsICJ0aHJlZSIpLAogICAgICAgICAgICJ3aGF0IHNlY3VyaXR5IGNvc3RzIOKAlCByb3cgMiIsIFRydWUpLAogICAgQ29uZmlnKCJDMTAiLCAiQzggdHJhaW5lZCBLMSwgdGVzdGVkIEsyIiwgKCJyZWtleSIsKSwKICAgICAgICAgICAia2V5LWxlYWthZ2UgdGVzdCDigJQgdGhlIG9uZSBub2JvZHkgcnVucyIsIFRydWUpLAogICAgQ29uZmlnKCJDMTEiLCAiQzggb24gaGVsZC1vdXQgdGFtcGVyIGZhbWlseSIsICgiaGVsZG91dF9mYW1pbHkiLCksCiAgICAgICAgICAgIm5lZ2F0aXZlIHRyYW5zZmVyIiwgVHJ1ZSksCikKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBnYXRlcwoKR0FURVMgPSB7CiAgICAiY292ZXJhZ2VfZmxvb3IiOiAwLjkwLAogICAgIl9ub3RlIjogKCJBIGNvbmZpZ3VyYXRpb24gd2hvc2UgY2xhc3NpY2FsIGxheWVyIHJlZnVzZXMgbW9yZSB0aGFuIDEwJSBvZiAiCiAgICAgICAgICAgICAgImJsb2NrcyBpcyBkaXNxdWFsaWZpZWQsIHdoYXRldmVyIGl0cyBJb1UuIEl0IGhhcyBub3QgbG9jYWxpc2VkOyAiCiAgICAgICAgICAgICAgIml0IGhhcyBhYnN0YWluZWQgaXRzIHdheSB0byBhIHNtYWxsIGRlbm9taW5hdG9yLiIpLAogICAgInJla2V5X21heF9kcm9wX2lvdSI6IDAuMDUsCiAgICAiX25vdGVfcmVrZXkiOiAoIlRyYWluaW5nIG9uIEsxIGFuZCB0ZXN0aW5nIG9uIEsyIG11c3QgY29zdCBsZXNzIHRoYW4gMC4wNSAiCiAgICAgICAgICAgICAgICAgICAgIklvVS4gQSBsYXJnZXIgZHJvcCBtZWFucyB0aGUgbmV0d29yayBsZWFybmVkIHRoZSBrZXksIHdoaWNoICIKICAgICAgICAgICAgICAgICAgICAiaXMgYSBzZWN1cml0eSBmYWlsdXJlLCBub3QgYW4gYWNjdXJhY3kgZmFpbHVyZS4iKSwKICAgICJ0ZXN0X29wZW5lZF9vbmNlIjogVHJ1ZSwKfQoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBzYW5pdHkKCmRlZiBhdWRpdCgpIC0+IGxpc3Rbc3RyXToKICAgICIiIlN0cnVjdHVyYWwgY2hlY2tzIHRoYXQgbXVzdCBwYXNzIGJlZm9yZSB0aGUgbWF0cml4IGlzIHJ1bi4iIiIKICAgIHByb2JsZW1zOiBsaXN0W3N0cl0gPSBbXQogICAgaWRzID0gW2MuY2lkIGZvciBjIGluIE1BVFJJWF0KICAgIGlmIGxlbihpZHMpICE9IGxlbihzZXQoaWRzKSk6CiAgICAgICAgcHJvYmxlbXMuYXBwZW5kKCJkdXBsaWNhdGUgY29uZmlndXJhdGlvbiBpZHMiKQogICAgZm9yIGMgaW4gTUFUUklYOgogICAgICAgIHVua25vd24gPSBzZXQoYy5jb21wb25lbnRzKSAtIHsiYWUiLCAicmVmaW5lIiwgImF0dG4iLCAiZGVncmFkIiwgInNlZyIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJ0aHJlZSIsICJrZXllZCIsICJyZWtleSIsICJoZWxkb3V0X2ZhbWlseSJ9CiAgICAgICAgaWYgdW5rbm93bjoKICAgICAgICAgICAgcHJvYmxlbXMuYXBwZW5kKGYie2MuY2lkfTogdW5rbm93biBjb21wb25lbnRzIHtzb3J0ZWQodW5rbm93bil9IikKICAgIGZhbSA9IFtmLmZpZCBmb3IgZiBpbiBUQU1QRVJfRkFNSUxJRVNdCiAgICBpZiBsZW4oZmFtKSAhPSBsZW4oc2V0KGZhbSkpOgogICAgICAgIHByb2JsZW1zLmFwcGVuZCgiZHVwbGljYXRlIGZhbWlseSBpZHMiKQogICAgaWYgYWJzKFNQTElULnRyYWluICsgU1BMSVQudmFsICsgU1BMSVQudGVzdCAtIDEuMCkgPiAxZS05OgogICAgICAgIHByb2JsZW1zLmFwcGVuZCgic3BsaXRzIGRvIG5vdCBzdW0gdG8gMSIpCiAgICByZXR1cm4gcHJvYmxlbXMKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgaXNzdWVzID0gYXVkaXQoKQogICAgcHJpbnQoZiJjb25maWd1cmF0aW9uczoge2xlbihNQVRSSVgpfSIpCiAgICBwcmludChmImZhbWlsaWVzOiAgICAgICB7bGVuKFRBTVBFUl9GQU1JTElFUyl9IikKICAgIHByaW50KGYibWV0cmljczogICAgICAgIHtsZW4oTUVUUklDUyl9IikKICAgIHByaW50KGYiYXVkaXQ6ICAgICAgICAgIHsnUEFTUycgaWYgbm90IGlzc3VlcyBlbHNlICdGQUlMJ30iKQogICAgZm9yIHAgaW4gaXNzdWVzOgogICAgICAgIHByaW50KCIgIC0iLCBwKQo=', 'metrics': 'IiIiTWV0cmljcy4gT25lIGltcGxlbWVudGF0aW9uLCB1c2VkIGJ5IGV2ZXJ5IGNvbmZpZ3VyYXRpb24gaW4gdGhlIG1hdHJpeC4KClR3byBydWxlcywgYm90aCBsZWFybmVkIHRoZSBoYXJkIHdheSBpbiB0aGUgY2xhc3NpY2FsIHBoYXNlOgoKMS4gKipHcm91bmQgdHJ1dGggaXMgbWVhc3VyZWQsIG5ldmVyIGRlY2xhcmVkLioqIEEgInRhbXBlcmVkIHJlY3RhbmdsZSIgaXMgbm90CiAgIGdyb3VuZCB0cnV0aC4gT24gUEgyLCBvbmUgb2YgY29uZGl0aW9uIDQncyB0aHJlZSBjbG9uZXMgbGFuZGVkIHdoaXRlCiAgIGJhY2tncm91bmQgb24gd2hpdGUgYmFja2dyb3VuZCBhbmQgNCBvZiA0OCBsYWJlbGxlZCBibG9ja3MgbmV2ZXIgY2hhbmdlZCBhCiAgIHBpeGVsLiBTY29yaW5nIGFnYWluc3QgdGhlIHJlY3RhbmdsZSBwZW5hbGlzZWQgdGhlIHZlcmlmaWVyIGZvciBjb3JyZWN0bHkKICAgcmVwb3J0aW5nIHRob3NlIGJsb2NrcyBJTlRBQ1QuIGBtZWFzdXJlZF9tYXNrYCBkaWZmcyB0aGUgcGl4ZWxzLgoKMi4gKipFdmVyeSBtZXRyaWMgcmV0dXJucyBhIGNvdW50IGFzIHdlbGwgYXMgYSByYXRpby4qKiBBbiBJb1Ugb2YgMC4wIGZyb20KICAgIjAgb2YgMCBibG9ja3MiIGFuZCBhbiBJb1Ugb2YgMC4wIGZyb20gIjAgb2YgNDAwIGJsb2NrcyIgYXJlIGRpZmZlcmVudCBmYWN0cy4KICAgUmF0aW9zIHdpdGhvdXQgZGVub21pbmF0b3JzIGFyZSBob3cgYSBzbWFsbCBkZW5vbWluYXRvciBiZWNvbWVzIGEgaGVhZGxpbmUuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IG51bXB5IGFzIG5wCgpCTE9DSyA9IDgKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gbWFza3MKCmRlZiBtZWFzdXJlZF9tYXNrKGJlZm9yZTogbnAubmRhcnJheSwgYWZ0ZXI6IG5wLm5kYXJyYXksIGJsb2NrOiBpbnQgPSBCTE9DSykgLT4gbnAubmRhcnJheToKICAgICIiIkdyb3VuZCB0cnV0aDogYmxvY2tzIHdob3NlIHBpeGVscyBhY3R1YWxseSBjaGFuZ2VkLgoKICAgIGBiZWZvcmVgL2BhZnRlcmAgYXJlIEh4V3gzIHVpbnQ4IGF0IHRoZSAqcmVjZWl2ZWQqIGdlb21ldHJ5LiBJZiB0aGUgZ2VvbWV0cnkKICAgIGNoYW5nZWQgKGNyb3AsIHNoaWZ0KSwgZXZlcnkgYmxvY2sgb2YgdGhlIHJlY2VpdmVkIGltYWdlIGlzIHVudmVyaWZpYWJsZQogICAgY29udGVudCBhbmQgdGhlIG1hc2sgY292ZXJzIGl0IGVudGlyZWx5LgogICAgIiIiCiAgICBiZWZvcmUgPSBucC5hc2FycmF5KGJlZm9yZSkKICAgIGFmdGVyID0gbnAuYXNhcnJheShhZnRlcikKICAgIGlmIGJlZm9yZS5zaGFwZSAhPSBhZnRlci5zaGFwZToKICAgICAgICBoLCB3ID0gYWZ0ZXIuc2hhcGVbMF0gLy8gYmxvY2ssIGFmdGVyLnNoYXBlWzFdIC8vIGJsb2NrCiAgICAgICAgcmV0dXJuIG5wLm9uZXMoKGgsIHcpLCBkdHlwZT1ib29sKQogICAgZGlmZiA9IChiZWZvcmUgIT0gYWZ0ZXIpLmFueShheGlzPTIpCiAgICBueSwgbnggPSBkaWZmLnNoYXBlWzBdIC8vIGJsb2NrLCBkaWZmLnNoYXBlWzFdIC8vIGJsb2NrCiAgICByZXR1cm4gZGlmZls6bnkgKiBibG9jaywgOm54ICogYmxvY2tdLnJlc2hhcGUobnksIGJsb2NrLCBueCwgYmxvY2spLmFueShheGlzPSgxLCAzKSkKCgpkZWYgdG9fYmxvY2tzKGJsb2NrX21hc2s6IG5wLm5kYXJyYXksIHNoYXBlOiB0dXBsZVtpbnQsIGludF0pIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJVcHNhbXBsZSBhIGJsb2NrIG1hc2sgdG8gcGl4ZWwgcmVzb2x1dGlvbi4iIiIKICAgIG55LCBueCA9IGJsb2NrX21hc2suc2hhcGUKICAgIHJldHVybiBucC5yZXBlYXQobnAucmVwZWF0KG5wLmFzYXJyYXkoYmxvY2tfbWFzaywgYm9vbCksIEJMT0NLLCAwKSwgQkxPQ0ssIDEpWwogICAgICAgIDogc2hhcGVbMF0sIDogc2hhcGVbMV1dCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIHByaW1pdGl2ZXMKCmRlZiBfY291bnRzKHByZWQ6IG5wLm5kYXJyYXksIGd0OiBucC5uZGFycmF5KSAtPiB0dXBsZVtpbnQsIGludCwgaW50XToKICAgIHAgPSBucC5hc2FycmF5KHByZWQsIGJvb2wpCiAgICBnID0gbnAuYXNhcnJheShndCwgYm9vbCkKICAgIGlmIHAuc2hhcGUgIT0gZy5zaGFwZToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYibWFzayBzaGFwZXMgZGlmZmVyOiB7cC5zaGFwZX0gdnMge2cuc2hhcGV9IikKICAgIHRwID0gaW50KChwICYgZykuc3VtKCkpCiAgICBmcCA9IGludCgocCAmIH5nKS5zdW0oKSkKICAgIGZuID0gaW50KCh+cCAmIGcpLnN1bSgpKQogICAgcmV0dXJuIHRwLCBmcCwgZm4KCgpkZWYgaW91KHByZWQ6IG5wLm5kYXJyYXksIGd0OiBucC5uZGFycmF5KSAtPiBmbG9hdDoKICAgICIiIkphY2NhcmQgaW5kZXguIFR3byBlbXB0eSBtYXNrcyBhZ3JlZSBwZXJmZWN0bHk6IElvVSA9IDEuMC4KCiAgICBUaGUgY29udmVudGlvbiBtYXR0ZXJzLiBUcmVhdGluZyBlbXB0eS12cy1lbXB0eSBhcyAwLjAgd291bGQgcHVuaXNoIGEKICAgIGNvcnJlY3QgYWJzdGVudGlvbjsgdHJlYXRpbmcgaXQgYXMgMS4wIHJld2FyZHMgYSBzeXN0ZW0gdGhhdCBwcmVkaWN0cwogICAgbm90aGluZy4gTmVpdGhlciBpcyByaWdodCBpbiBpc29sYXRpb24sIHNvIGNhbGxlcnMgbXVzdCByZWFkIGBjb3VudHNgIHRvby4KICAgICIiIgogICAgdHAsIGZwLCBmbiA9IF9jb3VudHMocHJlZCwgZ3QpCiAgICBkZW5vbSA9IHRwICsgZnAgKyBmbgogICAgcmV0dXJuIDEuMCBpZiBkZW5vbSA9PSAwIGVsc2UgdHAgLyBkZW5vbQoKCmRlZiBkaWNlKHByZWQ6IG5wLm5kYXJyYXksIGd0OiBucC5uZGFycmF5KSAtPiBmbG9hdDoKICAgIHRwLCBmcCwgZm4gPSBfY291bnRzKHByZWQsIGd0KQogICAgZGVub20gPSAyICogdHAgKyBmcCArIGZuCiAgICByZXR1cm4gMS4wIGlmIGRlbm9tID09IDAgZWxzZSAyICogdHAgLyBkZW5vbQoKCmRlZiBwcmVjaXNpb24ocHJlZDogbnAubmRhcnJheSwgZ3Q6IG5wLm5kYXJyYXkpIC0+IGZsb2F0OgogICAgdHAsIGZwLCBfID0gX2NvdW50cyhwcmVkLCBndCkKICAgIHJldHVybiAwLjAgaWYgdHAgKyBmcCA9PSAwIGVsc2UgdHAgLyAodHAgKyBmcCkKCgpkZWYgcmVjYWxsKHByZWQ6IG5wLm5kYXJyYXksIGd0OiBucC5uZGFycmF5KSAtPiBmbG9hdDoKICAgIHRwLCBfLCBmbiA9IF9jb3VudHMocHJlZCwgZ3QpCiAgICByZXR1cm4gMC4wIGlmIHRwICsgZm4gPT0gMCBlbHNlIHRwIC8gKHRwICsgZm4pCgoKZGVmIHBpeGVsX2YxKHByZWQ6IG5wLm5kYXJyYXksIGd0OiBucC5uZGFycmF5KSAtPiBmbG9hdDoKICAgIHByLCByYyA9IHByZWNpc2lvbihwcmVkLCBndCksIHJlY2FsbChwcmVkLCBndCkKICAgIHJldHVybiAwLjAgaWYgcHIgKyByYyA9PSAwIGVsc2UgMiAqIHByICogcmMgLyAocHIgKyByYykKCgpkZWYgcGl4ZWxfYWNjdXJhY3kocHJlZDogbnAubmRhcnJheSwgZ3Q6IG5wLm5kYXJyYXkpIC0+IGZsb2F0OgogICAgcCA9IG5wLmFzYXJyYXkocHJlZCwgYm9vbCkKICAgIGcgPSBucC5hc2FycmF5KGd0LCBib29sKQogICAgaWYgcC5zaXplID09IDA6CiAgICAgICAgcmV0dXJuIDEuMAogICAgcmV0dXJuIGZsb2F0KChwID09IGcpLm1lYW4oKSkKCgpkZWYgY291bnRzKHByZWQ6IG5wLm5kYXJyYXksIGd0OiBucC5uZGFycmF5KSAtPiBkaWN0W3N0ciwgaW50XToKICAgIHRwLCBmcCwgZm4gPSBfY291bnRzKHByZWQsIGd0KQogICAgcmV0dXJuIHsidHAiOiB0cCwgImZwIjogZnAsICJmbiI6IGZuLCAiZ3QiOiBpbnQobnAuYXNhcnJheShndCwgYm9vbCkuc3VtKCkpLAogICAgICAgICAgICAicHJlZCI6IGludChucC5hc2FycmF5KHByZWQsIGJvb2wpLnN1bSgpKX0KCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gaW1hZ2UgbGV2ZWwKCmRlZiBpbWFnZV9mbGFnZ2VkKHByZWQ6IG5wLm5kYXJyYXksIG1pbl9ibG9ja3M6IGludCA9IDEpIC0+IGJvb2w6CiAgICAiIiJEaWQgdGhlIHN5c3RlbSByYWlzZSBhIHRhbXBlciBjbGFpbSBhdCBhbGw/IiIiCiAgICByZXR1cm4gaW50KG5wLmFzYXJyYXkocHJlZCwgYm9vbCkuc3VtKCkpID49IG1pbl9ibG9ja3MKCgpkZWYgZGV0ZWN0aW9uX2FjY3VyYWN5KHJlc3VsdHM6IGxpc3RbZGljdF0pIC0+IGRpY3Rbc3RyLCBmbG9hdF06CiAgICAiIiJJbWFnZS1sZXZlbCBkZXRlY3Rpb24gb3ZlciBhIGxpc3Qgb2YgcGVyLWltYWdlIHJlc3VsdHMuCgogICAgRWFjaCByZXN1bHQ6IHsiZmFtaWx5Ijogc3RyLCAia2luZCI6ICJ0YW1wZXIifCJiZW5pZ24iLCAicHJlZCI6IG1hc2t9LgogICAgQSB0YW1wZXIgaW1hZ2UgaXMgZGV0ZWN0ZWQgaWYgYW55IGJsb2NrIGlzIGZsYWdnZWQuIEEgYmVuaWduIGltYWdlIGlzIGEKICAgIGZhbHNlIHBvc2l0aXZlIGlmIGFueSBibG9jayBpcyBmbGFnZ2VkLgogICAgIiIiCiAgICB0YW1wZXIgPSBbciBmb3IgciBpbiByZXN1bHRzIGlmIHJbImtpbmQiXSA9PSAidGFtcGVyIl0KICAgIGJlbmlnbiA9IFtyIGZvciByIGluIHJlc3VsdHMgaWYgclsia2luZCJdID09ICJiZW5pZ24iXQogICAgZGV0ID0gc3VtKGltYWdlX2ZsYWdnZWQoclsicHJlZCJdKSBmb3IgciBpbiB0YW1wZXIpIC8gbWF4KGxlbih0YW1wZXIpLCAxKQogICAgZnByID0gc3VtKGltYWdlX2ZsYWdnZWQoclsicHJlZCJdKSBmb3IgciBpbiBiZW5pZ24pIC8gbWF4KGxlbihiZW5pZ24pLCAxKQogICAgYWNjID0gKHN1bShpbWFnZV9mbGFnZ2VkKHJbInByZWQiXSkgZm9yIHIgaW4gdGFtcGVyKQogICAgICAgICAgICsgc3VtKG5vdCBpbWFnZV9mbGFnZ2VkKHJbInByZWQiXSkgZm9yIHIgaW4gYmVuaWduKSkgLyBtYXgobGVuKHJlc3VsdHMpLCAxKQogICAgcmV0dXJuIHsiZGV0ZWN0aW9uX2FjYyI6IGRldCwgImZhbHNlX3Bvc2l0aXZlX3JhdGUiOiBmcHIsICJvdmVyYWxsX2FjYyI6IGFjYywKICAgICAgICAgICAgIm5fdGFtcGVyIjogbGVuKHRhbXBlciksICJuX2JlbmlnbiI6IGxlbihiZW5pZ24pfQoKCmRlZiBiZW5lZml0X2F0KHJlc3VsdHM6IGxpc3RbZGljdF0sIHRocmVzaG9sZDogZmxvYXQgPSAwLjUpIC0+IGZsb2F0OgogICAgIiIiRnJhY3Rpb24gb2YgdGFtcGVyZWQgaW1hZ2VzIGxvY2FsaXNlZCB0byBhdCBsZWFzdCBgdGhyZXNob2xkYCBJb1UuIiIiCiAgICB0YW1wZXIgPSBbciBmb3IgciBpbiByZXN1bHRzIGlmIHJbImtpbmQiXSA9PSAidGFtcGVyIl0KICAgIGlmIG5vdCB0YW1wZXI6CiAgICAgICAgcmV0dXJuIDAuMAogICAgaGl0cyA9IHN1bShpb3UoclsicHJlZCJdLCByWyJndCJdKSA+PSB0aHJlc2hvbGQgZm9yIHIgaW4gdGFtcGVyKQogICAgcmV0dXJuIGhpdHMgLyBsZW4odGFtcGVyKQoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBhYnN0ZW50aW9uCgpkZWYgc2VsZWN0aXZlX3Jpc2soY29uZmlkZW5jZTogbnAubmRhcnJheSwgY29ycmVjdDogbnAubmRhcnJheSwKICAgICAgICAgICAgICAgICAgIGNvdmVyYWdlczogdHVwbGVbZmxvYXQsIC4uLl0gPSAoMS4wLCAwLjksIDAuNzUsIDAuNSwgMC4yNSwgMC4xKSkgLT4gZGljdFtzdHIsIGZsb2F0XToKICAgICIiIlJpc2sgYW1vbmcgdGhlIGJsb2NrcyB0aGUgc3lzdGVtIGNob3NlIHRvIGFuc3dlci4KCiAgICBUaGlzIGlzIHRoZSBtZXRyaWMgdGhhdCBtYWtlcyBhYnN0ZW50aW9uIGhvbmVzdC4gQSBzeXN0ZW0gdGhhdCBhYnN0YWlucyBvbgogICAgZXZlcnl0aGluZyBoYXMgY292ZXJhZ2UgMCBhbmQgdW5kZWZpbmVkIHJpc2sg4oCUIHRoZSBjdXJ2ZSBzaG93cyBpdAogICAgY29sbGFwc2luZywgd2hpY2ggaXMgdGhlIHBvaW50LiBSZXBvcnQgdGhlIHdob2xlIGN1cnZlLCBuZXZlciBvbmUgcG9pbnQuCiAgICAiIiIKICAgIGNvbmYgPSBucC5hc2FycmF5KGNvbmZpZGVuY2UsIG5wLmZsb2F0NjQpLnJhdmVsKCkKICAgIG9rID0gbnAuYXNhcnJheShjb3JyZWN0LCBib29sKS5yYXZlbCgpCiAgICBpZiBjb25mLnNpemUgIT0gb2suc2l6ZToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJjb25maWRlbmNlIGFuZCBjb3JyZWN0IG11c3QgYmUgdGhlIHNhbWUgbGVuZ3RoIikKICAgIGlmIGNvbmYuc2l6ZSA9PSAwOgogICAgICAgIHJldHVybiB7ZiJjb3Zfe2M6LjJmfSI6IGZsb2F0KCJuYW4iKSBmb3IgYyBpbiBjb3ZlcmFnZXN9CiAgICBvcmRlciA9IG5wLmFyZ3NvcnQoLWNvbmYpCiAgICBvdXQ6IGRpY3Rbc3RyLCBmbG9hdF0gPSB7fQogICAgZm9yIGMgaW4gY292ZXJhZ2VzOgogICAgICAgIGsgPSBtYXgoMSwgaW50KHJvdW5kKGMgKiBjb25mLnNpemUpKSkKICAgICAgICBzZWwgPSBvcmRlcls6a10KICAgICAgICBvdXRbZiJjb3Zfe2M6LjJmfSJdID0gZmxvYXQoMS4wIC0gb2tbc2VsXS5tZWFuKCkpCiAgICByZXR1cm4gb3V0CgoKZGVmIGF1cmMoY29uZmlkZW5jZTogbnAubmRhcnJheSwgY29ycmVjdDogbnAubmRhcnJheSwgc3RlcHM6IGludCA9IDUwKSAtPiBmbG9hdDoKICAgICIiIkFyZWEgdW5kZXIgdGhlIHJpc2stY292ZXJhZ2UgY3VydmUuIExvd2VyIGlzIGJldHRlci4KCiAgICBBIHN5c3RlbSB3aXRoIG5vIHJhbmtpbmcgYWJpbGl0eSBoYXMgQVVSQyBlcXVhbCB0byBpdHMgcmF3IGVycm9yIHJhdGU7IGEKICAgIHN5c3RlbSB3aG9zZSBjb25maWRlbmNlIG1lYW5zIHNvbWV0aGluZyBoYXMgQVVSQyB3ZWxsIGJlbG93IGl0LgogICAgIiIiCiAgICBjb25mID0gbnAuYXNhcnJheShjb25maWRlbmNlLCBucC5mbG9hdDY0KS5yYXZlbCgpCiAgICBvayA9IG5wLmFzYXJyYXkoY29ycmVjdCwgYm9vbCkucmF2ZWwoKQogICAgbiA9IGNvbmYuc2l6ZQogICAgaWYgbiA9PSAwOgogICAgICAgIHJldHVybiBmbG9hdCgibmFuIikKICAgIG9yZGVyID0gbnAuYXJnc29ydCgtY29uZikKICAgIGVyciA9ICh+b2tbb3JkZXJdKS5hc3R5cGUobnAuZmxvYXQ2NCkKICAgIGN1bSA9IG5wLmN1bXN1bShlcnIpIC8gbnAuYXJhbmdlKDEsIG4gKyAxKQogICAgY292ID0gbnAubGluc3BhY2UoMS4wIC8gbiwgMS4wLCBuKQogICAgcmV0dXJuIGZsb2F0KG5wLnRyYXBlem9pZChjdW0sIGNvdikpCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIGFnZ3JlZ2F0ZQoKZGVmIHN1bW1hcmlzZShyZXN1bHRzOiBsaXN0W2RpY3RdKSAtPiBkaWN0OgogICAgIiIiT25lIGNvbmZpZ3VyYXRpb24gb24gb25lIGNvcnB1cy4gUmF0aW9zIHdpdGggdGhlaXIgZGVub21pbmF0b3JzLiIiIgogICAgdGFtcGVyID0gW3IgZm9yIHIgaW4gcmVzdWx0cyBpZiByWyJraW5kIl0gPT0gInRhbXBlciJdCiAgICBvdXRfcGVyX2ZhbWlseTogZGljdFtzdHIsIGRpY3RdID0ge30KICAgIGZvciByIGluIHRhbXBlcjoKICAgICAgICBmYW0gPSByWyJmYW1pbHkiXQogICAgICAgIGJ1Y2tldCA9IG91dF9wZXJfZmFtaWx5LnNldGRlZmF1bHQoZmFtLCB7ImlvdXMiOiBbXSwgImNvdW50cyI6IFtdLCAibiI6IDB9KQogICAgICAgIGJ1Y2tldFsiaW91cyJdLmFwcGVuZChpb3UoclsicHJlZCJdLCByWyJndCJdKSkKICAgICAgICBidWNrZXRbImNvdW50cyJdLmFwcGVuZChjb3VudHMoclsicHJlZCJdLCByWyJndCJdKSkKICAgICAgICBidWNrZXRbIm4iXSArPSAxCiAgICBwZXJfZmFtaWx5ID0ge30KICAgIGZvciBmYW0sIGIgaW4gb3V0X3Blcl9mYW1pbHkuaXRlbXMoKToKICAgICAgICB0cCA9IHN1bShjWyJ0cCJdIGZvciBjIGluIGJbImNvdW50cyJdKQogICAgICAgIGZwID0gc3VtKGNbImZwIl0gZm9yIGMgaW4gYlsiY291bnRzIl0pCiAgICAgICAgZm4gPSBzdW0oY1siZm4iXSBmb3IgYyBpbiBiWyJjb3VudHMiXSkKICAgICAgICBkZW5vbSA9IHRwICsgZnAgKyBmbgogICAgICAgIHBlcl9mYW1pbHlbZmFtXSA9IHsKICAgICAgICAgICAgIm4iOiBiWyJuIl0sCiAgICAgICAgICAgICJpb3VfbWVhbiI6IGZsb2F0KG5wLm1lYW4oYlsiaW91cyJdKSksCiAgICAgICAgICAgICJpb3VfcG9vbGVkIjogMS4wIGlmIGRlbm9tID09IDAgZWxzZSB0cCAvIGRlbm9tLAogICAgICAgICAgICAidHAiOiB0cCwgImZwIjogZnAsICJmbiI6IGZuLAogICAgICAgIH0KICAgIGRldCA9IGRldGVjdGlvbl9hY2N1cmFjeShyZXN1bHRzKQogICAgcmV0dXJuIHsKICAgICAgICAiZGV0ZWN0aW9uIjogZGV0LAogICAgICAgICJiZW5lZml0X2F0XzAuNSI6IGJlbmVmaXRfYXQocmVzdWx0cywgMC41KSwKICAgICAgICAiaW91X21lYW4iOiBmbG9hdChucC5tZWFuKFtpb3UoclsicHJlZCJdLCByWyJndCJdKSBmb3IgciBpbiB0YW1wZXJdKSkgaWYgdGFtcGVyIGVsc2UgMC4wLAogICAgICAgICJwZXJfZmFtaWx5IjogcGVyX2ZhbWlseSwKICAgICAgICAibl9pbWFnZXMiOiBsZW4ocmVzdWx0cyksCiAgICB9CgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIGNvbXBhcmlzb24gZ3VhcmRzCgpkZWYgYmxvY2tfY2VpbGluZyhwaXhlbF9tYXNrOiBucC5uZGFycmF5LCBibG9jazogaW50ID0gQkxPQ0spIC0+IGZsb2F0OgogICAgIiIiUGl4ZWwgSW9VIHRoYXQgYSBQRVJGRUNUIGJsb2NrLWdyYW51bGFyIGRldGVjdG9yIHNjb3JlcyBvbiB0aGlzIG1hc2suCgogICAgVGhpcyBpcyB0aGUgbnVtYmVyIHRoYXQgZGVjaWRlcyB3aGV0aGVyIGEgY29tcGFyaXNvbiBpcyBmYWlyLiBBbiA4eDggdmVyZGljdAogICAgZ3JpZCBjYW5ub3QgZXhwcmVzcyBhIGJvdW5kYXJ5IGZpbmVyIHRoYW4gOCBwaXhlbHMsIHNvIGZvciBhIGNpcmN1bGFyIGxlc2lvbgogICAgb2YgcmFkaXVzIDMyICgxLjIlIG9mIGEgNTEyeDUxMiBmcmFtZSkgZXZlbiBhIGZsYXdsZXNzIGJsb2NrIGNsYXNzaWZpZXIgdG9wcwogICAgb3V0IGF0IDAuODA5IHBpeGVsIElvVS4gQSBzeXN0ZW0gcmVwb3J0aW5nIHBpeGVsIG1hc2tzIGZyb20gYSA2NHg2NCBncmlkCiAgICB0aGVyZWZvcmUgY2Fubm90IHJlYWNoIGEgcHVibGlzaGVkIHBpeGVsIElvVSBhYm92ZSB0aGlzIGNlaWxpbmcsIHdoYXRldmVyCiAgICBpdCBsZWFybnMuCgogICAgTWVhc3VyZWQsIGNpcmN1bGFyIGxlc2lvbnMgb24gNTEyeDUxMjoKICAgICAgICByPTggICAtPiAwLjUxMyAgICAgcj0zMiAtPiAwLjgwOSAgICAgcj05NiAgLT4gMC45MzAKICAgICAgICByPTE2ICAtPiAwLjY5MiAgICAgcj00OCAtPiAwLjg1NCAgICAgcj0xMjggLT4gMC45MzcgICAgcj0xNjAgLT4gMC45NTAKCiAgICBBbHdheXMgcmVwb3J0IHRoaXMgYmVzaWRlIGEgYmxvY2stZ3JhbnVsYXIgSW9VIHNvIGEgc2hvcnRmYWxsIGlzCiAgICBhdHRyaWJ1dGFibGUgdG8gdGhlIGFyY2hpdGVjdHVyZSByYXRoZXIgdGhhbiBibGFtZWQgb24gdGhlIG1vZGVsLgogICAgIiIiCiAgICBtID0gbnAuYXNhcnJheShwaXhlbF9tYXNrLCBib29sKQogICAgbnksIG54ID0gbS5zaGFwZVswXSAvLyBibG9jaywgbS5zaGFwZVsxXSAvLyBibG9jawogICAgYmxvY2tzID0gbVs6bnkgKiBibG9jaywgOm54ICogYmxvY2tdLnJlc2hhcGUobnksIGJsb2NrLCBueCwgYmxvY2spLmFueShheGlzPSgxLCAzKSkKICAgIGFwcHJveCA9IG5wLnJlcGVhdChucC5yZXBlYXQoYmxvY2tzLCBibG9jaywgMCksIGJsb2NrLCAxKQogICAgcmV0dXJuIGlvdShhcHByb3gsIG1bOm55ICogYmxvY2ssIDpueCAqIGJsb2NrXSkKCgpkZWYgYXJlYV9idWNrZXRzKGZyYWNzLCBlZGdlcz0oMC4wLCAwLjAwNSwgMC4wMiwgMC4wNSwgMC4yMCwgMS4wMSkpIC0+IGxpc3Rbc3RyXToKICAgICIiIlN0cmF0aWZ5IGltYWdlcyBieSB0YW1wZXJlZCBhcmVhLiBJb1UgaXMgc3Ryb25nbHkgYXJlYS1kZXBlbmRlbnQsIHNvIGEKICAgIG1lYW4gb3ZlciBhIG1peGVkIHNldCBpcyBkb21pbmF0ZWQgYnkgd2hpY2hldmVyIGJ1Y2tldCBpcyBtb3N0IG51bWVyb3VzIGFuZAogICAgc2F5cyBub3RoaW5nIGFib3V0IGFueSBwYXJ0aWN1bGFyIHNjYWxlLiIiIgogICAgb3V0ID0gW10KICAgIGZvciBmIGluIG5wLmFzYXJyYXkoZnJhY3MsIG5wLmZsb2F0NjQpOgogICAgICAgIGZvciBsbywgaGkgaW4gemlwKGVkZ2VzWzotMV0sIGVkZ2VzWzE6XSk6CiAgICAgICAgICAgIGlmIGxvIDw9IGYgPCBoaToKICAgICAgICAgICAgICAgIG91dC5hcHBlbmQoZiJhcmVhX3tsbzouM2Z9LXtoaTouM2Z9IikKICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgZWxzZToKICAgICAgICAgICAgb3V0LmFwcGVuZCgiYXJlYV9vdmVyZmxvdyIpCiAgICByZXR1cm4gb3V0CgoKZGVmIHN0cmF0aWZpZWRfaW91KHJlc3VsdHM6IGxpc3RbZGljdF0sIGJsb2NrOiBpbnQgPSBCTE9DSykgLT4gZGljdFtzdHIsIGRpY3RdOgogICAgIiIiSW9VIHBlciB0YW1wZXJlZC1hcmVhIGJ1Y2tldC4KCiAgICBBIHJlc3VsdCBtYXkgY2FycnkgYSBibG9jayBtYXNrIChgZ3RgLCBgcHJlZGApLCBhIHBpeGVsIG1hc2sgKGBndF9waXhlbHNgKSwKICAgIG9yIGEgcGl4ZWwgcHJlZGljdGlvbiAoYHByZWRfcGl4ZWxzYCkuIFRoZSBibG9jayBJb1UgaXMgYWx3YXlzIHJlcG9ydGVkOyB0aGUKICAgIHBpeGVsIElvVSBvbmx5IHdoZW4gcmVhbCBwaXhlbCBncm91bmQgdHJ1dGggaXMgcHJlc2VudCwgYmVjYXVzZSB1cHNhbXBsaW5nIGEKICAgIGJsb2NrIG1hc2sgdG8gcGl4ZWxzIGFuZCBzY29yaW5nIGl0IGFnYWluc3QgYSBibG9jayBtYXNrIHdvdWxkIGJlIGEKICAgIGNhdGVnb3J5IGVycm9yIC0tIGJvdGggc2lkZXMgd291bGQgYmUgdGhlIHNhbWUgZ3JpZC4KCiAgICBgY2VpbGluZ2AgaXMgdGhlIGJlc3QgcGl4ZWwgSW9VIGEgcGVyZmVjdCBibG9jayBkZXRlY3RvciBjb3VsZCByZWFjaCBvbiB0aGUKICAgIHBpeGVsIHRydXRoIGluIHRoYXQgYnVja2V0LiBRdW90ZSBpdCBiZXNpZGUgYW55IGJsb2NrLWdyYW51bGFyIG51bWJlci4KICAgICIiIgogICAgdGFtcGVyID0gW3IgZm9yIHIgaW4gcmVzdWx0cyBpZiByWyJraW5kIl0gPT0gInRhbXBlciJdCiAgICBpZiBub3QgdGFtcGVyOgogICAgICAgIHJldHVybiB7fQogICAgaGFzX3BpeGVscyA9IGFsbCgiZ3RfcGl4ZWxzIiBpbiByIGZvciByIGluIHRhbXBlcikKICAgIGZyYWNzID0gW25wLmFzYXJyYXkoclsiZ3RfcGl4ZWxzIl0gaWYgaGFzX3BpeGVscyBlbHNlIHJbImd0Il0sIGJvb2wpLm1lYW4oKQogICAgICAgICAgICAgZm9yIHIgaW4gdGFtcGVyXQogICAgbGFiZWxzID0gYXJlYV9idWNrZXRzKGZyYWNzKQogICAgb3V0OiBkaWN0W3N0ciwgZGljdF0gPSB7fQogICAgZm9yIGxhYiBpbiBzb3J0ZWQoc2V0KGxhYmVscykpOgogICAgICAgIHNlbCA9IFtpIGZvciBpLCBsIGluIGVudW1lcmF0ZShsYWJlbHMpIGlmIGwgPT0gbGFiXQogICAgICAgIHJvdyA9IHsKICAgICAgICAgICAgIm4iOiBsZW4oc2VsKSwKICAgICAgICAgICAgImlvdV9ibG9jayI6IGZsb2F0KG5wLm1lYW4oW2lvdSh0YW1wZXJbaV1bInByZWQiXSwgdGFtcGVyW2ldWyJndCJdKSBmb3IgaSBpbiBzZWxdKSksCiAgICAgICAgICAgICJndF9hcmVhX21lYW4iOiBmbG9hdChucC5tZWFuKFtmcmFjc1tpXSBmb3IgaSBpbiBzZWxdKSksCiAgICAgICAgfQogICAgICAgIGlmIGhhc19waXhlbHM6CiAgICAgICAgICAgIGdwID0gW25wLmFzYXJyYXkodGFtcGVyW2ldWyJndF9waXhlbHMiXSwgYm9vbCkgZm9yIGkgaW4gc2VsXQogICAgICAgICAgICBwcCA9IFtucC5hc2FycmF5KHRhbXBlcltpXVsicHJlZF9waXhlbHMiXSwgYm9vbCkKICAgICAgICAgICAgICAgICAgaWYgInByZWRfcGl4ZWxzIiBpbiB0YW1wZXJbaV0gZWxzZSB1cHNhbXBsZV90byh0YW1wZXJbaV1bInByZWQiXSwgZ3Bba10uc2hhcGUpCiAgICAgICAgICAgICAgICAgIGZvciBrLCBpIGluIGVudW1lcmF0ZShzZWwpXQogICAgICAgICAgICByb3dbImlvdV9waXhlbCJdID0gZmxvYXQobnAubWVhbihbaW91KHBwW2tdLCBncFtrXSkgZm9yIGsgaW4gcmFuZ2UobGVuKHNlbCkpXSkpCiAgICAgICAgICAgIHJvd1siY2VpbGluZyJdID0gZmxvYXQobnAubWVhbihbYmxvY2tfY2VpbGluZyhncFtrXSwgYmxvY2spIGZvciBrIGluIHJhbmdlKGxlbihzZWwpKV0pKQogICAgICAgIG91dFtsYWJdID0gcm93CiAgICByZXR1cm4gb3V0CgoKZGVmIHVwc2FtcGxlX3RvKGJsb2NrX21hc2s6IG5wLm5kYXJyYXksIHNoYXBlOiB0dXBsZVtpbnQsIGludF0sIGJsb2NrOiBpbnQgPSBCTE9DSwogICAgICAgICAgICAgICAgKSAtPiBucC5uZGFycmF5OgogICAgIiIiQmxvY2sgbWFzayAtPiBwaXhlbCBtYXNrIGJ5IGJveCByZXBsaWNhdGlvbi4iIiIKICAgIG0gPSBucC5hc2FycmF5KGJsb2NrX21hc2ssIGJvb2wpCiAgICB1cCA9IG5wLnJlcGVhdChucC5yZXBlYXQobSwgYmxvY2ssIDApLCBibG9jaywgMSkKICAgIG91dCA9IG5wLnplcm9zKHNoYXBlLCBib29sKQogICAgaCA9IG1pbihzaGFwZVswXSwgdXAuc2hhcGVbMF0pCiAgICB3ID0gbWluKHNoYXBlWzFdLCB1cC5zaGFwZVsxXSkKICAgIG91dFs6aCwgOnddID0gdXBbOmgsIDp3XQogICAgcmV0dXJuIG91dAoKCmRlZiBtZWFzdXJlZF9waXhlbF9tYXNrKGJlZm9yZTogbnAubmRhcnJheSwgYWZ0ZXI6IG5wLm5kYXJyYXkpIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJQaXhlbC1sZXZlbCBncm91bmQgdHJ1dGg6IGV4YWN0bHkgd2hpY2ggcGl4ZWxzIGNoYW5nZWQuCgogICAgVGhlIGJsb2NrIG1hc2sgaXMgd2hhdCB0aGUgdmVyZGljdCBsYXllciByZWFzb25zIG92ZXI7IHRoZSBwaXhlbCBtYXNrIGlzCiAgICB3aGF0IGxvY2FsaXNhdGlvbiBpcyBzY29yZWQgb24uIEtlZXBpbmcgYm90aCBtZWFucyB0aGUgcmVwb3J0ZWQgcGl4ZWwgSW9VCiAgICBpcyBhIHJlYWwgcGl4ZWwgSW9VIGFuZCBub3QgYW4gOHggdXBzY2FsZSBvZiBhIGJsb2NrIGRlY2lzaW9uIC0tIHdoaWNoCiAgICB3b3VsZCBiZSBpbXBvc3NpYmxlIHRvIGV4Y2VlZCB0aGUgYmxvY2sgY2VpbGluZyBhbmQgd291bGQgbG9vayBsaWtlIGEgbW9kZWwKICAgIGZhaWx1cmUgd2hlbiBpdCBpcyBhIG1lYXN1cmVtZW50IGNob2ljZS4KICAgICIiIgogICAgYmVmb3JlID0gbnAuYXNhcnJheShiZWZvcmUpCiAgICBhZnRlciA9IG5wLmFzYXJyYXkoYWZ0ZXIpCiAgICBpZiBiZWZvcmUuc2hhcGUgIT0gYWZ0ZXIuc2hhcGU6CiAgICAgICAgcmV0dXJuIG5wLm9uZXMoYWZ0ZXIuc2hhcGVbOjJdLCBib29sKQogICAgcmV0dXJuIChiZWZvcmUgIT0gYWZ0ZXIpLmFueShheGlzPTIpCg==', 'evidence': 'IiIiVGhlIGtleWVkIGV2aWRlbmNlIGNoYW5uZWwuCgpUaGlzIG1vZHVsZSBpcyByb3cgMiBvZiB0aGUgc3ludGhlc2lzIHRhYmxlLCBhbmQgaXQgaXMgdGhlIG9uZSBjb21wb25lbnQgbm8KcHVibGlzaGVkIHN5c3RlbSBpbiB0aGUgY29tcGFyaXNvbiBoYXMuIEV2ZXJ5dGhpbmcgZG93bnN0cmVhbSDigJQgYXV0b2VuY29kZXIsClJlZmluZU5ldCwgZGVjaXNpb24gaGVhZCDigJQgcmVhZHMgdGhlIHR3byBtYXBzIHRoaXMgcHJvZHVjZXM6CgogICAgRSAgIHBlci1ibG9jayBkaXNhZ3JlZW1lbnQgd2l0aCB0aGUga2V5ZWQgdGFnICAgICAgKDY0eDY0LCBoaWdoID0gc3VzcGljaW91cykKICAgIFMgICBwZXItYmxvY2sgbWFyayBzdXJ2aXZhbCAgICAgICAgICAgICAgICAgICAgICAgICg2NHg2NCwgbG93ID0gdGhlIG1hcmsKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB3YXMgZXJhc2VkIG9yIG5ldmVyIGZpdCkKClRoZSBwcmluY2lwbGUgdGhlIGFyY2hpdGVjdHVyZSBkZXBlbmRzIG9uOgoKICAgIFRIRSBORVRXT1JLIE5FVkVSIFJFUExBQ0VTIFRISVMgRVZJREVOQ0UuIElUIFJFQURTIElULgoKVGhlIG1hcmsgaXMga2V5ZWQuIEFuIGF0dGFja2VyIHdpdGhvdXQgdGhlIGtleSBjYW5ub3QgY29uc3RydWN0IGEgdmFsaWQgdGFnIGZvcgpjb250ZW50IHRoZXkgZmFicmljYXRlZCwgc28gdGhlIGV2aWRlbmNlIGNhbm5vdCBiZSBtYW51ZmFjdHVyZWQgYnkgdGhlIGF0dGFja2VyLgpBIGxlYXJuZWQgZW1iZWRkZXIgaGFzIG5vIHN1Y2ggcHJvcGVydHk6IGdpdmVuIGVub3VnaCBpbnB1dC9vdXRwdXQgcGFpcnMsIHRoZQppbnZlcnNlIGNhbiBiZSB0cmFpbmVkLiBUaGF0IGFzeW1tZXRyeSBpcyB0aGUgY29udHJpYnV0aW9uLCBhbmQgQzkgaW4gdGhlCmV4cGVyaW1lbnQgbWF0cml4IGlzIHRoZSBydW4gdGhhdCBwcmljZXMgaXQuCgpEZXNpZ24gY2FycmllZCBvdmVyIGZyb20gdGhlIGNsYXNzaWNhbCBwaGFzZToKCiAgKiA4eDggYmxvY2tzLCAyLWxldmVsIEhhYXIsIDYwIGRldGFpbCBjb2VmZmljaWVudHMgcGVyIGJsb2NrICg0OCArIDEyKQogICogUUlNIHdpdGggc3RlcCBERUxUQSwgYW1wbGl0dWRlIERFTFRBLzQsIHNvIHRoZSBlbWJlZGRpbmcgbmV2ZXIgY3Jvc3NlcyBhCiAgICBsYXR0aWNlIGJvdW5kYXJ5IGFuZCB0aGUgcXVhbnRpc2F0aW9uIGluZGV4IGlzIGV4YWN0bHkgcmVjb3ZlcmFibGUKICAqIHRoZSB0YWcgYmluZHMgS0VZICsgSU1BR0UgSUQgKyBBRERSRVNTICsgdGhlIGJsb2NrJ3Mgb3duIHF1YW50aXNhdGlvbgogICAgc2lnbmF0dXJlLCBzbyBhIGJsb2NrIHBhc3RlZCBmcm9tIGVsc2V3aGVyZSBjYXJyaWVzIHRoZSB3cm9uZyBhZGRyZXNzIGFuZCBhCiAgICBibG9jayB3aG9zZSBjb250ZW50IGNoYW5nZWQgY2FycmllcyB0aGUgd3Jvbmcgc2lnbmF0dXJlCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGhtYWMKCmltcG9ydCBudW1weSBhcyBucAoKQkxPQ0sgPSA4CkRFTFRBID0gOC4wCk5fU0xPVFMgPSA2MAoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBsdW1hCgpkZWYgbHVtYShyZ2I6IG5wLm5kYXJyYXkpIC0+IG5wLm5kYXJyYXk6CiAgICByZ2IgPSBucC5hc2FycmF5KHJnYiwgbnAuZmxvYXQ2NCkKICAgIHJldHVybiAwLjI5OSAqIHJnYlsuLi4sIDBdICsgMC41ODcgKiByZ2JbLi4uLCAxXSArIDAuMTE0ICogcmdiWy4uLiwgMl0KCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gaGFhcgoKZGVmIGhhYXIyKHg6IG5wLm5kYXJyYXkpIC0+IGRpY3Rbc3RyLCBucC5uZGFycmF5XToKICAgICIiIlR3by1sZXZlbCAyRCBIYWFyIG9uIGEgKEgsIFcpIHBsYW5lLCBzdHJpZGUtMiwgYmxvY2stbG9jYWwgZm9yIDh4OCB0aWxlcy4KCiAgICBXaXRoIEgsIFcgbXVsdGlwbGVzIG9mIDggdGhlIGxldmVsLTEgdGlsZXMgYXJlIDR4NCBhbmQgbGV2ZWwtMiB0aWxlcyBhcmUKICAgIDJ4Miwgc28gbm8gY29lZmZpY2llbnQgZXZlciBtaXhlcyB0d28gYmxvY2tzLiBUaGF0IGlzIHdoYXQgbGV0cyB0aGUgdmVyaWZpZXIKICAgIHJlYWQgZWFjaCBibG9jayBpbiBpc29sYXRpb24uCiAgICAiIiIKICAgIGRlZiBfbHZsKGEpOgogICAgICAgIGEwMCwgYTAxID0gYVswOjoyLCAwOjoyXSwgYVswOjoyLCAxOjoyXQogICAgICAgIGExMCwgYTExID0gYVsxOjoyLCAwOjoyXSwgYVsxOjoyLCAxOjoyXQogICAgICAgIHJldHVybiAoKGEwMCArIGEwMSArIGExMCArIGExMSkgLyAyLjAsCiAgICAgICAgICAgICAgICAoYTAwICsgYTAxIC0gYTEwIC0gYTExKSAvIDIuMCwKICAgICAgICAgICAgICAgIChhMDAgLSBhMDEgKyBhMTAgLSBhMTEpIC8gMi4wLAogICAgICAgICAgICAgICAgKGEwMCAtIGEwMSAtIGExMCArIGExMSkgLyAyLjApCgogICAgbGwxLCBsaDEsIGhsMSwgaGgxID0gX2x2bCh4KQogICAgbGwyLCBsaDIsIGhsMiwgaGgyID0gX2x2bChsbDEpCiAgICByZXR1cm4geyJMTDIiOiBsbDIsICJMSDIiOiBsaDIsICJITDIiOiBobDIsICJISDIiOiBoaDIsCiAgICAgICAgICAgICJMSDEiOiBsaDEsICJITDEiOiBobDEsICJISDEiOiBoaDF9CgoKZGVmIGhhYXIyX2ludmVyc2UoYzogZGljdFtzdHIsIG5wLm5kYXJyYXldKSAtPiBucC5uZGFycmF5OgogICAgZGVmIF9pbHZsKGxsLCBsaCwgaGwsIGhoKToKICAgICAgICBhID0gbnAuemVyb3MoKGxsLnNoYXBlWzBdICogMiwgbGwuc2hhcGVbMV0gKiAyKSwgbnAuZmxvYXQ2NCkKICAgICAgICBhWzA6OjIsIDA6OjJdID0gKGxsICsgbGggKyBobCArIGhoKSAvIDIuMAogICAgICAgIGFbMDo6MiwgMTo6Ml0gPSAobGwgKyBsaCAtIGhsIC0gaGgpIC8gMi4wCiAgICAgICAgYVsxOjoyLCAwOjoyXSA9IChsbCAtIGxoICsgaGwgLSBoaCkgLyAyLjAKICAgICAgICBhWzE6OjIsIDE6OjJdID0gKGxsIC0gbGggLSBobCArIGhoKSAvIDIuMAogICAgICAgIHJldHVybiBhCiAgICBsbDEgPSBfaWx2bChjWyJMTDIiXSwgY1siTEgyIl0sIGNbIkhMMiJdLCBjWyJISDIiXSkKICAgIHJldHVybiBfaWx2bChsbDEsIGNbIkxIMSJdLCBjWyJITDEiXSwgY1siSEgxIl0pCgoKZGVmIGRldGFpbHMoYzogZGljdFtzdHIsIG5wLm5kYXJyYXldKSAtPiBucC5uZGFycmF5OgogICAgIiIiKG55LCBueCwgNjApIHZpZXcgb2YgdGhlIGRldGFpbCBjb2VmZmljaWVudHMsIGJhbmQgb3JkZXIgZml4ZWQuIiIiCiAgICBiYW5kcyA9IFtjWyJMSDEiXSwgY1siSEwxIl0sIGNbIkhIMSJdLCBjWyJMSDIiXSwgY1siSEwyIl0sIGNbIkhIMiJdXQogICAgbnksIG54ID0gY1siTEwyIl0uc2hhcGVbMF0gLy8gMiwgY1siTEwyIl0uc2hhcGVbMV0gLy8gMgogICAgb3V0ID0gW10KICAgIGZvciBiIGluIGJhbmRzOgogICAgICAgIGsgPSBiLnNoYXBlWzBdIC8vIG55CiAgICAgICAgIyAobnksIGssIG54LCBrKSAtPiB0cmFuc3Bvc2UgLT4gKG55LCBueCwgaywgaykgLT4gZmxhdHRlbiB0aGUgdGlsZS4KICAgICAgICAjIFJlc2hhcGluZyAobnksIGssIG54LCBrKSBzdHJhaWdodCB0byAobnksIG54LCBrKmspIGNvbGxhcHNlcyB0aGUgYXhlcwogICAgICAgICMgaW4gdGhlIHdyb25nIG9yZGVyIGFuZCBzY2F0dGVycyBvbmUgYmxvY2sncyBjb2VmZmljaWVudHMgYWNyb3NzCiAgICAgICAgIyBzZXZlcmFsIGJsb2Nrcy4gSXQgc3RpbGwgcm91bmQtdHJpcHMsIHNvIGFuIGludmVyc2Utb25seSB0ZXN0IHBhc3NlcwogICAgICAgICMgd2hpbGUgYmxvY2sgbG9jYWxpdHkgaXMgZGVzdHJveWVkLgogICAgICAgIG91dC5hcHBlbmQoYi5yZXNoYXBlKG55LCBrLCBueCwgaykudHJhbnNwb3NlKDAsIDIsIDEsIDMpLnJlc2hhcGUobnksIG54LCBrICogaykpCiAgICByZXR1cm4gbnAuY29uY2F0ZW5hdGUob3V0LCBheGlzPTIpCgoKZGVmIGRldGFpbHNfc2V0KGM6IGRpY3Rbc3RyLCBucC5uZGFycmF5XSwgZmxhdDogbnAubmRhcnJheSkgLT4gZGljdFtzdHIsIG5wLm5kYXJyYXldOgogICAgIiIiSW52ZXJzZSBvZiBgZGV0YWlsc2AuIiIiCiAgICBueSwgbnggPSBjWyJMTDIiXS5zaGFwZVswXSAvLyAyLCBjWyJMTDIiXS5zaGFwZVsxXSAvLyAyCiAgICBvdXQgPSBkaWN0KGMpCiAgICBpID0gMAogICAgZm9yIG5hbWUsIGsgaW4gWygiTEgxIiwgNCksICgiSEwxIiwgNCksICgiSEgxIiwgNCksICgiTEgyIiwgMiksICgiSEwyIiwgMiksICgiSEgyIiwgMildOgogICAgICAgIGJsayA9IChmbGF0WzosIDosIGk6aSArIGsgKiBrXS5yZXNoYXBlKG55LCBueCwgaywgaykKICAgICAgICAgICAgICAgLnRyYW5zcG9zZSgwLCAyLCAxLCAzKS5yZXNoYXBlKG55ICogaywgbnggKiBrKSkKICAgICAgICBvdXRbbmFtZV0gPSBibGsKICAgICAgICBpICs9IGsgKiBrCiAgICByZXR1cm4gb3V0CgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIGtleWluZwoKZGVmIF9rZXlfc3RyZWFtKGtleTogYnl0ZXMsIGltYWdlX2lkOiBpbnQsIG5fYmxvY2tzOiBpbnQpIC0+IHR1cGxlW25wLm5kYXJyYXksIG5wLm5kYXJyYXldOgogICAgIiIiNjAga2V5ZWQgcGFkIGJpdHMgYW5kIGEgMTYtZW50cnkgc2VsZWN0b3IsIHBlciBibG9jay4iIiIKICAgICMgaW50KCkgY29lcmNpb24gaXMgZGVsaWJlcmF0ZTogY2FsbGVycyBwYXNzIG51bXB5IGludGVnZXJzIGNvbnN0YW50bHkKICAgICMgKGFycmF5IGluZGljZXMpLCBhbmQgbnVtcHkgaW50cyBoYXZlIG5vIC50b19ieXRlcy4KICAgIG0gPSBobWFjLm5ldyhrZXksIGludChpbWFnZV9pZCkudG9fYnl0ZXMoNCwgImJpZyIpLCBoYXNobGliLnNoYTI1NikuZGlnZXN0KCkKICAgIHJuZyA9IG5wLnJhbmRvbS5kZWZhdWx0X3JuZyhpbnQuZnJvbV9ieXRlcyhtWzo4XSwgImJpZyIpKQogICAgcGFkID0gcm5nLmludGVnZXJzKDAsIDIsIChuX2Jsb2NrcywgTl9TTE9UUykpLmFzdHlwZShucC51aW50OCkKICAgIHNlbCA9IHJuZy5pbnRlZ2VycygwLCAyLCAobl9ibG9ja3MsIDE2KSkuYXN0eXBlKG5wLnVpbnQ4KQogICAgcmV0dXJuIHBhZCwgc2VsCgoKZGVmIHNpZ25hdHVyZShmbGF0OiBucC5uZGFycmF5LCBkZWx0YTogZmxvYXQgPSBERUxUQSkgLT4gbnAubmRhcnJheToKICAgICIiIldoaWNoIGxhdHRpY2UgY2VsbCBlYWNoIGNvZWZmaWNpZW50IHNhdCBpbi4gRXhhY3RseSByZWNvdmVyYWJsZSBhZnRlcgogICAgbWFya2luZywgYmVjYXVzZSB0aGUgUUlNIG9mZnNldCBuZXZlciBjcm9zc2VzIGEgY2VsbCBib3VuZGFyeS4iIiIKICAgIHJldHVybiBucC5yb3VuZChmbGF0IC8gZGVsdGEpLmFzdHlwZShucC5pbnQ2NCkKCgpkZWYgX2RjX2NvZGUobGwyOiBucC5uZGFycmF5KSAtPiBucC5uZGFycmF5OgogICAgIiIiQSBjb2Fyc2UgNC1iaXQgY29kZSBvZiB0aGUgYmxvY2sgbWVhbiwgZnJvbSB0aGUgYmFuZCB0aGUgbWFyayBuZXZlciB3cml0ZXMuCgogICAgTEwyIGhvbGRzIDR4IHRoZSBibG9jayBtZWFuICh0d28gbGV2ZWxzIG9mIHVubm9ybWFsaXNlZCBIYWFyKSwgc28gaXRzIHJhbmdlCiAgICBpcyAwLi4xMDIwIG9uIGFuIDgtYml0IGltYWdlLiBEaXZpZGluZyBieSAxNiBhbmQgbWFza2luZyB0byA0IGJpdHMgZ2l2ZXMgYQogICAgMTYtd2F5IGNvZGU6IGNvYXJzZSBlbm91Z2ggdGhhdCBhIHNtYWxsIHBlcnR1cmJhdGlvbiBjYW5ub3QgZmxpcCBpdCwgZmluZQogICAgZW5vdWdoIHRoYXQgcmVwbGFjaW5nIGEgYmxvY2sncyBjb250ZW50IHVzdWFsbHkgZG9lcy4KICAgICIiIgogICAgbGwyID0gbnAuYXNhcnJheShsbDIsIG5wLmZsb2F0NjQpCiAgICBueSwgbnggPSBsbDIuc2hhcGVbMF0gLy8gMiwgbGwyLnNoYXBlWzFdIC8vIDIKICAgIG0gPSBsbDJbOm55ICogMiwgOm54ICogMl0ucmVzaGFwZShueSwgMiwgbngsIDIpLm1lYW4oYXhpcz0oMSwgMykpCiAgICByZXR1cm4gKG5wLmZsb29yKG0gLyAxNi4wKS5hc3R5cGUobnAuaW50NjQpICYgMHhGKQoKCmRlZiB0YWdzKGtleTogYnl0ZXMsIGltYWdlX2lkOiBpbnQsIGZsYXQ6IG5wLm5kYXJyYXksCiAgICAgICAgIGxsMl9tZWFuczogbnAubmRhcnJheSwgZGVsdGE6IGZsb2F0ID0gREVMVEEpIC0+IG5wLm5kYXJyYXk6CiAgICAiIiIobnksIG54LCA2MCkgZXhwZWN0ZWQgdGFnIGJpdHMuIiIiCiAgICBueSwgbnggPSBmbGF0LnNoYXBlWzoyXQogICAgcGFkLCBzZWwgPSBfa2V5X3N0cmVhbShrZXksIGltYWdlX2lkLCBueSAqIG54KQogICAgc2lnID0gc2lnbmF0dXJlKGZsYXQsIGRlbHRhKS5yZXNoYXBlKG55ICogbngsIE5fU0xPVFMpCiAgICBkYyA9IF9kY19jb2RlKGxsMl9tZWFucykucmVzaGFwZShueSAqIG54LCAxKQogICAgaWR4ID0gKHNpZyArIGRjKSAlIDE2CiAgICBjaG9zZW4gPSBucC50YWtlX2Fsb25nX2F4aXMoc2VsLCBpZHgsIGF4aXM9MSkKICAgIHJldHVybiAocGFkIF4gY2hvc2VuKS5yZXNoYXBlKG55LCBueCwgTl9TTE9UUykuYXN0eXBlKG5wLnVpbnQ4KQoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBlbWJlZCAvIHJlYWQKCmRlZiBlbWJlZChyZ2I6IG5wLm5kYXJyYXksIGtleTogYnl0ZXMsIGltYWdlX2lkOiBpbnQsCiAgICAgICAgICBkZWx0YTogZmxvYXQgPSBERUxUQSwgYW1wOiBmbG9hdCB8IE5vbmUgPSBOb25lKSAtPiBucC5uZGFycmF5OgogICAgIiIiV3JpdGUgdGhlIGtleWVkIHRhZyBpbnRvIHRoZSA2MCBkZXRhaWwgc2xvdHMgb2YgZXZlcnkgYmxvY2suIiIiCiAgICB5ID0gbHVtYShyZ2IpCiAgICBjID0gaGFhcjIoeSkKICAgIGZsYXQgPSBkZXRhaWxzKGMpCiAgICBhID0gZGVsdGEgLyA0LjAgaWYgYW1wIGlzIE5vbmUgZWxzZSBmbG9hdChhbXApCiAgICB0ID0gdGFncyhrZXksIGltYWdlX2lkLCBmbGF0LCBjWyJMTDIiXSwgZGVsdGEpCiAgICBxID0gbnAucm91bmQoZmxhdCAvIGRlbHRhKSAqIGRlbHRhCiAgICBtYXJrZWQgPSBxICsgKDIuMCAqIHQuYXN0eXBlKG5wLmZsb2F0NjQpIC0gMS4wKSAqIGEKICAgIHkyID0gaGFhcjJfaW52ZXJzZShkZXRhaWxzX3NldChjLCBtYXJrZWQpKQogICAgeTIgPSBucC5jbGlwKHkyLCAwLCAyNTUpCiAgICAjIHB1dCB0aGUgbWFya2VkIGx1bWEgYmFjaywga2VlcGluZyBjb2xvdXIKICAgIHJnYiA9IG5wLmFzYXJyYXkocmdiLCBucC5mbG9hdDY0KQogICAgZ2FpbmVkID0geTIgLSB5CiAgICBvdXQgPSByZ2IgKyBnYWluZWRbLi4uLCBOb25lXQogICAgcmV0dXJuIG5wLmNsaXAobnAucm91bmQob3V0KSwgMCwgMjU1KS5hc3R5cGUobnAudWludDgpCgoKZGVmIHJlYWQocmdiOiBucC5uZGFycmF5LCBrZXk6IGJ5dGVzLCBpbWFnZV9pZDogaW50LAogICAgICAgICBkZWx0YTogZmxvYXQgPSBERUxUQSwgYW1wOiBmbG9hdCB8IE5vbmUgPSBOb25lKSAtPiBkaWN0W3N0ciwgbnAubmRhcnJheV06CiAgICAiIiJQcm9kdWNlIHRoZSB0d28gZXZpZGVuY2UgbWFwcyB0aGUgbmV0d29yayBjb25zdW1lcy4iIiIKICAgIGEgPSBkZWx0YSAvIDQuMCBpZiBhbXAgaXMgTm9uZSBlbHNlIGZsb2F0KGFtcCkKICAgIHkgPSBsdW1hKHJnYikKICAgIGMgPSBoYWFyMih5KQogICAgZmxhdCA9IGRldGFpbHMoYykKICAgIHIgPSBmbGF0IC0gbnAucm91bmQoZmxhdCAvIGRlbHRhKSAqIGRlbHRhICAgICAgICAgIyByZXNpZHVhbCBmcm9tIHRoZSBsYXR0aWNlCiAgICB0ID0gdGFncyhrZXksIGltYWdlX2lkLCBmbGF0LCBjWyJMTDIiXSwgZGVsdGEpCiAgICBzID0gMi4wICogdC5hc3R5cGUobnAuZmxvYXQ2NCkgLSAxLjAKCiAgICBudW0gPSAociAqIHMpLnN1bShheGlzPTIpCiAgICBkZW4gPSBucC5tYXhpbXVtKG5wLmFicyhyKS5zdW0oYXhpcz0yKSwgMWUtOSkKICAgIGNvcnIgPSBudW0gLyBkZW4gICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICMgMSA9IGV4YWN0IHRhZywgMCA9IG5vbmUKICAgIHN1cnZpdmFsID0gKG5wLmFicyhyKSA+IDAuNSAqIGEpLm1lYW4oYXhpcz0yKSAgICAgICMgZnJhY3Rpb24gb2YgcmVhZGFibGUgc2xvdHMKICAgIHJldHVybiB7ImNvcnIiOiBjb3JyLCAiRSI6IDEuMCAtIGNvcnIsICJTIjogc3Vydml2YWx9CgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIHJpY2hlciByZWFkCgpCQU5EX1NMSUNFUyA9ICgoMCwgMTYpLCAoMTYsIDMyKSwgKDMyLCA0OCksICg0OCwgNTIpLCAoNTIsIDU2KSwgKDU2LCA2MCkpCkJBTkRfTkFNRVMgPSAoIkxIMSIsICJITDEiLCAiSEgxIiwgIkxIMiIsICJITDIiLCAiSEgyIikKCgpkZWYgcmVhZF9yaWNoKHJnYiwga2V5LCBpbWFnZV9pZCwgZGVsdGE9REVMVEEsIGFtcD1Ob25lKToKICAgICIiIkxpa2UgYHJlYWRgLCBidXQga2VlcHMgdGhlIHBlci1iYW5kIHN0cnVjdHVyZSBpbnN0ZWFkIG9mIGNvbGxhcHNpbmcgdG8gdHdvCiAgICBzY2FsYXJzLgoKICAgIFdoeTogYHJlYWRgIHRocm93cyBhd2F5IDU4IG9mIHRoZSA2MCBwZXItYmxvY2sgb2JzZXJ2YXRpb25zLiBNZWFzdXJpbmcKICAgIGxpbmVhciBzZXBhcmFiaWxpdHkgb2YgdGFtcGVyZWQgdnMgdW50b3VjaGVkIGJsb2NrcywgdGhlIGNvbGxhcHNlZCBwYWlyCiAgICAoY29yciwgc3Vydml2YWwpIHJlYWNoZWQgMC45OTAgaGVsZC1vdXQgQVVDIGFuZCB0aGUgcmljaGVyIHNldCByZWFjaGVkCiAgICAwLjk5NS4gVGhhdCBpcyBhIHNtYWxsIGdhaW4sIG5vdCBhIGxhcmdlIG9uZSAtLSB0aGUga2V5ZWQgZXZpZGVuY2UgaXMKICAgIGFscmVhZHkgaGlnaGx5IHNlcGFyYWJsZSAtLSBidXQgaXQgaXMgZnJlZSwgYW5kIGEgZG93bnN0cmVhbSBuZXR3b3JrIGNhbgogICAgb25seSB1c2Ugd2hhdCBpdCBpcyBnaXZlbi4KCiAgICBSZXR1cm5zLCBhbGwgKG55LCBueCk6CiAgICAgICAgY29yciAgICAgICB3aG9sZS1ibG9jayB0YWcgYWdyZWVtZW50ICAgICAgKGhpZ2ggPSBpbnRhY3QpCiAgICAgICAgc3VydiAgICAgICBmcmFjdGlvbiBvZiByZWFkYWJsZSBzbG90cyAgICAgKGxvdyA9IG1hcmsgZGVjYXllZCkKICAgICAgICBwZWFrICAgICAgIGxhcmdlc3QgfHJlc2lkdWFsfCBpbiBibG9jayAgICAoQVVDIDAuOTAgdG93YXJkIHRhbXBlcmVkKQogICAgICAgIG1hZyAgICAgICAgbWVhbiB8cmVzaWR1YWx8CiAgICAgICAgYmFuZF8qICAgICA2IHBlci1iYW5kIGFncmVlbWVudHMKICAgICIiIgogICAgYSA9IGRlbHRhIC8gNC4wIGlmIGFtcCBpcyBOb25lIGVsc2UgZmxvYXQoYW1wKQogICAgeSA9IGx1bWEocmdiKQogICAgYyA9IGhhYXIyKHkpCiAgICBmbGF0ID0gZGV0YWlscyhjKQogICAgciA9IGZsYXQgLSBucC5yb3VuZChmbGF0IC8gZGVsdGEpICogZGVsdGEKICAgIHQgPSB0YWdzKGtleSwgaW1hZ2VfaWQsIGZsYXQsIGNbIkxMMiJdLCBkZWx0YSkKICAgIHMgPSAyLjAgKiB0LmFzdHlwZShucC5mbG9hdDY0KSAtIDEuMAoKICAgIGNvcnIgPSAociAqIHMpLnN1bShheGlzPTIpIC8gbnAubWF4aW11bShucC5hYnMocikuc3VtKGF4aXM9MiksIDFlLTkpCiAgICBvdXQgPSB7CiAgICAgICAgImNvcnIiOiBjb3JyLAogICAgICAgICJFIjogMS4wIC0gY29yciwKICAgICAgICAic3VydiI6IChucC5hYnMocikgPiAwLjUgKiBhKS5tZWFuKGF4aXM9MiksCiAgICAgICAgInBlYWsiOiBucC5hYnMocikubWF4KGF4aXM9MiksCiAgICAgICAgIm1hZyI6IG5wLmFicyhyKS5tZWFuKGF4aXM9MiksCiAgICB9CiAgICBmb3IgbmFtZSwgKGxvLCBoaSkgaW4gemlwKEJBTkRfTkFNRVMsIEJBTkRfU0xJQ0VTKToKICAgICAgICBvdXRbImJhbmRfIiArIG5hbWVdID0gKChyWzosIDosIGxvOmhpXSAqIHNbOiwgOiwgbG86aGldKS5zdW0oYXhpcz0yKQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgLyBucC5tYXhpbXVtKG5wLmFicyhyWzosIDosIGxvOmhpXSkuc3VtKGF4aXM9MiksIDFlLTkpKQogICAgcmV0dXJuIG91dAoKClJJQ0hfQ0hBTk5FTFMgPSAoIkUiLCAic3VydiIsICJwZWFrIiwgIm1hZyIsCiAgICAgICAgICAgICAgICAgImJhbmRfTEgxIiwgImJhbmRfSEwxIiwgImJhbmRfSEgxIiwgImJhbmRfTEgyIiwgImJhbmRfSEwyIiwgImJhbmRfSEgyIikKCgpkZWYgcmljaF90ZW5zb3IocmdiLCBrZXksIGltYWdlX2lkLCAqKmt3KToKICAgICIiIigxMCwgbnksIG54KSBmbG9hdDMyIHN0YWNrIGluIHRoZSBvcmRlciBnaXZlbiBieSBSSUNIX0NIQU5ORUxTLiIiIgogICAgZCA9IHJlYWRfcmljaChyZ2IsIGtleSwgaW1hZ2VfaWQsICoqa3cpCiAgICByZXR1cm4gbnAuc3RhY2soW2RbY10gZm9yIGMgaW4gUklDSF9DSEFOTkVMU10pLmFzdHlwZShucC5mbG9hdDMyKQoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBkdWFsIGNhcnJpZXIKIwojIEV2ZXJ5dGhpbmcgYWJvdmUgaXMgQ2FycmllciBBIGFsb25lOiB0aGUgZnJhZ2lsZSBtYXJrLiBUaGUgcmVmZXJlbmNlIHN5c3RlbQojIGVtYmVkcyB0d28gbWFya3MsIGFuZCBpdHMgb3duIGFibGF0aW9uIHByaWNlcyB0aGUgc2Vjb25kIG9uZSBhdCAtMC4wNjcgSW9VIHdoZW4KIyByZW1vdmVkIC0tIG1vcmUgdGhhbiBpdHMgYXR0ZW50aW9uIG1vZHVsZSBjb3N0cy4gYGNhcnJpZXJzLnB5YCBpbXBsZW1lbnRzIGl0LgojCiMgVGhlIG9yZGVyIGJlbG93IGlzIE5PVCBpbmNpZGVudGFsLiBTZWUgYW1lbmRtZW50IEEtMDAyIGFuZCBjYXJyaWVycy5weS4KCmZyb20gLiBpbXBvcnQgY2FycmllcnMgYXMgX2NhcnJpZXJzICAgICAgIyBub3FhOiBFNDAyCgojIFR3byBuYW1lZCBwcm90ZWN0aW9uIHByb2ZpbGVzLiBUaGUgc2luZ2xlLWNhcnJpZXIgcHJvZmlsZSBpcyB0aGUgb25lIHRoZQojIG1lYXN1cmVkIGNsYXNzaWNhbCByZXN1bHQgd2FzIHByb2R1Y2VkIHVuZGVyIGFuZCBtdXN0IHN0YXkgYml0LXJlcHJvZHVjaWJsZTsKIyB0aGUgZHVhbC1jYXJyaWVyIHByb2ZpbGUgaXMgdGhlIGJhbGFuY2VkIGNvbmZpZ3VyYXRpb24gYWJvdmUuIEFkZGluZyBDYXJyaWVyIEIKIyBmb3JjZWQgYSByZWJhbGFuY2Ugb2YgQ2FycmllciBBJ3MgYW1wbGl0dWRlLCBhbmQgdGhhdCByZWJhbGFuY2UgaXMgYSBtZWFzdXJlZAojIGRlY2lzaW9uLCBub3QgYSB0dW5pbmcgY29udmVuaWVuY2UgLS0gc2VlIHRoZSBzd2VlcCBpbiBjYXJyaWVycy5weS4KUFJPVEVDVF9QUk9GSUxFUyA9IHsKICAgICJzaW5nbGUiOiB7ImRlbHRhIjogREVMVEEsICJiZXRhIjogTm9uZX0sICAgICAjIENhcnJpZXIgQSBvbmx5OyB0aGUgbWVhc3VyZWQgQzAKICAgICJkdWFsIjogICB7ImRlbHRhIjogNC4wLCAgICJiZXRhIjogMy4wfSwgICAgICAjIGJvdGggY2FycmllcnM7IFBTTlIgNDAuNzYgZEIKfQpERUZBVUxUX1BST0ZJTEUgPSAiZHVhbCIKClNFTUlfQ0hBTk5FTFMgPSAoInNlbWkiLCAic2VtaV9hYnMiLCAic2VtaV9zdXJ2IikKRlVMTF9DSEFOTkVMUyA9IFJJQ0hfQ0hBTk5FTFMgKyBTRU1JX0NIQU5ORUxTICAgICAgICAgICMgMTMgY2hhbm5lbHMKTl9GVUxMID0gbGVuKEZVTExfQ0hBTk5FTFMpCgoKZGVmIF9wcm9maWxlKHByb2ZpbGU6IHN0ciwgZGVsdGE6IGZsb2F0IHwgTm9uZSwgYmV0YTogZmxvYXQgfCBOb25lKToKICAgICIiIlJlc29sdmUgYSBuYW1lZCBwcm9maWxlLCBsZXR0aW5nIGV4cGxpY2l0IGFyZ3VtZW50cyBvdmVycmlkZSBpdC4iIiIKICAgIGlmIHByb2ZpbGUgbm90IGluIFBST1RFQ1RfUFJPRklMRVM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmInVua25vd24gcHJvZmlsZSB7cHJvZmlsZSFyfTsgIgogICAgICAgICAgICAgICAgICAgICAgICAgZiJjaG9vc2UgZnJvbSB7c29ydGVkKFBST1RFQ1RfUFJPRklMRVMpfSIpCiAgICBwID0gUFJPVEVDVF9QUk9GSUxFU1twcm9maWxlXQogICAgcmV0dXJuIChwWyJkZWx0YSJdIGlmIGRlbHRhIGlzIE5vbmUgZWxzZSBmbG9hdChkZWx0YSksCiAgICAgICAgICAgIHBbImJldGEiXSBpZiBiZXRhIGlzIE5vbmUgZWxzZSBmbG9hdChiZXRhKSkKCgpkZWYgcHJvdGVjdChyZ2I6IG5wLm5kYXJyYXksIGtleTogYnl0ZXMsIGltYWdlX2lkOiBpbnQsCiAgICAgICAgICAgIHByb2ZpbGU6IHN0ciA9IERFRkFVTFRfUFJPRklMRSwKICAgICAgICAgICAgZGVsdGE6IGZsb2F0IHwgTm9uZSA9IE5vbmUsIGJldGE6IGZsb2F0IHwgTm9uZSA9IE5vbmUpIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJBcHBseSBib3RoIGNhcnJpZXJzIGluIHRoZSBvbmx5IG9yZGVyIHRoYXQgd29ya3MuCgogICAgICAgIENhcnJpZXIgQiAoc2VtaS1yb2J1c3QsIERDVCkgIC0+ICBDYXJyaWVyIEEgKGZyYWdpbGUsIEhhYXIpCgogICAgQiBmaXJzdCBiZWNhdXNlIGl0IGlzIHRoZSBsYXJnZXIgcGVydHVyYmF0aW9uIChiZXRhID0gNCBwZXIgY29lZmZpY2llbnQKICAgIGFnYWluc3QgQSdzIERlbHRhLzQgPSAyKSwgYW5kIGJlY2F1c2UgQSdzIHRhZyBpcyBhIGZ1bmN0aW9uIG9mIHRoZSBibG9jaydzCiAgICBvd24gZGV0YWlsIGNvZWZmaWNpZW50cyAtLSBjb21wdXRpbmcgdGhhdCBmdW5jdGlvbiBhZnRlciBCIGlzIHdyaXR0ZW4gYmluZHMKICAgIHRoZSB0YWcgdG8gdGhlIGNvbnRlbnQgYXMgaXQgd2lsbCBhY3R1YWxseSBiZSBzdG9yZWQuIFdyaXRpbmcgQiBhZnRlcndhcmRzCiAgICB3b3VsZCBsZWF2ZSBldmVyeSB0YWcgc3RhbGUgYnkgZXhhY3RseSBCJ3MgY29udHJpYnV0aW9uLgogICAgIiIiCiAgICBkLCBiID0gX3Byb2ZpbGUocHJvZmlsZSwgZGVsdGEsIGJldGEpCiAgICBtYXJrZWQgPSBfY2FycmllcnMuZW1iZWQocmdiLCBrZXksIGltYWdlX2lkLCBiZXRhPV9jYXJyaWVycy5CRVRBIGlmIGIgaXMgTm9uZSBlbHNlIGIpCiAgICByZXR1cm4gZW1iZWQobWFya2VkLCBrZXksIGltYWdlX2lkLCBkZWx0YT1kKQoKCmRlZiByZWFkX2Z1bGwocmdiOiBucC5uZGFycmF5LCBrZXk6IGJ5dGVzLCBpbWFnZV9pZDogaW50LAogICAgICAgICAgICAgIHByb2ZpbGU6IHN0ciA9IERFRkFVTFRfUFJPRklMRSwKICAgICAgICAgICAgICBkZWx0YTogZmxvYXQgfCBOb25lID0gTm9uZSwKICAgICAgICAgICAgICBiZXRhOiBmbG9hdCB8IE5vbmUgPSBOb25lKSAtPiBkaWN0W3N0ciwgbnAubmRhcnJheV06CiAgICAiIiJBbGwgMTMgZXZpZGVuY2UgY2hhbm5lbHM6IDEwIGZyYWdpbGUgKyAzIHNlbWktcm9idXN0LgoKICAgIFRoZSB0aHJlZSBzZW1pIGNoYW5uZWxzIGFuc3dlciBhIHF1ZXN0aW9uIHRoZSBmcmFnaWxlIHRlbiBjYW5ub3Q6CgogICAgICAgIHNlbWkgY29sbGFwc2VzLCBzZW1pX2FicyBjb2xsYXBzZXMgICAtPiBmbGF0dGVuZWQgYnkgYSBjb2RlYyAgKGJlbmlnbikKICAgICAgICBzZW1pIGNvbGxhcHNlcywgc2VtaV9hYnMgc3RheXMgaGlnaCAgLT4gY29udGVudCB3YXMgcmV3cml0dGVuICh0YW1wZXJpbmcpCgogICAgVGhhdCBzZXBhcmF0aW9uIGlzIHRoZSByZWFzb24gdGhlIHNlY29uZCBjYXJyaWVyIGV4aXN0cy4gSXQgaXMgYSBmZWF0dXJlIGZvcgogICAgdGhlIGxlYXJuZWQgcmVhZGVyLCBuZXZlciBhIGRlY2lzaW9uIHJ1bGUgKGNvbnRyYXN0IEEtMDMyKS4KICAgICIiIgogICAgZHQsIGJ0ID0gX3Byb2ZpbGUocHJvZmlsZSwgZGVsdGEsIGJldGEpCiAgICBvdXQgPSByZWFkX3JpY2gocmdiLCBrZXksIGltYWdlX2lkLCBkZWx0YT1kdCkKICAgIG91dC51cGRhdGUoX2NhcnJpZXJzLnJlYWQocmdiLCBrZXksIGltYWdlX2lkLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICBiZXRhPV9jYXJyaWVycy5CRVRBIGlmIGJ0IGlzIE5vbmUgZWxzZSBidCkpCiAgICByZXR1cm4gb3V0CgoKZGVmIGZ1bGxfdGVuc29yKHJnYjogbnAubmRhcnJheSwga2V5OiBieXRlcywgaW1hZ2VfaWQ6IGludCwKICAgICAgICAgICAgICAgICoqa3cpIC0+IG5wLm5kYXJyYXk6CiAgICAiIiIoMTMsIG55LCBueCkgZmxvYXQzMiBpbiB0aGUgb3JkZXIgZ2l2ZW4gYnkgRlVMTF9DSEFOTkVMUy4iIiIKICAgIGQgPSByZWFkX2Z1bGwocmdiLCBrZXksIGltYWdlX2lkLCAqKmt3KQogICAgcmV0dXJuIG5wLnN0YWNrKFtkW2NdIGZvciBjIGluIEZVTExfQ0hBTk5FTFNdKS5hc3R5cGUobnAuZmxvYXQzMikK', 'carriers': 'IiIiQ2FycmllciBCOiB0aGUgc2VtaS1yb2J1c3QgRENUIHdhdGVybWFyay4KCldoeSB0aGlzIG1vZHVsZSBleGlzdHMKLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQoKVGhlIHJlZmVyZW5jZSBzeXN0ZW0gZW1iZWRzICoqdHdvKiogbWFya3M6IGEgZnJhZ2lsZSBvbmUgaW4gd2F2ZWxldCBkZXRhaWxzIGFuZCBhCnNlbWktcm9idXN0IG9uZSBpbiBtaWQtZnJlcXVlbmN5IERDVC4gSXRzIG93biBhYmxhdGlvbiBwcmljZXMgdGhlIHNlY29uZCBjYXJyaWVyCmF0ICoqLTAuMDY3IElvVSoqIHdoZW4gcmVtb3ZlZCAtLSBtb3JlIHRoYW4gaXRzIGF0dGVudGlvbiBtb2R1bGUgKC0wLjAzMSkgY29zdHMuCldlIGhhZCBvbmx5IHRoZSBmcmFnaWxlIGNhcnJpZXIgYW5kIHN1YnN0aXR1dGVkIGZvdXIgaGFuZC1idWlsdCBkZWdyYWRhdGlvbgpzY2FsYXJzIGluIGl0cyBwbGFjZS4gVGhvc2Ugc2NhbGFycyBlc3RpbWF0ZSAqaG93IG11Y2gqIGFuIGltYWdlIHdhcyBjb21wcmVzc2VkOwphIGNhcnJpZXIgZGVzaWduZWQgdG8gc3Vydml2ZSBjb21wcmVzc2lvbiBhbnN3ZXJzIHRoZSBxdWVzdGlvbiB0aGF0IGFjdHVhbGx5Cm1hdHRlcnM6ICoqZGlkIHRoaXMgc3BlY2lmaWMgYmxvY2sga2VlcCBpdHMgbWFyayB0aHJvdWdoIHRoZSBjb2RlYz8qKgoKVGhhdCBkaXN0aW5jdGlvbiBpcyB0aGUgd2hvbGUgcG9pbnQuIEEgSlBFRyBxdWFsaXR5IGVzdGltYXRlIGlzIGdsb2JhbCAtLSBpdCBzYXlzCiJ0aGlzIGltYWdlIGlzIFE3NSIuIFRoZSBzZW1pLXJvYnVzdCByZXNwb25zZSBpcyBwZXItYmxvY2sgLS0gaXQgc2F5cyAidGhpcyBibG9jawpzdGlsbCBjYXJyaWVzIGl0cyBtYXJrIGF0IFE3NSwgYW5kIHRoYXQgb25lIGRvZXMgbm90Ii4gVGhlIGZpcnN0IGNhbm5vdCBsb2NhbGlzZTsKdGhlIHNlY29uZCBjYW4uCgpXaGF0IGl0IGlzIG5vdAotLS0tLS0tLS0tLS0tLQoKVGhpcyBpcyAqKm5vdCoqIGEgc2Vjb25kIGRlY2lzaW9uIHJ1bGUuIEluIHRoZSBjbGFzc2ljYWwgcGhhc2UgYSBjb25mbGljdC1vbmx5CnNlY29uZCBydWxlIGNvdWxkIG5vdCBwcm9kdWNlIGEgVEFNUEVSRUQgdmVyZGljdCwgYW5kIGl0IHdhcyBkaXNhYmxlZCAoQS0wMzIpLgpUaGF0IHJlYXNvbmluZyBkb2VzIG5vdCB0cmFuc2ZlcjogaGVyZSBDYXJyaWVyIEIgaXMgYW4gKmlucHV0IGZlYXR1cmUqIHRvIHRoZQpsZWFybmVkIHJlYWRlciwgbm90IGEgZGVjaXNpb24uIEl0IHN1cHBsaWVzIHRoZSBldmlkZW5jZSB0aGF0IHNlcGFyYXRlcwoKICAgICJ0aGlzIGJsb2NrIGxvc3QgaXRzIG1hcmsgYmVjYXVzZSBhIGNvZGVjIGRlc3Ryb3llZCBpdCIgICAoYmVuaWduKQogICAgInRoaXMgYmxvY2sgbG9zdCBpdHMgbWFyayBiZWNhdXNlIHNvbWVvbmUgcmV3cm90ZSBpdCIgICAgICAodGFtcGVyaW5nKQoKd2hpY2ggaXMgZXhhY3RseSB0aGUgZGlzdGluY3Rpb24gdGhlIGZyYWdpbGUgY2FycmllciBhbG9uZSBjYW5ub3QgbWFrZS4KCk9yZGVyaW5nCi0tLS0tLS0tCgpDYXJyaWVyIEIgaXMgd3JpdHRlbiAqKmJlZm9yZSoqIENhcnJpZXIgQSAoYW1lbmRtZW50IEEtMDAyKS4gVHdvIHJlYXNvbnM6CgogICogQiBpcyB0aGUgbGFyZ2VyIHBlcnR1cmJhdGlvbiAoYmV0YSA9IDggcGVyIGNvZWZmaWNpZW50IGFnYWluc3QgQSdzIERlbHRhLzQgPQogICAgMiksIHNvIHdyaXRpbmcgQiBzZWNvbmQgd291bGQgY29ycnVwdCBBIGV4YWN0bHkgd2hlcmUgQSBpcyBtb3N0IGRlbGljYXRlLgogICogQSdzIHRhZyBpcyBhIGZ1bmN0aW9uIG9mIHRoZSBibG9jaydzIG93biBkZXRhaWwgY29lZmZpY2llbnRzLiBDb21wdXRpbmcgdGhhdAogICAgZnVuY3Rpb24gKmFmdGVyKiBCIGlzIHdyaXR0ZW4gbWVhbnMgdGhlIHRhZyBiaW5kcyB0aGUgY29udGVudCBhcyBpdCB3aWxsCiAgICBhY3R1YWxseSBiZSBzdG9yZWQgLS0gaW5jbHVkaW5nIEIncyBtYXJrLiBXcml0aW5nIEIgYWZ0ZXJ3YXJkcyB3b3VsZCBsZWF2ZQogICAgZXZlcnkgYmxvY2sncyB0YWcgc3RhbGUgYnkgZXhhY3RseSBCJ3MgY29udHJpYnV0aW9uLgoKQ29uc3RhbnRzIGFyZSBgTUVBU1VSRURgIG9yIGBDSE9TRU5gIGFuZCBjYXJyeSBwcm92ZW5hbmNlLCBwZXIgYGNvbmZpZy5weWAgaW4gdGhlCmBkZXZgIGJyYW5jaC4gYEJFVEFgIHdhcyBjaG9zZW4gYnkgdGhlIHN3ZWVwIGluIGB0ZXN0cy90ZXN0X2NhcnJpZXJzLnB5YC4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgaGFzaGxpYgoKaW1wb3J0IG51bXB5IGFzIG5wCmZyb20gc2NpcHkuZmZ0IGltcG9ydCBkY3RuLCBpZGN0bgoKQkxPQ0sgPSA4CgojIHUgKyB2IGluIFszLCA2XSBvdmVyIGFuIDh4OCBibG9jazogNCArIDUgKyA2ICsgNyA9IDIyIGNvZWZmaWNpZW50cy4KIyBMb3cgZW5vdWdoIHRvIHN1cnZpdmUgSlBFRywgaGlnaCBlbm91Z2ggdGhhdCB0aGUgREMgdGVybSBhbmQgdGhlIGZpcnN0CiMgZGlhZ29uYWxzICh3aGljaCBjYXJyeSBtb3N0IG9mIHRoZSBpbWFnZSBlbmVyZ3kpIGFyZSBsZWZ0IGFsb25lLgpTRU1JX0NPRUZGUzogdHVwbGVbdHVwbGVbaW50LCBpbnRdLCAuLi5dID0gdHVwbGUoCiAgICAodSwgdikgZm9yIHUgaW4gcmFuZ2UoQkxPQ0spIGZvciB2IGluIHJhbmdlKEJMT0NLKSBpZiAzIDw9IHUgKyB2IDw9IDYKKQphc3NlcnQgbGVuKFNFTUlfQ09FRkZTKSA9PSAyMiwgbGVuKFNFTUlfQ09FRkZTKQoKIyBNRUFTVVJFRCwgbm90IGNob3Nlbi4gU3dlcHQgb24gMjAyNi0xMS0wMiBvdmVyIGJldGEgaW4gezIsMyw0LDUsNiw4fSBvbiBhCiMgMjU2XjIgcHJvY2VkdXJhbCBmaWVsZCwgc2NvcmluZyBQU05SIGFuZCB0aGUgbWVhbiBzZW1pIHJlc3BvbnNlIGFmdGVyIEpQRUc6CiMKIyAgIGJldGEgIFBTTlIgICBjbGVhbiAgIFE5NSAgIFE5MCAgIFE4NSAgIFE3NSAgIFE2MCAgIFE1MCAgIHdyb25nLWtleQojICAgIDIuMCAgNDYuNTAgICAgMC45OSAgMS4wMSAgMS4wMCAgMC45MiAgMC42OCAgMC4zOSAgMC4yMyAgICAgLTAuMDExCiMgICAgMy4wICA0My4xMSAgICAxLjAwICAxLjAxICAxLjAwICAwLjk0ICAwLjcxICAwLjQzICAwLjI2ICAgICAtMC4wMDkKIyAgICA0LjAgIDQwLjY2ICAgIDEuMDAgIDEuMDEgIDEuMDAgIDAuOTUgIDAuNzQgIDAuNDYgIDAuMzAgICAgIC0wLjAwOSAgIDwtLSBrZXB0CiMgICAgNS4wICAzOC43NSAgICAxLjAwICAxLjAxICAxLjAxICAwLjk3ICAwLjc3ICAwLjUxICAwLjM2ICAgICAtMC4wMDkKIyAgICA2LjAgIDM3LjE4ICAgIDEuMDAgIDEuMDAgIDEuMDEgIDAuOTkgIDAuODEgIDAuNTUgIDAuNDEgICAgIC0wLjAwOAojICAgIDguMCAgMzQuNjkgICAgMS4wMCAgMS4wMSAgMS4wMSAgMS4wMSAgMC45MCAgMC42NyAgMC41MiAgICAgLTAuMDA4CiMKIyBiZXRhID0gNC4wIHdhcyB0aGUga25lZSBmb3IgQ2FycmllciBCICphbG9uZSogKDQwLjY2IGRCKS4gV2l0aCBib3RoIGNhcnJpZXJzCiMgc3RhY2tlZCB0aGUgZmlkZWxpdHkgbG9zcyBjb21wb3VuZHMsIHNvIHRoZSBwYWlyIHdhcyBzd2VwdCBqb2ludGx5IG9uIHRoZSBzYW1lCiMgZGF0ZS4gRSBhbmQgc2VtaSBhcmUgbWVhbnMgb3ZlciBibG9ja3M7ICJzZXAiIGlzIHRoZSB0YW1wZXJlZC1taW51cy1pbnRhY3QKIyBzZXBhcmF0aW9uIG9uIGEgYmxvY2stYWxpZ25lZCBwYXN0ZToKIwojICAgZGVsdGEgIGJldGEgICBQU05SICAgaW50YWN0IEUvc2VtaSAgIFE5MCBFL3NlbWkgICBRNzUgRS9zZW1pCiMgICAgICAgOCAgICAgNCAgIDM2LjU1ICAgMC4wMSAvIDEuMDAgICAgMC43MyAvIDEuMDAgICAwLjkzIC8gMC44MgojICAgICAgIDggICAgIDMgICAzNy4zNCAgIDAuMDEgLyAxLjAwICAgIDAuNzIgLyAxLjAxICAgMC45MyAvIDAuODAKIyAgICAgICA2ICAgICA0ICAgMzcuODkgICAwLjAxIC8gMS4wMCAgICAwLjg0IC8gMS4wMSAgIDAuOTYgLyAwLjc5CiMgICAgICAgNiAgICAgMyAgIDM5LjAxICAgMC4wMSAvIDEuMDAgICAgMC44NCAvIDEuMDAgICAwLjk3IC8gMC43NgojICAgICAgIDQgICAgIDQgICAzOS4xNyAgIDAuMDAgLyAxLjAwICAgIDAuOTMgLyAxLjAxICAgMC45OCAvIDAuNzYKIyAgICAgICA0ICAgICAzICAgNDAuNzYgICAwLjAwIC8gMS4wMCAgICAwLjkzIC8gMS4wMSAgIDAuOTkgLyAwLjc0ICAgPC0tIGtlcHQKIwojIEF0IGRlbHRhID0gNCwgYmV0YSA9IDMgdGhlIGR1YWwtY2FycmllciBzeXN0ZW0gcmVhY2hlcyA0MC43NiBkQiAtLSBtYXRjaGluZyB0aGUKIyByZWZlcmVuY2Ugc3lzdGVtJ3MgcmVwb3J0ZWQgNDAuNDcgZEIgLS0gd2hpbGUgcmV0YWluaW5nIEUgc2VwYXJhdGlvbiAxLjAwNyBhbmQKIyBzZW1pIHNlcGFyYXRpb24gMS4wMzAgb24gYSBibG9jay1hbGlnbmVkIHBhc3RlLiBJdCBpcyBzdHJpY3RseSBiZXR0ZXIgdGhhbiB0aGUKIyBzdHJvbmdlciBzZXR0aW5nOiBpZGVudGljYWwgZGV0ZWN0aW9uLCA0LjIgZEIgYmV0dGVyIGZpZGVsaXR5LgojCiMgTG93ZXJpbmcgRGVsdGEgYWxzbyBtYWtlcyBDYXJyaWVyIEEgKmdlbnVpbmVseSogZnJhZ2lsZSwgd2hpY2ggaXMgdGhlIGRlc2lnbgojIGludGVudDogdGhlIGZyYWdpbGUgY2FycmllcidzIGpvYiBpcyB0byBicmVhayB1bmRlciBhbnkgZWRpdCwgYW5kIHRoZQojIHNlbWktcm9idXN0IGNhcnJpZXIncyBqb2IgaXMgdG8gc3Vydml2ZSB0aGUgY29kZWMuIFNlcGFyYXRpbmcgdGhvc2UgdHdvIHJvbGVzCiMgaXMgdGhlIHJlYXNvbiB0aGUgcmVmZXJlbmNlIGFyY2hpdGVjdHVyZSBoYXMgdHdvIGNhcnJpZXJzIGF0IGFsbC4KQkVUQSA9IDMuMApOX1NFTUkgPSBsZW4oU0VNSV9DT0VGRlMpCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIGJsb2NrIERDVAoKZGVmIF90b19ibG9ja3MoeTogbnAubmRhcnJheSwgYjogaW50ID0gQkxPQ0spIC0+IG5wLm5kYXJyYXk6CiAgICAiIiIoSCwgVykgLT4gKG55LCBueCwgYiwgYikuIFRoZSB0cmFuc3Bvc2UgaXMgbG9hZC1iZWFyaW5nOiBhIHBsYWluCiAgICByZXNoYXBlKG55LCBiLCBueCwgYikgbGVhdmVzIHRoZSBibG9jayBheGVzIGludGVybGVhdmVkIGFuZCBzY2F0dGVycyBvbmUKICAgIGJsb2NrJ3MgY29lZmZpY2llbnRzIGFjcm9zcyBzZXZlcmFsIGJsb2Nrcy4gVGhhdCBleGFjdCBidWcgaXMgcmVjb3JkZWQgaW4KICAgIHRoZSBSRUFETUU7IGl0IHJvdW5kLXRyaXBzIHBlcmZlY3RseSwgc28gb25seSBhIGxvY2FsaXR5IHRlc3QgZmluZHMgaXQuIiIiCiAgICBueSwgbnggPSB5LnNoYXBlWzBdIC8vIGIsIHkuc2hhcGVbMV0gLy8gYgogICAgdCA9IHlbOm55ICogYiwgOm54ICogYl0ucmVzaGFwZShueSwgYiwgbngsIGIpCiAgICByZXR1cm4gbnAuYXNjb250aWd1b3VzYXJyYXkodC50cmFuc3Bvc2UoMCwgMiwgMSwgMykpCgoKZGVmIF9mcm9tX2Jsb2NrcyhibDogbnAubmRhcnJheSwgYjogaW50ID0gQkxPQ0spIC0+IG5wLm5kYXJyYXk6CiAgICBueSwgbnggPSBibC5zaGFwZVs6Ml0KICAgIHQgPSBibC50cmFuc3Bvc2UoMCwgMiwgMSwgMykKICAgIHJldHVybiBucC5hc2NvbnRpZ3VvdXNhcnJheSh0LnJlc2hhcGUobnkgKiBiLCBueCAqIGIpKQoKCmRlZiBkY3RfYmxvY2tzKHk6IG5wLm5kYXJyYXkpIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJPcnRob25vcm1hbCA4eDggRENUIHBlciBibG9jayAtPiAobnksIG54LCA4LCA4KS4iIiIKICAgIHJldHVybiBkY3RuKF90b19ibG9ja3MoeSkuYXN0eXBlKG5wLmZsb2F0NjQpLCBheGVzPSgtMiwgLTEpLCBub3JtPSJvcnRobyIpCgoKZGVmIGlkY3RfYmxvY2tzKGM6IG5wLm5kYXJyYXkpIC0+IG5wLm5kYXJyYXk6CiAgICByZXR1cm4gX2Zyb21fYmxvY2tzKGlkY3RuKGMsIGF4ZXM9KC0yLCAtMSksIG5vcm09Im9ydGhvIikpCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIGtleWVkIHNpZ25zCgpkZWYgX3NpZ25zKGtleTogYnl0ZXMsIGltYWdlX2lkOiBpbnQsIG55OiBpbnQsIG54OiBpbnQpIC0+IG5wLm5kYXJyYXk6CiAgICAiIiIobnksIG54LCAyMikgb2YgKy8tMSwga2V5ZWQgYnkgdGhlIHNlY3JldCBhbmQgdGhlIGJsb2NrIGFkZHJlc3MuCgogICAgQWRkcmVzcy1rZXllZCBtZWFucyBhIGJsb2NrIHBhc3RlZCBmcm9tIGVsc2V3aGVyZSBpbiB0aGUgc2FtZSBpbWFnZSBtZWV0cyB0aGUKICAgIHNpZ25zIG9mIGl0cyAqbmV3KiBhZGRyZXNzLCBub3QgdGhlIG9uZXMgaXQgd2FzIG1hcmtlZCB3aXRoIC0tIHNvIHRoZQogICAgc2VtaS1yb2J1c3QgY2FycmllciBpcyBhbHNvIGEgKHdlYWtlcikgbG9jYWxpc2VyLCBub3QgbWVyZWx5IGEKICAgIGNvbXByZXNzaW9uIGRldGVjdG9yLgogICAgIiIiCiAgICBvdXQgPSBucC5lbXB0eSgobnksIG54LCBOX1NFTUkpLCBucC5mbG9hdDY0KQogICAgZm9yIGkgaW4gcmFuZ2UobnkpOgogICAgICAgIGZvciBqIGluIHJhbmdlKG54KToKICAgICAgICAgICAgaCA9IGhhc2hsaWIuYmxha2UyYigKICAgICAgICAgICAgICAgIGtleSArIGIifHNlbWl8IiArIGludChpbWFnZV9pZCkudG9fYnl0ZXMoOCwgImJpZyIpCiAgICAgICAgICAgICAgICArIGludChpKS50b19ieXRlcyg0LCAiYmlnIikgKyBpbnQoaikudG9fYnl0ZXMoNCwgImJpZyIpLAogICAgICAgICAgICAgICAgZGlnZXN0X3NpemU9MzIsCiAgICAgICAgICAgICkuZGlnZXN0KCkKICAgICAgICAgICAgYml0cyA9IG5wLnVucGFja2JpdHMobnAuZnJvbWJ1ZmZlcihoLCBucC51aW50OCkpWzpOX1NFTUldCiAgICAgICAgICAgIG91dFtpLCBqXSA9IDIuMCAqIGJpdHMgLSAxLjAKICAgIHJldHVybiBvdXQKCgpkZWYgX3ZlY3RvcmlzZWRfc2lnbnMoa2V5OiBieXRlcywgaW1hZ2VfaWQ6IGludCwgbnk6IGludCwgbng6IGludCkgLT4gbnAubmRhcnJheToKICAgICIiIlNhbWUgb3V0cHV0IGFzIGBfc2lnbnNgLCBjb21wdXRlZCBvbmUga2V5ZWQgc3RyZWFtIGF0IGEgdGltZS4KCiAgICBgaGFzaGxpYmAgaGFzIG5vIGJhdGNoIGludGVyZmFjZSwgc28gdGhlIGxvb3AgaXMgdW5hdm9pZGFibGU7IGl0IGlzIHdyaXR0ZW4KICAgIG9uY2UgaGVyZSBhbmQgdGhlIHJlZmVyZW5jZSBpbXBsZW1lbnRhdGlvbiBhYm92ZSBpcyBrZXB0IGZvciB0aGUgdGVzdCB0aGF0CiAgICBhc3NlcnRzIHRoZSB0d28gYWdyZWUuCiAgICAiIiIKICAgIHJldHVybiBfc2lnbnMoa2V5LCBpbWFnZV9pZCwgbnksIG54KQoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSB3cml0ZSAvIHJlYWQKCmRlZiB3cml0ZSh5OiBucC5uZGFycmF5LCBrZXk6IGJ5dGVzLCBpbWFnZV9pZDogaW50LCBiZXRhOiBmbG9hdCA9IEJFVEEpIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJBZGQgYSBrZXllZCArLy0gYmV0YSBiaWFzIHRvIHRoZSAyMiBzZW1pLXJvYnVzdCBjb2VmZmljaWVudHMgcGVyIGJsb2NrLgoKICAgIEFkZGl0aXZlIHJhdGhlciB0aGFuIHF1YW50aXNhdGlvbi1iYXNlZCBvbiBwdXJwb3NlLiBBIFFJTSBsYXR0aWNlIHdvdWxkIGdpdmUKICAgIGFuIGV4YWN0bHktcmVjb3ZlcmFibGUgaW5kZXggKGFzIENhcnJpZXIgQSBkb2VzKSBidXQgaXMgZGVzdHJveWVkIGJ5IGFueQogICAgY29lZmZpY2llbnQtZG9tYWluIHJlc2NhbGluZywgd2hpY2ggaXMgcHJlY2lzZWx5IHdoYXQgSlBFRyBkb2VzLiBBbiBhZGRpdGl2ZQogICAgYmlhcyBzdXJ2aXZlcyBxdWFudGlzYXRpb24gZm9yIGFzIGxvbmcgYXMgYmV0YSBleGNlZWRzIGhhbGYgdGhlIHF1YW50aXNhdGlvbgogICAgc3RlcCAtLSB3aGljaCBpcyB0aGUgcm9idXN0bmVzcyB0aGUgc2Vjb25kIGNhcnJpZXIgaXMgZm9yLgogICAgIiIiCiAgICBjID0gZGN0X2Jsb2Nrcyh5KQogICAgbnksIG54ID0gYy5zaGFwZVs6Ml0KICAgIHMgPSBfc2lnbnMoa2V5LCBpbWFnZV9pZCwgbnksIG54KQogICAgZm9yIGssICh1LCB2KSBpbiBlbnVtZXJhdGUoU0VNSV9DT0VGRlMpOgogICAgICAgIGNbOiwgOiwgdSwgdl0gKz0gYmV0YSAqIHNbOiwgOiwga10KICAgIHJldHVybiBpZGN0X2Jsb2NrcyhjKQoKCmRlZiByZWFkKHJnYjogbnAubmRhcnJheSwga2V5OiBieXRlcywgaW1hZ2VfaWQ6IGludCwKICAgICAgICAgYmV0YTogZmxvYXQgPSBCRVRBKSAtPiBkaWN0W3N0ciwgbnAubmRhcnJheV06CiAgICAiIiJQZXItYmxvY2sgcmVhZGluZ3Mgb2YgdGhlIHNlbWktcm9idXN0IG1hcmsuCgogICAgc2VtaSAgICAgICAgIG1lYW4oYyAqIHMpIC8gYmV0YS4gQWJvdXQgKzEgd2hlcmUgdGhlIG1hcmsgc3Vydml2ZWQsIGFib3V0IDAKICAgICAgICAgICAgICAgICB3aGVyZSBpdCBuZXZlciBleGlzdGVkIG9yIHdhcyBkZXN0cm95ZWQuIFNpZ25lZCBvbiBwdXJwb3NlOiBhCiAgICAgICAgICAgICAgICAgbmVnYXRpdmUgdmFsdWUgbWVhbnMgdGhlIGNvZWZmaWNpZW50cyBtb3ZlZCAqYWdhaW5zdCogdGhlIGtleSwKICAgICAgICAgICAgICAgICB3aGljaCBpcyBhIHN0cm9uZ2VyIHRhbXBlciBzaWduYWwgdGhhbiBtZXJlIGFic2VuY2UuCiAgICBzZW1pX2FicyAgICAgbWVhbih8Y3wpIG92ZXIgdGhlIG1hcmtlZCBjb2VmZmljaWVudHMuIFdoZW4gdGhpcyBjb2xsYXBzZXMKICAgICAgICAgICAgICAgICBhbG9uZ3NpZGUgYHNlbWlgLCB0aGUgYmxvY2sgd2FzIGZsYXR0ZW5lZCBieSBhIGNvZGVjOyB3aGVuIGl0CiAgICAgICAgICAgICAgICAgc3RheXMgaGlnaCBhbmQgYHNlbWlgIGNvbGxhcHNlcywgdGhlIGNvbnRlbnQgd2FzIHJld3JpdHRlbi4KICAgIHNlbWlfc3VydiAgICBmcmFjdGlvbiBvZiB0aGUgMjIgY29lZmZpY2llbnRzIHdob3NlIHNpZ24gc3RpbGwgYWdyZWVzIHdpdGgKICAgICAgICAgICAgICAgICB0aGUga2V5LiBDb2Fyc2VyIHRoYW4gYHNlbWlgIGFuZCBtb3JlIHJvYnVzdCBhdCBsb3cgcXVhbGl0eS4KICAgICIiIgogICAgcmdiID0gbnAuYXNhcnJheShyZ2IsIG5wLmZsb2F0NjQpCiAgICB5ID0gMC4yOTkgKiByZ2JbLi4uLCAwXSArIDAuNTg3ICogcmdiWy4uLiwgMV0gKyAwLjExNCAqIHJnYlsuLi4sIDJdCiAgICBjID0gZGN0X2Jsb2Nrcyh5KQogICAgbnksIG54ID0gYy5zaGFwZVs6Ml0KICAgIHMgPSBfc2lnbnMoa2V5LCBpbWFnZV9pZCwgbnksIG54KQoKICAgIHZhbHMgPSBucC5zdGFjayhbY1s6LCA6LCB1LCB2XSBmb3IgKHUsIHYpIGluIFNFTUlfQ09FRkZTXSwgYXhpcz0yKSAgICMgKG55LG54LDIyKQogICAgc2VtaSA9ICh2YWxzICogcykubWVhbihheGlzPTIpIC8gYmV0YQogICAgc2VtaV9hYnMgPSBucC5hYnModmFscykubWVhbihheGlzPTIpCiAgICBzZW1pX3N1cnYgPSAobnAuc2lnbih2YWxzKSA9PSBzKS5tZWFuKGF4aXM9MikKICAgIHJldHVybiB7InNlbWkiOiBzZW1pLCAic2VtaV9hYnMiOiBzZW1pX2FicywgInNlbWlfc3VydiI6IHNlbWlfc3Vydn0KCgpkZWYgZW1iZWQocmdiOiBucC5uZGFycmF5LCBrZXk6IGJ5dGVzLCBpbWFnZV9pZDogaW50LCBiZXRhOiBmbG9hdCA9IEJFVEEpIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJXcml0ZSBDYXJyaWVyIEIgaW50byBhbiBSR0IgaW1hZ2UsIHByZXNlcnZpbmcgY29sb3VyIGV4YWN0bHkuCgogICAgVGhlIG1hcmsgbGl2ZXMgaW4gbHVtYS4gVGhlIGRlbHRhIGFwcGxpZWQgdG8gbHVtYSBpcyBhZGRlZCBlcXVhbGx5IHRvIFIsIEcgYW5kCiAgICBCLCB3aGljaCBsZWF2ZXMgdGhlIGNocm9tYSBkaWZmZXJlbmNlcyB1bnRvdWNoZWQgdG8gd2l0aGluIHJvdW5kaW5nIC0tIHRoZQogICAgc2FtZSB0cmljayBDYXJyaWVyIEEgdXNlcywgZm9yIHRoZSBzYW1lIHJlYXNvbi4KICAgICIiIgogICAgcmdiID0gbnAuYXNhcnJheShyZ2IsIG5wLmZsb2F0NjQpCiAgICB5ID0gMC4yOTkgKiByZ2JbLi4uLCAwXSArIDAuNTg3ICogcmdiWy4uLiwgMV0gKyAwLjExNCAqIHJnYlsuLi4sIDJdCiAgICB5MiA9IHdyaXRlKHksIGtleSwgaW1hZ2VfaWQsIGJldGEpCiAgICBvdXQgPSByZ2IgKyAoeTIgLSB5KVsuLi4sIE5vbmVdCiAgICByZXR1cm4gbnAuY2xpcChucC5yb3VuZChvdXQpLCAwLCAyNTUpLmFzdHlwZShucC51aW50OCkK', 'tamper': 'IiIiVGFtcGVyIGFuZCBiZW5pZ24gZmFtaWxpZXMuIEVpZ2h0IG9mIHRoZW0sIG9uZSBpbnRlcmZhY2UuCgogICAgYXBwbHkocmdiLCBybmcsICoqa3cpIC0+IChjaGFuZ2VkX3JnYiwgYmxvY2tfbWFzaykKClRoZSBibG9jayBtYXNrIGlzIEFMV0FZUyBtZWFzdXJlZCBieSBkaWZmZXJlbmNpbmcgdGhlIHBpeGVscy4gQSBmYW1pbHkgdGhhdApyZXR1cm5zIGEgZGVjbGFyZWQgcmVjdGFuZ2xlIGlzIGEgYnVnOiBvbiBQSDIgdGhlIGRlY2xhcmVkIHJlZ2lvbiBvZiBjb25kaXRpb24gNApjb250YWluZWQgYmxvY2tzIHRoYXQgbmV2ZXIgY2hhbmdlZCBhIHBpeGVsLCBhbmQgc2NvcmluZyBhZ2FpbnN0IGl0IHJld2FyZGVkIHRoZQp3cm9uZyBhbnN3ZXIuCgpFdmVyeSBmYW1pbHkgaXMgYmxvY2stYWxpZ25lZCB3aGVyZSB0aGUgY2xhc3NpY2FsIHByb3RvY29sIHdhcyBibG9jay1hbGlnbmVkLCBzbwp0aGUgOHg4IHZlcmRpY3QgZ3JpZCBsaW5lcyB1cCB3aXRoIHRoZSB0cnV0aC4gTm9uLWFsaWduZWQgZmFtaWxpZXMgKGJsdXIsIG5vaXNlKQphcmUgaGFuZGxlZCBieSBtZWFzdXJpbmcsIG5vdCBieSBhc3N1bWluZy4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgY3YyCmltcG9ydCBudW1weSBhcyBucAoKZnJvbSAubWV0cmljcyBpbXBvcnQgbWVhc3VyZWRfbWFzawoKQkxPQ0sgPSA4ClNJWkUgPSA1MTIKCgpkZWYgX3JuZyhzZWVkOiBpbnQpIC0+IG5wLnJhbmRvbS5HZW5lcmF0b3I6CiAgICByZXR1cm4gbnAucmFuZG9tLmRlZmF1bHRfcm5nKHNlZWQpCgoKZGVmIF9ibG9ja190b3BfbGVmdChybmcsIHNpemU6IGludCwgc3BhbjogaW50KSAtPiBpbnQ6CiAgICAiIiJBIGJsb2NrLWFsaWduZWQgb3JpZ2luIGxlYXZpbmcgYHNwYW5gIHBpeGVscyBvZiByb29tLiIiIgogICAgbiA9IG1heCgxLCAoc2l6ZSAtIHNwYW4pIC8vIEJMT0NLKQogICAgcmV0dXJuIGludChybmcuaW50ZWdlcnMoMCwgbikpICogQkxPQ0sKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gVDEgY29weS1tb3ZlCgpkZWYgY29weV9tb3ZlKHJnYjogbnAubmRhcnJheSwgcm5nOiBucC5yYW5kb20uR2VuZXJhdG9yLCBzcGFuOiBpbnQgPSAzMiwKICAgICAgICAgICAgICBuX2NvcGllczogaW50ID0gMSkgLT4gbnAubmRhcnJheToKICAgICIiIlBhc3RlIGEgYmxvY2stYWxpZ25lZCBzcXVhcmUgZnJvbSBvbmUgcGxhY2UgdG8gYW5vdGhlciBpbiB0aGUgc2FtZSBpbWFnZS4iIiIKICAgIG91dCA9IHJnYi5jb3B5KCkKICAgIGZvciBfIGluIHJhbmdlKG5fY29waWVzKToKICAgICAgICBzX3IgPSBfYmxvY2tfdG9wX2xlZnQocm5nLCBTSVpFLCBzcGFuKQogICAgICAgIHNfYyA9IF9ibG9ja190b3BfbGVmdChybmcsIFNJWkUsIHNwYW4pCiAgICAgICAgZm9yIF9hdHRlbXB0IGluIHJhbmdlKDIwKToKICAgICAgICAgICAgZF9yID0gX2Jsb2NrX3RvcF9sZWZ0KHJuZywgU0laRSwgc3BhbikKICAgICAgICAgICAgZF9jID0gX2Jsb2NrX3RvcF9sZWZ0KHJuZywgU0laRSwgc3BhbikKICAgICAgICAgICAgIyBrZWVwIHNvdXJjZSBhbmQgZGVzdGluYXRpb24gZGlzam9pbnQKICAgICAgICAgICAgaWYgYWJzKGRfciAtIHNfcikgPj0gc3BhbiBvciBhYnMoZF9jIC0gc19jKSA+PSBzcGFuOgogICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICBlbHNlOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIG91dFtkX3I6ZF9yICsgc3BhbiwgZF9jOmRfYyArIHNwYW5dID0gb3V0W3NfcjpzX3IgKyBzcGFuLCBzX2M6c19jICsgc3Bhbl0KICAgIHJldHVybiBvdXQKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gVDIgbGVzaW9uIGluc2VydGlvbgoKZGVmIGxlc2lvbl9pbnNlcnQocmdiOiBucC5uZGFycmF5LCBybmc6IG5wLnJhbmRvbS5HZW5lcmF0b3IsCiAgICAgICAgICAgICAgICAgIHJlZ2lvbjogbnAubmRhcnJheSB8IE5vbmUgPSBOb25lLAogICAgICAgICAgICAgICAgICByYWRpdXM6IGludCB8IE5vbmUgPSBOb25lLAogICAgICAgICAgICAgICAgICBjb250cmFzdDogZmxvYXQgfCBOb25lID0gTm9uZSkgLT4gbnAubmRhcnJheToKICAgICIiIlBhc3RlIGEgc3ludGhldGljIG1hc3M6IGEgc29mdC1lZGdlZCBlbGxpcHNlIHdpdGggaXRzIG93biB0ZXh0dXJlLgoKICAgIGByZWdpb25gIGlzIGFuIG9wdGlvbmFsIEh4VyBib29sIHJlc3RyaWN0aW5nIHdoZXJlIHRoZSBtYXNzIG1heSBsYW5kIOKAlCBmb3IKICAgIExpVFMgdGhhdCBpcyB0aGUgbGl2ZXIgbWFzaywgd2hpY2ggaXMgd2hhdCBtYWtlcyB0aGlzIHRoZSBQRkYtV00gcHJvdG9jb2wKICAgIHJhdGhlciB0aGFuIGFuIGFyYml0cmFyeSBibG9iLgogICAgIiIiCiAgICBoLCB3ID0gcmdiLnNoYXBlWzoyXQogICAgciA9IGludChyYWRpdXMgaWYgcmFkaXVzIGlzIG5vdCBOb25lIGVsc2Ugcm5nLmludGVnZXJzKDE4LCA0NCkpCiAgICBjeSA9IGludChybmcuaW50ZWdlcnMociArIDQsIGggLSByIC0gNCkpCiAgICBjeCA9IGludChybmcuaW50ZWdlcnMociArIDQsIHcgLSByIC0gNCkpCiAgICBpZiByZWdpb24gaXMgbm90IE5vbmU6CiAgICAgICAgY2FuZGlkYXRlcyA9IG5wLmFyZ3doZXJlKHJlZ2lvbikKICAgICAgICBpZiBsZW4oY2FuZGlkYXRlcykgPiA1MDoKICAgICAgICAgICAgY3ksIGN4ID0gY2FuZGlkYXRlc1tpbnQocm5nLmludGVnZXJzKDAsIGxlbihjYW5kaWRhdGVzKSkpXQoKICAgIHl5LCB4eCA9IG5wLm1ncmlkWy1yOnIgKyAxLCAtcjpyICsgMV0KICAgIGVsbGlwc2UgPSAoKHh4IC8gcikgKiogMiArICh5eSAvIChyICogMC44NSkpICoqIDIpIDw9IDEuMAogICAgIyBzb2Z0IGVkZ2Ugc28gaXQgbG9va3MgbGlrZSB0aXNzdWUsIG5vdCBhIHN0aWNrZXIKICAgIGRpc3QgPSBucC5zcXJ0KCh4eCAvIHIpICoqIDIgKyAoeXkgLyAociAqIDAuODUpKSAqKiAyKQogICAgYWxwaGEgPSBucC5jbGlwKCgxLjAgLSBkaXN0KSAvIDAuMTgsIDAuMCwgMS4wKSAqIGVsbGlwc2UKCiAgICB5MCwgeTEgPSBjeSAtIHIsIGN5ICsgciArIDEKICAgIHgwLCB4MSA9IGN4IC0gciwgY3ggKyByICsgMQogICAgaWYgeTAgPCAwIG9yIHgwIDwgMCBvciB5MSA+IGggb3IgeDEgPiB3OgogICAgICAgIHJldHVybiByZ2IuY29weSgpCgogICAgcGF0Y2ggPSByZ2JbeTA6eTEsIHgwOngxXS5hc3R5cGUobnAuZmxvYXQ2NCkKICAgIGdhaW4gPSBmbG9hdChjb250cmFzdCBpZiBjb250cmFzdCBpcyBub3QgTm9uZSBlbHNlIHJuZy51bmlmb3JtKC0wLjI1LCAwLjI1KSkKICAgIHRleHR1cmUgPSBfcm5nKGludChybmcuaW50ZWdlcnMoMCwgMioqMzEgLSAxKSkpLm5vcm1hbCgwLCA3LCBwYXRjaC5zaGFwZSkKCiAgICAjIGEgbWFzcyBpcyBhIGNvaGVyZW50IHN0cnVjdHVyZSwgbm90IG5vaXNlOiBsb3ctZnJlcXVlbmN5IHNoYWRpbmcgcGx1cyBncmFpbgogICAgc2hhZGUgPSBjdjIuR2F1c3NpYW5CbHVyKHRleHR1cmUuYXN0eXBlKG5wLmZsb2F0MzIpLCAoMCwgMCksIHNpZ21hWD0zKQogICAgYmxvYiA9IHBhdGNoICogKDEuMCArIGdhaW4pICsgc2hhZGUgKyB0ZXh0dXJlCiAgICBhID0gYWxwaGFbLi4uLCBOb25lXQogICAgb3V0ID0gcmdiLmNvcHkoKQogICAgb3V0W3kwOnkxLCB4MDp4MV0gPSBucC5jbGlwKHBhdGNoICogKDEgLSBhKSArIGJsb2IgKiBhLCAwLCAyNTUpLmFzdHlwZShucC51aW50OCkKICAgIHJldHVybiBvdXQKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gVDMgbGVzaW9uIHJlbW92YWwKCmRlZiBsZXNpb25fcmVtb3ZlKHJnYjogbnAubmRhcnJheSwgcm5nOiBucC5yYW5kb20uR2VuZXJhdG9yLAogICAgICAgICAgICAgICAgICByYWRpdXM6IGludCB8IE5vbmUgPSBOb25lLCBtZXRob2Q6IHN0ciA9ICJ0ZWxlYSIpIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJFcmFzZSBhIHJlZ2lvbiBhbmQgaW5wYWludCBpdC4gVGhlIGNsaW5pY2FsbHkgaW50ZXJlc3RpbmcgZm9yZ2VyeS4iIiIKICAgIGgsIHcgPSByZ2Iuc2hhcGVbOjJdCiAgICByID0gaW50KHJhZGl1cyBpZiByYWRpdXMgaXMgbm90IE5vbmUgZWxzZSBybmcuaW50ZWdlcnMoMjIsIDUyKSkKICAgIGN5ID0gaW50KHJuZy5pbnRlZ2VycyhyICsgNCwgaCAtIHIgLSA0KSkKICAgIGN4ID0gaW50KHJuZy5pbnRlZ2VycyhyICsgNCwgdyAtIHIgLSA0KSkKICAgIG1hc2sgPSBucC56ZXJvcygoaCwgdyksIG5wLnVpbnQ4KQogICAgY3YyLmVsbGlwc2UobWFzaywgKGN4LCBjeSksIChyLCBpbnQociAqIDAuODUpKSwgaW50KHJuZy5pbnRlZ2VycygwLCAxODApKSwKICAgICAgICAgICAgICAgIDAsIDM2MCwgMjU1LCAtMSkKICAgIGZsYWcgPSB7InRlbGVhIjogY3YyLklOUEFJTlRfVEVMRUEsICJucyI6IGN2Mi5JTlBBSU5UX05TfS5nZXQobWV0aG9kLCBjdjIuSU5QQUlOVF9URUxFQSkKICAgIHJldHVybiBjdjIuaW5wYWludChyZ2IsIG1hc2ssIDcsIGZsYWcpCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIFQ0IGJ1cm5lZC1pbiB0ZXh0CgpkZWYgYnVybmVkX3RleHQocmdiOiBucC5uZGFycmF5LCBybmc6IG5wLnJhbmRvbS5HZW5lcmF0b3IsCiAgICAgICAgICAgICAgICBib3g6IHR1cGxlW2ludCwgaW50XSA9ICg2NCwgMjQpLAogICAgICAgICAgICAgICAgdGV4dDogc3RyIHwgTm9uZSA9IE5vbmUpIC0+IG5wLm5kYXJyYXk6CiAgICBoLCB3ID0gcmdiLnNoYXBlWzoyXQogICAgb3V0ID0gcmdiLmNvcHkoKQogICAgYmgsIGJ3ID0gYm94CiAgICB5ID0gX2Jsb2NrX3RvcF9sZWZ0KHJuZywgaCAtIGJoLCBiaCkKICAgIHggPSBfYmxvY2tfdG9wX2xlZnQocm5nLCB3IC0gYncsIGJ3KQogICAgbGFiZWwgPSB0ZXh0IG9yIHN0cihybmcuY2hvaWNlKFsiUEFUSUVOVCBBIiwgIkNPTkZJREVOVElBTCIsICJSRVYgMiIsICJTQU1QTEUgMDQiLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAiMjAyNi0xMC0yNCIsICJNUk4gODgyMTMiXSkpCiAgICBkYXJrID0gYm9vbChybmcuaW50ZWdlcnMoMCwgMikpCiAgICBjb2xvdXIgPSAoMCwgMCwgMCkgaWYgZGFyayBlbHNlICgyNTUsIDI1NSwgMjU1KQogICAgY3YyLnB1dFRleHQob3V0LCBsYWJlbCwgKHgsIHkgKyBiaCAtIDYpLCBjdjIuRk9OVF9IRVJTSEVZX1NJTVBMRVgsCiAgICAgICAgICAgICAgICBiaCAvIDM0LjAsIGNvbG91ciwgbWF4KDEsIGJoIC8vIDEyKSwgY3YyLkxJTkVfQUEpCiAgICByZXR1cm4gb3V0CgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIFQ1IGNyb3NzLWltYWdlIHNwbGljaW5nCgpkZWYgc3BsaWNlKGRvbm9yOiBucC5uZGFycmF5LCBob3N0OiBucC5uZGFycmF5LCBybmc6IG5wLnJhbmRvbS5HZW5lcmF0b3IsCiAgICAgICAgICAgc3BhbjogaW50IHwgTm9uZSA9IE5vbmUpIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJQYXN0ZSBhIHJlZ2lvbiB0YWtlbiBmcm9tIGEgRElGRkVSRU5UIGltYWdlLgoKICAgIFRoZSBjbGFzc2ljYWwgYmF0dGVyeSBuZXZlciBtZWFzdXJlZCB0aGlzIGFuZCB0aGUgbGltaXRhdGlvbiBzZWN0aW9uIGFkbWl0cwogICAgaXQgaXMgb25seSBwYXJ0bHkgY2F1Z2h0LiBJdCBpcyBoZXJlIGJlY2F1c2UgaXQgaXMgdGhlIGF0dGFjayBhIGZvcmdlZAogICAgbWVkaWNhbCByZWNvcmQgd291bGQgYWN0dWFsbHkgdXNlLgogICAgIiIiCiAgICBzcGFuID0gaW50KHNwYW4gaWYgc3BhbiBpcyBub3QgTm9uZSBlbHNlIHJuZy5pbnRlZ2VycygzMiwgMTIwKSkKICAgIHNfciA9IF9ibG9ja190b3BfbGVmdChybmcsIFNJWkUsIHNwYW4pCiAgICBzX2MgPSBfYmxvY2tfdG9wX2xlZnQocm5nLCBTSVpFLCBzcGFuKQogICAgZF9yID0gX2Jsb2NrX3RvcF9sZWZ0KHJuZywgU0laRSwgc3BhbikKICAgIGRfYyA9IF9ibG9ja190b3BfbGVmdChybmcsIFNJWkUsIHNwYW4pCiAgICBvdXQgPSBob3N0LmNvcHkoKQogICAgb3V0W2RfcjpkX3IgKyBzcGFuLCBkX2M6ZF9jICsgc3Bhbl0gPSBkb25vcltzX3I6c19yICsgc3Bhbiwgc19jOnNfYyArIHNwYW5dCiAgICByZXR1cm4gb3V0CgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIGJlbmlnbgoKZGVmIGJlbmlnbl9vcChyZ2I6IG5wLm5kYXJyYXksIGtpbmQ6IHN0ciA9ICJqcGVnNzUiLCBybmc6IG5wLnJhbmRvbS5HZW5lcmF0b3IgfCBOb25lID0gTm9uZSkgLT4gbnAubmRhcnJheToKICAgICIiIlJvdXRpbmUgaGFuZGxpbmcuIE11c3QgbmV2ZXIgcHJvZHVjZSBhIHRhbXBlciBjbGFpbS4iIiIKICAgIGlmIGtpbmQuc3RhcnRzd2l0aCgianBlZyIpOgogICAgICAgIHEgPSBpbnQoa2luZC5yZXBsYWNlKCJqcGVnIiwgIiIpKQogICAgICAgIG9rLCBidWYgPSBjdjIuaW1lbmNvZGUoIi5qcGciLCBjdjIuY3Z0Q29sb3IocmdiLCBjdjIuQ09MT1JfUkdCMkJHUiksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBbaW50KGN2Mi5JTVdSSVRFX0pQRUdfUVVBTElUWSksIHFdKQogICAgICAgIHJldHVybiBjdjIuY3Z0Q29sb3IoY3YyLmltZGVjb2RlKGJ1ZiwgY3YyLklNUkVBRF9DT0xPUiksIGN2Mi5DT0xPUl9CR1IyUkdCKQogICAgaWYga2luZC5zdGFydHN3aXRoKCJzaWdtYSIpOgogICAgICAgIHMgPSBmbG9hdChraW5kLnJlcGxhY2UoInNpZ21hIiwgIiIpKQogICAgICAgIHIgPSBybmcgb3IgX3JuZygwKQogICAgICAgIG5vaXNlID0gci5ub3JtYWwoMCwgcywgcmdiLnNoYXBlKQogICAgICAgIHJldHVybiBucC5jbGlwKHJnYi5hc3R5cGUobnAuZmxvYXQ2NCkgKyBub2lzZSwgMCwgMjU1KS5hc3R5cGUobnAudWludDgpCiAgICBpZiBraW5kLnN0YXJ0c3dpdGgoInJlc2l6ZSIpOgogICAgICAgIHBjdCA9IGludChraW5kLnJlcGxhY2UoInJlc2l6ZSIsICIiKSkgLyAxMDAuMAogICAgICAgIGgsIHcgPSByZ2Iuc2hhcGVbOjJdCiAgICAgICAgc21hbGwgPSBjdjIucmVzaXplKHJnYiwgKG1heCg4LCBpbnQodyAqIHBjdCkpLCBtYXgoOCwgaW50KGggKiBwY3QpKSksCiAgICAgICAgICAgICAgICAgICAgICAgICAgIGludGVycG9sYXRpb249Y3YyLklOVEVSX0NVQklDKQogICAgICAgIHJldHVybiBjdjIucmVzaXplKHNtYWxsLCAodywgaCksIGludGVycG9sYXRpb249Y3YyLklOVEVSX0NVQklDKQogICAgaWYga2luZCA9PSAiYmx1cjIiOgogICAgICAgIHJldHVybiBjdjIuR2F1c3NpYW5CbHVyKHJnYiwgKDAsIDApLCAyLjApCiAgICByYWlzZSBWYWx1ZUVycm9yKGYidW5rbm93biBiZW5pZ24gb3Age2tpbmQhcn0iKQoKCmRlZiBiZW5pZ25fY2hhaW4ocmdiOiBucC5uZGFycmF5LCBybmc6IG5wLnJhbmRvbS5HZW5lcmF0b3IgfCBOb25lID0gTm9uZSkgLT4gbnAubmRhcnJheToKICAgICIiInNjYWxlIC0+IGpwZWcgLT4gc2NhbGUgLT4ganBlZy4gVGhlIHJlYWxpc3RpYyB0cmFuc21pc3Npb24gcGF0aC4iIiIKICAgIG91dCA9IGJlbmlnbl9vcChyZ2IsICJyZXNpemU4MCIpCiAgICBvdXQgPSBiZW5pZ25fb3Aob3V0LCAianBlZzYwIikKICAgIG91dCA9IGJlbmlnbl9vcChvdXQsICJyZXNpemUxMjUiKQogICAgaCwgdyA9IHJnYi5zaGFwZVs6Ml0KICAgIG91dCA9IGN2Mi5yZXNpemUob3V0LCAodywgaCksIGludGVycG9sYXRpb249Y3YyLklOVEVSX0NVQklDKQogICAgcmV0dXJuIGJlbmlnbl9vcChvdXQsICJqcGVnNjAiKQoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSByZWdpc3RyeQoKVEFNUEVSID0gewogICAgIlQxIjogbGFtYmRhIHJnYiwgcm5nOiBjb3B5X21vdmUocmdiLCBybmcsIHNwYW49aW50KHJuZy5jaG9pY2UoWzgsIDE2LCAzMiwgNjRdKSksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBuX2NvcGllcz1pbnQocm5nLmludGVnZXJzKDEsIDMpKSksCiAgICAiVDIiOiBsYW1iZGEgcmdiLCBybmc6IGxlc2lvbl9pbnNlcnQocmdiLCBybmcpLAogICAgIlQzIjogbGFtYmRhIHJnYiwgcm5nOiBsZXNpb25fcmVtb3ZlKHJnYiwgcm5nKSwKICAgICJUNCI6IGxhbWJkYSByZ2IsIHJuZzogYnVybmVkX3RleHQocmdiLCBybmcpLAp9CgpCRU5JR04gPSB7CiAgICAiQjEiOiBsYW1iZGEgcmdiLCBybmc6IGJlbmlnbl9vcChyZ2IsIHN0cihybmcuY2hvaWNlKAogICAgICAgIFsianBlZzkwIiwgImpwZWc3NSIsICJqcGVnNTAiLCAic2lnbWExIiwgInJlc2l6ZTkwIl0pKSksCiAgICAiQjIiOiBsYW1iZGEgcmdiLCBybmc6IGJlbmlnbl9jaGFpbihyZ2IpLAp9CgoKZGVmIG1ha2VfcGFpcihyZ2I6IG5wLm5kYXJyYXksIGZhbWlseTogc3RyLCBzZWVkOiBpbnQsCiAgICAgICAgICAgICAgZG9ub3I6IG5wLm5kYXJyYXkgfCBOb25lID0gTm9uZSkgLT4gdHVwbGVbbnAubmRhcnJheSwgbnAubmRhcnJheV06CiAgICAiIiJBcHBseSBhIGZhbWlseSBhbmQgcmV0dXJuIChjaGFuZ2VkIGltYWdlLCBtZWFzdXJlZCBibG9jayBtYXNrKS4iIiIKICAgIHJuZyA9IF9ybmcoc2VlZCkKICAgIHJnYiA9IG5wLmFzYXJyYXkocmdiLCBucC51aW50OCkKICAgIGlmIGZhbWlseSBpbiBUQU1QRVI6CiAgICAgICAgb3V0ID0gVEFNUEVSW2ZhbWlseV0ocmdiLCBybmcpCiAgICBlbGlmIGZhbWlseSA9PSAiVDUiOgogICAgICAgIGlmIGRvbm9yIGlzIE5vbmU6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlQ1IG5lZWRzIGEgZG9ub3IgaW1hZ2UiKQogICAgICAgIG91dCA9IHNwbGljZShucC5hc2FycmF5KGRvbm9yLCBucC51aW50OCksIHJnYiwgcm5nKQogICAgZWxpZiBmYW1pbHkgaW4gQkVOSUdOOgogICAgICAgIG91dCA9IEJFTklHTltmYW1pbHldKHJnYiwgcm5nKQogICAgZWxzZToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYidW5rbm93biBmYW1pbHkge2ZhbWlseSFyfSIpCiAgICBvdXQgPSBucC5hc2FycmF5KG91dCwgbnAudWludDgpCiAgICBtYXNrID0gbWVhc3VyZWRfbWFzayhyZ2IsIG91dCkKICAgIHJldHVybiBvdXQsIG1hc2sK', 'degradation': 'IiIiRGVncmFkYXRpb24gZXN0aW1hdGVzOiB0aGUgZm91ciBzY2FsYXJzIHRoZSBSZWZpbmVOZXQgY29uZGl0aW9ucyBvbi4KClJvdyA2IG9mIHRoZSBzeW50aGVzaXMgdGFibGUgKE9tbmlHdWFyZCkuIEEgZGVncmFkYXRpb24tYmxpbmQgbG9jYWxpc2VyIGNhbm5vdAp0ZWxsICJ0aGlzIGJsb2NrIGxvc3QgaXRzIG1hcmsgYmVjYXVzZSBzb21lb25lIGVkaXRlZCBpdCIgZnJvbSAidGhpcyBibG9jayBsb3N0Cml0cyBtYXJrIGJlY2F1c2UgdGhlIHdob2xlIGZpbGUgd2VudCB0aHJvdWdoIEpQRUcgNTAiLiBDb25kaXRpb25pbmcgb24gYSBnbG9iYWwKZXN0aW1hdGUgb2Ygd2hhdCB0aGUgZmlsZSBoYXMgYmVlbiB0aHJvdWdoIGxldHMgdGhlIG5ldHdvcmsgc2VwYXJhdGUgdGhlIHR3by4KCkV2ZXJ5IGVzdGltYXRvciBoZXJlIGlzIHRyYWluaW5nLWZyZWUgYW5kIGNvbXB1dGVkIGZyb20gdGhlIHJlY2VpdmVkIGltYWdlCmFsb25lLiBOb25lIG9mIHRoZW0gaXMgZXhhY3Q7IHRoZXkgb25seSBoYXZlIHRvIGJlIG1vbm90b25lIGluIHRoZSB0aGluZyB0aGV5Cm1lYXN1cmUsIGJlY2F1c2UgdGhlIG5ldHdvcmsgbGVhcm5zIHRvIHVzZSB0aGUgc2NhbGUsIG5vdCB0aGUgdmFsdWUuCgpUaGUgZm91ciBjaGFubmVsczoKICAgIGpwZWdpbmVzcyAgIGJsb2NrLWJvdW5kYXJ5IGVuZXJneSByZWxhdGl2ZSB0byBpbnRlcmlvciBlbmVyZ3kgICgwID0gbm9uZSkKICAgIG5vaXNlICAgICAgIHJvYnVzdCBzaWdtYSBmcm9tIHRoZSBoaWdoLXBhc3MgcmVzaWR1YWwKICAgIHNoYXJwbmVzcyAgIGdyYWRpZW50IGVuZXJneSwgZmFsbHMgdW5kZXIgYmx1ciBhbmQgcmVzYW1wbGluZwogICAgY2xpcHBpbmcgICAgZnJhY3Rpb24gb2YgcGl4ZWxzIHBpbm5lZCBhdCAwIG9yIDI1NQoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBjdjIKaW1wb3J0IG51bXB5IGFzIG5wCgpCTE9DSyA9IDgKCgpkZWYgX2dyYXkocmdiOiBucC5uZGFycmF5KSAtPiBucC5uZGFycmF5OgogICAgcmdiID0gbnAuYXNhcnJheShyZ2IpCiAgICBpZiByZ2IubmRpbSA9PSAyOgogICAgICAgIHJldHVybiByZ2IuYXN0eXBlKG5wLmZsb2F0NjQpCiAgICByZXR1cm4gKDAuMjk5ICogcmdiWy4uLiwgMF0gKyAwLjU4NyAqIHJnYlsuLi4sIDFdICsgMC4xMTQgKiByZ2JbLi4uLCAyXSkuYXN0eXBlKG5wLmZsb2F0NjQpCgoKZGVmIGpwZWdpbmVzcyhncmF5OiBucC5uZGFycmF5KSAtPiBmbG9hdDoKICAgICIiIkVuZXJneSBhY3Jvc3MgOC1waXhlbCBib3VuZGFyaWVzIGRpdmlkZWQgYnkgZW5lcmd5IGluc2lkZSBibG9ja3MuCgogICAgQSBKUEVHLWRlY29tcHJlc3NlZCBpbWFnZSBoYXMgYSBkaXNjb250aW51aXR5IGV2ZXJ5IDggcGl4ZWxzIHRoYXQgdGhlCiAgICBvcmlnaW5hbCBkaWQgbm90IGhhdmUuIFRoZSByYXRpbyBpcyBuZWFyIDEuMCBmb3IgYSBwcmlzdGluZSBQTkcgYW5kIHJpc2VzCiAgICB3aXRoIGNvbXByZXNzaW9uLiBOb3JtYWxpc2VkIHRvIDAuLjEgd2l0aCBhIHNvZnQga25lZS4KICAgICIiIgogICAgZyA9IF9ncmF5KGdyYXkpCiAgICBoLCB3ID0gZy5zaGFwZQogICAgaCAtPSBoICUgQkxPQ0sKICAgIHcgLT0gdyAlIEJMT0NLCiAgICBnID0gZ1s6aCwgOnddCiAgICBkeCA9IG5wLmFicyhucC5kaWZmKGcsIGF4aXM9MSkpCiAgICBkeSA9IG5wLmFicyhucC5kaWZmKGcsIGF4aXM9MCkpCiAgICBieCA9IGR4WzosIEJMT0NLIC0gMTo6QkxPQ0tdLm1lYW4oKSBpZiBkeFs6LCBCTE9DSyAtIDE6OkJMT0NLXS5zaXplIGVsc2UgMC4wCiAgICBieSA9IGR5W0JMT0NLIC0gMTo6QkxPQ0ssIDpdLm1lYW4oKSBpZiBkeVtCTE9DSyAtIDE6OkJMT0NLLCA6XS5zaXplIGVsc2UgMC4wCiAgICBpbm5lcl94ID0gbnAuZGVsZXRlKGR4LCBucC5zX1tCTE9DSyAtIDE6OkJMT0NLXSwgYXhpcz0xKS5tZWFuKCkKICAgIGlubmVyX3kgPSBucC5kZWxldGUoZHksIG5wLnNfW0JMT0NLIC0gMTo6QkxPQ0tdLCBheGlzPTApLm1lYW4oKQogICAgaW5uZXIgPSAoaW5uZXJfeCArIGlubmVyX3kpIC8gMi4wICsgMWUtNgogICAgcmF0aW8gPSAoKGJ4ICsgYnkpIC8gMi4wKSAvIGlubmVyCiAgICByZXR1cm4gZmxvYXQobnAuY2xpcCgocmF0aW8gLSAxLjApIC8gMS41LCAwLjAsIDEuMCkpCgoKZGVmIG5vaXNlX3NpZ21hKGdyYXk6IG5wLm5kYXJyYXkpIC0+IGZsb2F0OgogICAgIiIiUm9idXN0IHNpZ21hIGZyb20gdGhlIExhcGxhY2lhbiwgdGhlIHN0YW5kYXJkIE1BRCBlc3RpbWF0b3IuIiIiCiAgICBnID0gX2dyYXkoZ3JheSkKICAgIGxhcCA9IGN2Mi5MYXBsYWNpYW4oZy5hc3R5cGUobnAuZmxvYXQzMiksIGN2Mi5DVl8zMkYpCiAgICByZXR1cm4gZmxvYXQobnAubWVkaWFuKG5wLmFicyhsYXApKSAvIDAuNjc0NSkKCgpkZWYgc2hhcnBuZXNzKGdyYXk6IG5wLm5kYXJyYXkpIC0+IGZsb2F0OgogICAgIiIiTWVhbiBncmFkaWVudCBtYWduaXR1ZGUuIERyb3BzIHVuZGVyIGJsdXIgYW5kIHJlc2FtcGxpbmcuIiIiCiAgICBnID0gX2dyYXkoZ3JheSkKICAgIGd4ID0gY3YyLlNvYmVsKGcuYXN0eXBlKG5wLmZsb2F0MzIpLCBjdjIuQ1ZfMzJGLCAxLCAwLCBrc2l6ZT0zKQogICAgZ3kgPSBjdjIuU29iZWwoZy5hc3R5cGUobnAuZmxvYXQzMiksIGN2Mi5DVl8zMkYsIDAsIDEsIGtzaXplPTMpCiAgICByZXR1cm4gZmxvYXQobnAubWVhbihucC5oeXBvdChneCwgZ3kpKSkKCgpkZWYgY2xpcHBpbmcocmdiOiBucC5uZGFycmF5KSAtPiBmbG9hdDoKICAgICIiIkZyYWN0aW9uIG9mIHBpeGVscyBwaW5uZWQgYXQgZWl0aGVyIGVuZCBvZiB0aGUgcmFuZ2UuIiIiCiAgICBhID0gbnAuYXNhcnJheShyZ2IpCiAgICByZXR1cm4gZmxvYXQoKChhID09IDApIHwgKGEgPT0gMjU1KSkubWVhbigpKQoKCmRlZiBlc3RpbWF0ZShyZ2I6IG5wLm5kYXJyYXksIHJlZl9zaGFycG5lc3M6IGZsb2F0IHwgTm9uZSA9IE5vbmUpIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJUaGUgZm91ciBjaGFubmVscywgc2NhbGVkIHRvIGEgY29tcGFyYWJsZSByYW5nZSBmb3IgYSBuZXR3b3JrIGlucHV0LgoKICAgIGByZWZfc2hhcnBuZXNzYCBsZXRzIGEgY2FsbGVyIG5vcm1hbGlzZSBhZ2FpbnN0IHRoZSBjb3JwdXMgbWVkaWFuLCBzbyB0aGUKICAgIHNoYXJwbmVzcyBjaGFubmVsIHJlYWRzIGFzICdyZWxhdGl2ZSB0byBhIHR5cGljYWwgaW1hZ2UnIHJhdGhlciB0aGFuIGFuCiAgICBhYnNvbHV0ZSBncmFkaWVudCBlbmVyZ3kgdGhhdCB2YXJpZXMgd2l0aCBjb250ZW50LgogICAgIiIiCiAgICBnID0gX2dyYXkocmdiKQogICAgcyA9IHNoYXJwbmVzcyhnKQogICAgcmVsID0gMS4wIGlmIHJlZl9zaGFycG5lc3MgaXMgTm9uZSBvciByZWZfc2hhcnBuZXNzIDw9IDAgZWxzZSBmbG9hdCgKICAgICAgICBucC5jbGlwKHMgLyByZWZfc2hhcnBuZXNzLCAwLjAsIDIuMCkpCiAgICByZXR1cm4gbnAuYXJyYXkoWwogICAgICAgIGpwZWdpbmVzcyhnKSwKICAgICAgICBmbG9hdChucC5jbGlwKG5vaXNlX3NpZ21hKGcpIC8gOC4wLCAwLjAsIDIuMCkpLAogICAgICAgIHJlbCwKICAgICAgICBjbGlwcGluZyhyZ2IpLAogICAgXSwgbnAuZmxvYXQzMikK', 'models': '...', 'isic_meta': 'IiIiSVNJQyAyMDE5IG1ldGFkYXRhOiBsZXNpb24gZ3JvdXBpbmcgYW5kIGRpYWdub3N0aWMgc3RyYXRhLgoKVHdvIGpvYnMsIGJvdGggbmVlZGVkIGZvciBhbiBJU0lDLTIwMTktb25seSBldmFsdWF0aW9uIHRvIGJlIHdvcnRoIGFueXRoaW5nOgoKMS4gKipHcm91cGluZy4qKiBJbWFnZXMgb2YgdGhlIHNhbWUgbGVzaW9uIG11c3QgbmV2ZXIgc3RyYWRkbGUgdHJhaW4gYW5kIHRlc3QuCiAgIElTSUMgMjAxOSBzaGlwcyBhIGBsZXNpb25faWRgOyB3aGVyZSBpdCBpcyBtaXNzaW5nIHdlIGZhbGwgYmFjayB0byB0aGUgaW1hZ2UKICAgc3RlbSwgd2hpY2ggaXMgY29uc2VydmF0aXZlIChpdCBjYW4gb25seSBzcGxpdCBtb3JlIGZpbmVseSwgbmV2ZXIgbGVhayBtb3JlKS4KCjIuICoqU3RyYXRpZmljYXRpb24gYnkgZGlhZ25vc2lzLioqIFRoZSBlaWdodCBJU0lDIGNhdGVnb3JpZXMgYXJlIG5vdAogICBpbnRlcmNoYW5nZWFibGUgZm9yIGxvY2FsaXNhdGlvbi4gQSBmbGF0IG5hZXZ1cyBhbmQgYSB0ZXh0dXJlZCBiYXNhbC1jZWxsCiAgIGNhcmNpbm9tYSBsb3NlIHRoZWlyIG1hcmsgaW4gZGlmZmVyZW50IHBsYWNlcywgYW5kIGEgc2luZ2xlIGFnZ3JlZ2F0ZSBJb1UKICAgaGlkZXMgdGhhdC4gUmVwb3J0aW5nIHBlci1kaWFnbm9zaXMgaXMgd2hhdCB0dXJucyBvbmUgY29ycHVzIGludG8gYSByZWFsCiAgIHN0dWR5IGluc3RlYWQgb2YgYSBzaW5nbGUgaGVhZGxpbmUgbnVtYmVyLgoKTm90aGluZyBoZXJlIGFzc3VtZXMgYSBjb2x1bW4gbGF5b3V0LiBJU0lDIDIwMTkgc2hpcHMgaW4gYXQgbGVhc3QgdGhyZWUgc2hhcGVzCmRlcGVuZGluZyBvbiB0aGUgbWlycm9yOgoKICAgIGltYWdlLCBsZXNpb25faWQsIE1FTCwgTlYsIEJDQywgQUssIEJLTCwgREYsIFZBU0MsIFNDQywgVU5LICAgKG9uZS1ob3QpCiAgICBpbWFnZSwgbGVzaW9uX2lkLCBkaWFnbm9zaXMgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAoc2luZ2xlIGNvbCkKICAgIGltYWdlICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIChub3RoaW5nKQoKVGhlIGxvYWRlciBkZXRlY3RzIHdoaWNoIG9uZSBpdCBoYXMgYW5kIHNheXMgc28uIElmIGl0IGZpbmRzIG5vIGRpYWdub3NpcyBhdAphbGwsIHN0cmF0aWZpY2F0aW9uIGlzIGRpc2FibGVkIGxvdWRseSByYXRoZXIgdGhhbiBzaWxlbnRseSBwcm9kdWNpbmcgYSBjb2x1bW4Kb2YgInVua25vd24iLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBjc3YKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpESUFHTk9TRVMgPSAoIk1FTCIsICJOViIsICJCQ0MiLCAiQUsiLCAiQktMIiwgIkRGIiwgIlZBU0MiLCAiU0NDIikKVU5LTk9XTiA9ICJVTksiCgoKY2xhc3MgTWV0YWRhdGE6CiAgICAiIiJQYXJzZWQgSVNJQyBtZXRhZGF0YSwgd2l0aCB0aGUgc2hhcGUgaXQgd2FzIGZvdW5kIGluIHJlY29yZGVkLiIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBwYXRoOiBzdHIgfCBQYXRoIHwgTm9uZSk6CiAgICAgICAgc2VsZi5wYXRoID0gUGF0aChwYXRoKSBpZiBwYXRoIGVsc2UgTm9uZQogICAgICAgIHNlbGYubGVzaW9uOiBkaWN0W3N0ciwgc3RyXSA9IHt9CiAgICAgICAgc2VsZi5kaWFnbm9zaXM6IGRpY3Rbc3RyLCBzdHJdID0ge30KICAgICAgICBzZWxmLnNjaGVtYSA9ICJub25lIgogICAgICAgIHNlbGYubm90ZSA9ICIiCiAgICAgICAgaWYgc2VsZi5wYXRoIGlzIG5vdCBOb25lOgogICAgICAgICAgICBzZWxmLl9sb2FkKCkKCiAgICBkZWYgX2xvYWQoc2VsZikgLT4gTm9uZToKICAgICAgICB3aXRoIG9wZW4oc2VsZi5wYXRoLCBuZXdsaW5lPSIiKSBhcyBmaDoKICAgICAgICAgICAgcm93cyA9IGxpc3QoY3N2LkRpY3RSZWFkZXIoZmgpKQogICAgICAgIGlmIG5vdCByb3dzOgogICAgICAgICAgICBzZWxmLm5vdGUgPSAibWV0YWRhdGEgZmlsZSBpcyBlbXB0eSIKICAgICAgICAgICAgcmV0dXJuCiAgICAgICAgY29scyA9IHtjLnN0cmlwKCkgZm9yIGMgaW4gcm93c1swXS5rZXlzKCl9CiAgICAgICAgbG93ZXIgPSB7Yy5sb3dlcigpOiBjIGZvciBjIGluIGNvbHN9CgogICAgICAgIGltZ19rZXkgPSBsb3dlci5nZXQoImltYWdlIikgb3IgbG93ZXIuZ2V0KCJpbWFnZV9uYW1lIikgb3IgbG93ZXIuZ2V0KCJmaWxlbmFtZSIpCiAgICAgICAgaWYgaW1nX2tleSBpcyBOb25lOgogICAgICAgICAgICBzZWxmLm5vdGUgPSBmIm5vIGltYWdlIGNvbHVtbiBpbiB7c29ydGVkKGNvbHMpfSIKICAgICAgICAgICAgcmV0dXJuCgogICAgICAgIG9uZV9ob3QgPSBbZCBmb3IgZCBpbiBESUFHTk9TRVMgaWYgZCBpbiBjb2xzXQogICAgICAgIHNpbmdsZSA9IGxvd2VyLmdldCgiZGlhZ25vc2lzIikgb3IgbG93ZXIuZ2V0KCJkeCIpCgogICAgICAgIGlmIG9uZV9ob3Q6CiAgICAgICAgICAgIHNlbGYuc2NoZW1hID0gIm9uZS1ob3QiCiAgICAgICAgZWxpZiBzaW5nbGU6CiAgICAgICAgICAgIHNlbGYuc2NoZW1hID0gInNpbmdsZSIKICAgICAgICBlbHNlOgogICAgICAgICAgICBzZWxmLnNjaGVtYSA9ICJuby1kaWFnbm9zaXMiCgogICAgICAgIGZvciByIGluIHJvd3M6CiAgICAgICAgICAgIHN0ZW0gPSBQYXRoKHN0cihyW2ltZ19rZXldKSkuc3RlbQogICAgICAgICAgICBsaWQgPSByLmdldChsb3dlci5nZXQoImxlc2lvbl9pZCIsICIiKSwgIiIpIG9yICIiCiAgICAgICAgICAgIHNlbGYubGVzaW9uW3N0ZW1dID0gc3RyKGxpZCkuc3RyaXAoKSBvciBzdGVtCiAgICAgICAgICAgIGlmIHNlbGYuc2NoZW1hID09ICJvbmUtaG90IjoKICAgICAgICAgICAgICAgIGhpdCA9IFtkIGZvciBkIGluIG9uZV9ob3QgaWYgc3RyKHIuZ2V0KGQsICIwIikpLnN0cmlwKCkgbm90IGluICgiIiwgIjAiLCAiMC4wIildCiAgICAgICAgICAgICAgICBzZWxmLmRpYWdub3Npc1tzdGVtXSA9IGhpdFswXSBpZiBoaXQgZWxzZSBVTktOT1dOCiAgICAgICAgICAgIGVsaWYgc2VsZi5zY2hlbWEgPT0gInNpbmdsZSI6CiAgICAgICAgICAgICAgICBzZWxmLmRpYWdub3Npc1tzdGVtXSA9IHN0cihyW3NpbmdsZV0pLnN0cmlwKCkudXBwZXIoKSBvciBVTktOT1dOCgogICAgICAgIHNlbGYubm90ZSA9IChmInNjaGVtYT17c2VsZi5zY2hlbWF9IHJvd3M9e2xlbihyb3dzKX0gIgogICAgICAgICAgICAgICAgICAgICBmIndpdGhfbGVzaW9uPXtzdW0oMSBmb3IgdiBpbiBzZWxmLmxlc2lvbi52YWx1ZXMoKSBpZiB2KX0gIgogICAgICAgICAgICAgICAgICAgICBmIndpdGhfZHg9e2xlbihzZWxmLmRpYWdub3Npcyl9IikKCiAgICAjIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQogICAgQHByb3BlcnR5CiAgICBkZWYgaGFzX2RpYWdub3NpcyhzZWxmKSAtPiBib29sOgogICAgICAgIHJldHVybiBib29sKHNlbGYuZGlhZ25vc2lzKQoKICAgIGRlZiBncm91cF9vZihzZWxmLCBzdGVtOiBzdHIpIC0+IHN0cjoKICAgICAgICAiIiJUaGUgZ3JvdXBpbmcga2V5IGZvciBzcGxpdHRpbmcuIEZhbGxzIGJhY2sgdG8gdGhlIHN0ZW0uIiIiCiAgICAgICAgcmV0dXJuIHNlbGYubGVzaW9uLmdldChzdGVtLCBzdGVtKQoKICAgIGRlZiBkeF9vZihzZWxmLCBzdGVtOiBzdHIpIC0+IHN0cjoKICAgICAgICAiIiJEaWFnbm9zaXMsIG9yICdVTktOT1dOJyB3aGVuIHRoZSBjb3JwdXMgZG9lcyBub3QgY2Fycnkgb25lLiIiIgogICAgICAgIHJldHVybiBzZWxmLmRpYWdub3Npcy5nZXQoc3RlbSwgVU5LTk9XTikgaWYgc2VsZi5kaWFnbm9zaXMgZWxzZSBVTktOT1dOCgoKZGVmIHN0cmF0aWZ5X2J5KGl0ZW1zOiBsaXN0W2RpY3RdLCBrZXk6IHN0cikgLT4gZGljdFtzdHIsIGxpc3RbaW50XV06CiAgICAiIiJJbmRpY2VzIGdyb3VwZWQgYnkgYGl0ZW1zW2ldW2tleV1gLiBFdmVyeSBncm91cCBpcyBrZXB0LCBpbmNsdWRpbmcgb25lcwogICAgdGhhdCB0dXJuIG91dCBzbWFsbCAtLSBhbiBvbWl0dGVkIGJ1Y2tldCBpcyBhIGhpZGRlbiByZXN1bHQuIiIiCiAgICBvdXQ6IGRpY3Rbc3RyLCBsaXN0W2ludF1dID0ge30KICAgIGZvciBpLCBpdCBpbiBlbnVtZXJhdGUoaXRlbXMpOgogICAgICAgIG91dC5zZXRkZWZhdWx0KHN0cihpdC5nZXQoa2V5LCBVTktOT1dOKSksIFtdKS5hcHBlbmQoaSkKICAgIHJldHVybiBvdXQKCgpkZWYgcmVwb3J0KHN0cmF0YTogZGljdFtzdHIsIGxpc3RbaW50XV0sIHZhbHVlczogbGlzdFtmbG9hdF0sCiAgICAgICAgICAgbWluX25fd2FybjogaW50ID0gMTApIC0+IGxpc3RbZGljdF06CiAgICAiIiJPbmUgcm93IHBlciBzdHJhdHVtIHdpdGggbiwgbWVhbiBhbmQgYSBzbWFsbC1uIGZsYWcuCgogICAgQSBidWNrZXQgd2l0aCBuPTMgaXMgbm90IGV2aWRlbmNlIGFuZCBtdXN0IG5vdCBiZSBwcmludGVkIGJlc2lkZSBhIGJ1Y2tldAogICAgd2l0aCBuPTMwMDAgYXMgdGhvdWdoIHRoZSB0d28gY2FycmllZCB0aGUgc2FtZSB3ZWlnaHQuCiAgICAiIiIKICAgIHJvd3MgPSBbXQogICAgZm9yIG5hbWUgaW4gc29ydGVkKHN0cmF0YSk6CiAgICAgICAgaWR4ID0gc3RyYXRhW25hbWVdCiAgICAgICAgdmFscyA9IFt2YWx1ZXNbaV0gZm9yIGkgaW4gaWR4XQogICAgICAgIHJvd3MuYXBwZW5kKHsKICAgICAgICAgICAgInN0cmF0dW0iOiBuYW1lLAogICAgICAgICAgICAibiI6IGxlbihpZHgpLAogICAgICAgICAgICAibWVhbiI6IGZsb2F0KHN1bSh2YWxzKSAvIGxlbih2YWxzKSkgaWYgdmFscyBlbHNlIGZsb2F0KCJuYW4iKSwKICAgICAgICAgICAgInNtYWxsX24iOiBsZW4oaWR4KSA8IG1pbl9uX3dhcm4sCiAgICAgICAgfSkKICAgIHJldHVybiByb3dzCg=='}

for _name, _b64 in FILES.items():
    with open(os.path.join(PKG_ROOT, "keyswm", _name + ".py"), "wb") as fh:
        fh.write(base64.b64decode(_b64))
open(os.path.join(PKG_ROOT, "keyswm", "__init__.py"), "w").write("")
if PKG_ROOT not in sys.path:
    sys.path.insert(0, PKG_ROOT)

import keyswm.protocol as protocol
import keyswm.metrics as metrics
import keyswm.evidence as evidence
import keyswm.tamper as tamper
import keyswm.degradation as degradation
import keyswm.models as models
import keyswm.isic_meta as isic_meta

print("package      :", PKG_ROOT)
print("modules      :", sorted(FILES))
print("protocol     :", len(protocol.MATRIX), "configurations,",
      len(protocol.TAMPER_FAMILIES), "families,", len(protocol.METRICS), "metrics")
print("audit        :", protocol.audit() or "PASS")
print("torch        :", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu          :", torch.cuda.get_device_name(0))

## 2. Configuration

Every value here is frozen before training. Changing one after a run means a new protocol label.

In [ ]:
import torch

SEED = 20261024

CFG = dict(
    seed=SEED,
    size=512,
    key=b"isic2019-synthesis-v1",
    n_total=9000,
    n_smoke=120,           # images used when no dataset is mounted
    split=(0.70, 0.15, 0.15),
    # batch 8 at 512x512 fits a T4 (16 GB) with room to spare. On CPU the
    # activations dominate and a small box will OOM, so shrink it there; this
    # changes only throughput, never a code path.
    batch=8 if torch.cuda.is_available() else 2,
    lr=2e-4,
    epochs=30,             # ~6.3 GPU-h at n=9000; see docs/KAGGLE_RUNBOOK.md
    width=32,              # 1.54M params; width 96 = 13.7M to match PFF-WM
    ablation_n=400,
    pixel_size=512,
)

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

WORK = Path("/kaggle/working") if os.path.isdir("/kaggle/working") else Path("/tmp/keyswm_work")
WORK.mkdir(parents=True, exist_ok=True)
IMG_DIR = WORK / "tampered"; MASK_DIR = WORK / "masks"
IMG_DIR.mkdir(exist_ok=True); MASK_DIR.mkdir(exist_ok=True)
print("device:", DEVICE, "| work:", WORK)
for k, v in CFG.items():
    print(f"  {k:12s} {v}")

## 3. Data

ISIC 2019 if mounted, procedural images otherwise so the notebook still executes end to end.

In [ ]:
# Kaggle mounts datasets under /kaggle/input/<slug>/. Slugs vary, so discover.
CANDIDATES = [
    "/kaggle/input/isic-2019",
    "/kaggle/input/isic2019",
    "/kaggle/input/skin-cancer-malignant-vs-benign",
    "/kaggle/input/isic-2019-skin-cancer",
]
ROOTS = [p for p in CANDIDATES if os.path.isdir(p)]
if not ROOTS:
    found = glob.glob("/kaggle/input/*")
    ROOTS = [p for p in found if os.path.isdir(p)]

FILES = []
for root in ROOTS:
    FILES += glob.glob(os.path.join(root, "**", "*.jpg"), recursive=True)
    FILES += glob.glob(os.path.join(root, "**", "*.JPG"), recursive=True)
FILES = sorted(set(FILES))

META = {}
for root in ROOTS:
    for cand in glob.glob(os.path.join(root, "**", "*Metadata*.csv"), recursive=True):
        META.setdefault("metadata_csv", cand)
    for cand in glob.glob(os.path.join(root, "**", "*GroundTruth*.csv"), recursive=True):
        META.setdefault("groundtruth_csv", cand)
    for cand in glob.glob(os.path.join(root, "**", "*.csv"), recursive=True):
        META.setdefault("any_csv", cand)

META_PATH = META.get("metadata_csv") or META.get("groundtruth_csv") or META.get("any_csv")
METAOBJ = isic_meta.Metadata(META_PATH)
print("metadata   :", META_PATH)
print("schema     :", METAOBJ.schema, "|", METAOBJ.note)
if not METAOBJ.has_diagnosis:
    print()
    print("!! No diagnosis column found. Per-diagnosis stratification will be")
    print("!! skipped and labelled as unavailable. It will NOT be reported as a")
    print("!! single silent 'unknown' bucket.")

MODE = "real" if len(FILES) >= 200 else "smoke"
print("roots      :", ROOTS)
print("jpg files  :", len(FILES))
print("meta       :", META)
print("MODE       :", MODE)
if MODE == "smoke":
    print()
    print("!! No dataset found. Running on PROCEDURAL images so that every cell")
    print("!! executes. Numbers from a smoke run are not results and the receipt")
    print("!! records the mode so they cannot be quoted by accident.")

print("lesion ids :", len(METAOBJ.lesion))

### 3.1 Preprocess: embed, tamper, extract evidence

One pass. The cache holds evidence maps, not images, so the working set is ~150 MB instead of ~7 GB.

In [ ]:
SIZE = CFG["size"]
N = CFG["n_total"] if MODE == "real" else CFG["n_smoke"]
N = min(N, len(FILES)) if MODE == "real" else N

FAMILY_PLAN = ["clean", "B1", "B2", "T1", "T2", "T3", "T4", "T5"]
BENIGN = {"clean", "B1", "B2"}
TAMPERING = {"T1", "T2", "T3", "T4", "T5"}


def load_image(path):
    import cv2
    bgr = cv2.imread(path, cv2.IMREAD_COLOR)
    if bgr is None:
        return None
    return cv2.cvtColor(cv2.resize(bgr, (SIZE, SIZE), interpolation=cv2.INTER_AREA),
                        cv2.COLOR_BGR2RGB)


def procedural(seed):
    import cv2
    rng = np.random.default_rng(seed)
    coarse = rng.integers(50, 200, (16, 16, 3)).astype(np.uint8)
    im = cv2.resize(coarse, (SIZE, SIZE), interpolation=cv2.INTER_CUBIC)
    return np.clip(im + rng.normal(0, 6, im.shape), 0, 255).astype(np.uint8)


# ---- split by lesion, not by image
rng = np.random.default_rng(CFG["seed"])
keys = []
for i in range(N):
    stem = Path(FILES[i]).stem if MODE == "real" else f"synth{i:05d}"
    keys.append(METAOBJ.group_of(stem))
uniq = sorted(set(keys)); rng.shuffle(uniq)
n_tr = int(len(uniq) * CFG["split"][0]); n_va = int(len(uniq) * CFG["split"][1])
assign = {k: ("train" if i < n_tr else ("val" if i < n_tr + n_va else "test"))
          for i, k in enumerate(uniq)}
SPLIT_OF = np.array([assign[k] for k in keys])
print("split:", {s: int((SPLIT_OF == s).sum()) for s in ("train", "val", "test")})

FAMILY_OF = np.array([FAMILY_PLAN[k] for k in rng.permutation(np.arange(N) % len(FAMILY_PLAN))])

import cv2
E_list, D_list, F_list, P_list, B_list = [], [], [], [], []
DX_list = []
t_start = time.time()

for i in range(N):
    rgb = load_image(FILES[i]) if MODE == "real" else procedural(CFG["seed"] + i)
    if rgb is None:
        continue
    family = str(FAMILY_OF[i])
    donor = procedural(CFG["seed"] + i + 100000) if family == "T5" else None

    marked = evidence.protect(rgb, CFG["key"], image_id=i)   # dual carrier
    if family in TAMPERING:
        changed, _blk = tamper.make_pair(marked, family, seed=CFG["seed"] + i, donor=donor)
    elif family == "clean":
        changed = marked
    else:
        changed, _ = tamper.make_pair(marked, family, seed=CFG["seed"] + i)

    # Block mask drives the verdict head; pixel mask drives localisation.
    # Keeping both means the reported pixel IoU is a real pixel IoU.
    pmask = metrics.measured_pixel_mask(marked, changed)
    bmask = metrics.measured_mask(marked, changed)

    rich = evidence.read_full(changed, CFG["key"], image_id=i)
    E_list.append(np.stack([rich[c] for c in evidence.FULL_CHANNELS]).astype(np.float16))
    D_list.append(degradation.estimate(changed))
    F_list.append(FAMILY_PLAN.index(family))
    P_list.append(np.packbits(pmask.reshape(-1)))
    B_list.append(bmask)

    stem_i = Path(FILES[i]).stem if MODE == "real" else f"synth{i:05d}"
    DX_list.append(METAOBJ.dx_of(stem_i))
    cv2.imwrite(str(IMG_DIR / f"{i:06d}.png"), cv2.cvtColor(changed, cv2.COLOR_RGB2BGR))
    np.save(MASK_DIR / f"{i:06d}.npy", bmask)

    if (i + 1) % 500 == 0 or i == N - 1:
        el = time.time() - t_start
        print(f"  {i+1}/{N}  {el:6.1f}s  eta {el/(i+1)*(N-i-1):6.1f}s", flush=True)

E = np.stack(E_list); D = np.stack(D_list); F = np.array(F_list)
P = np.stack(P_list); BM = np.stack(B_list)
DX = np.array(DX_list)
print("evidence", E.shape, E.dtype, "| degrade", D.shape, "| packed pixel truth", P.shape)

In [ ]:
CACHE_F = WORK / "cache_v1.npz"
np.savez_compressed(CACHE_F, E=E, D=D, F=F, P=P, BM=BM, DX=DX, SPLIT=SPLIT_OF,
                    FAMILY_PLAN=np.array(FAMILY_PLAN))
print("cache:", CACHE_F, f"{CACHE_F.stat().st_size/1e6:.1f} MB")

assert E.shape[1] == evidence.N_FULL and E.shape[2:] == (64, 64)
assert np.isfinite(E).all()

# ---- separation check. Averaging E over the whole map is useless: tampered
# blocks are ~1% of the grid. Compare E ON tampered blocks to E on the rest.
print()
print(f"  {'fam':6s} {'n':>4s} {'E_tamp':>9s} {'E_clean':>9s} {'sep':>9s}")
SEP = {}
for fi, fam in enumerate(FAMILY_PLAN):
    m = F == fi
    if m.sum() == 0:
        continue
    gt = BM[m].astype(bool)
    e = E[m][:, 0].astype(np.float32)
    if gt.sum() == 0:
        print(f"  {fam:6s} {int(m.sum()):4d} {'-':>9s} {e.mean():9.4f} {'-':>9s}")
    else:
        SEP[fam] = float(e[gt].mean() - e[~gt].mean())
        print(f"  {fam:6s} {int(m.sum()):4d} {e[gt].mean():9.4f} {e[~gt].mean():9.4f} {SEP[fam]:9.4f}")
_bad = [f for f, v in SEP.items() if v < 0.10]
print("families whose keyed evidence does not separate (sep < 0.10):", _bad or "none")
print("B1/B2 raise E EVERYWHERE -- routine handling destroying the mark. That is")
print("precisely why the system needs a CANNOT_SAY verdict.")

In [ ]:
NCLS = 3
CLS_OF = np.full(len(F), 2, np.int64)          # CANNOT_SAY
CLS_OF[F == 0] = 0                             # clean  -> INTACT
for _fi, _fam in enumerate(FAMILY_PLAN):
    if _fam in ("T1", "T2", "T3", "T4", "T5"):
        CLS_OF[F == _fi] = 1                   # tampered
print("class balance:", {n: int((CLS_OF == c).sum())
                        for c, n in enumerate(["INTACT", "TAMPERED", "CANNOT_SAY"])})


# Loads the tampered image and both masks on demand. Images stay on disk: 9000
# frames at 512x512 in uint8 is 7 GB and would not survive beside the model.
class FullSet(torch.utils.data.Dataset):
    def __init__(self, idx, train=False):
        self.idx = idx; self.train = train
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, k):
        j = int(self.idx[k])
        import cv2
        img = cv2.imread(str(IMG_DIR / f"{j:06d}.png"), cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        pm = np.unpackbits(P[j])[:SIZE*SIZE].reshape(SIZE, SIZE).astype(np.float32)
        bm = np.load(MASK_DIR / f"{j:06d}.npy").astype(np.float32)
        return (torch.from_numpy(img.transpose(2, 0, 1).copy()).float() / 255.0,
                torch.from_numpy(E[j].astype(np.float32)),
                torch.from_numpy(D[j]),
                torch.from_numpy(bm).unsqueeze(0),
                torch.from_numpy(pm).unsqueeze(0),
                int(CLS_OF[j]))


tr_idx = np.where(SPLIT_OF == "train")[0]
va_idx = np.where(SPLIT_OF == "val")[0]
te_idx = np.where(SPLIT_OF == "test")[0]
tr = torch.utils.data.DataLoader(FullSet(tr_idx), batch_size=CFG["batch"], shuffle=True, drop_last=True, num_workers=2)
va = torch.utils.data.DataLoader(FullSet(va_idx), batch_size=CFG["batch"], shuffle=False, num_workers=2)
te = torch.utils.data.DataLoader(FullSet(te_idx), batch_size=CFG["batch"], shuffle=False, num_workers=2)
print("train", len(tr_idx), "| val", len(va_idx), "| test", len(te_idx))
print("TEST IS NOT TOUCHED UNTIL THE FINAL CELL.")

## 4. Training

Rows 3 and 4 of the table. The autoencoder sees **only authentic images**; the fusion model sees everything.

In [ ]:
# ---- C1: one-class autoencoder, authentic images only (row 3)
authentic = np.where((SPLIT_OF == "train") & np.isin(F, [0, 1, 2]))[0]
print("authentic training images:", len(authentic))


class EvOnly(torch.utils.data.Dataset):
    def __init__(self, idx):
        self.idx = idx
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, k):
        return torch.from_numpy(E[int(self.idx[k])].astype(np.float32))


ae_loader = torch.utils.data.DataLoader(EvOnly(authentic), batch_size=64, shuffle=True, drop_last=True)
ae = models.OneClassAE(evidence.N_FULL, width=max(8, CFG["width"] // 2)).to(DEVICE)
opt = torch.optim.AdamW(ae.parameters(), lr=1e-3)
t0 = time.time()
for ep in range(15):
    ae.train(); tot = 0.0
    for ev in ae_loader:
        ev = ev.to(DEVICE)
        rec, _ = ae(ev)
        loss = torch.nn.functional.mse_loss(rec, ev)
        opt.zero_grad(); loss.backward(); opt.step()
        tot += float(loss.detach()) * ev.shape[0]
    if ep % 5 == 0 or ep == 14:
        print(f"  epoch {ep:2d}  mse {tot/max(len(authentic),1):.6f}  {time.time()-t0:.1f}s")
print("AE trained on clean and benignly processed images only.")

In [ ]:
# ---- C8: the assembled system
sal_net = models.SaliencyNet().to(DEVICE)
model = models.PixelRefineNet(n_ev=evidence.N_FULL,
                              width=CFG["width"]).to(DEVICE)
assert evidence.N_FULL == len(evidence.FULL_CHANNELS) == 13
params = list(model.parameters()) + list(sal_net.parameters())
opt = torch.optim.AdamW(params, lr=CFG["lr"], weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG["epochs"])
best = {"val_pixel_iou": -1.0, "epoch": -1}


def predict(model, sal_net, loader):
    model.eval(); sal_net.eval()
    pix, blk, cls = [], [], []
    with torch.no_grad():
        for img, ev, d, _bm, _pm, _c in loader:
            img, ev, d = img.to(DEVICE), ev.to(DEVICE), d.to(DEVICE)
            s = sal_net(img)
            out = model(ev, d, s, img)
            pix.append(torch.sigmoid(out["pixel_logits"]).cpu().numpy()[:, 0])
            blk.append(torch.sigmoid(out["block_logits"]).cpu().numpy()[:, 0])
            cls.append(out["cls"].cpu().numpy())
    return (np.concatenate(pix), np.concatenate(blk), np.concatenate(cls))


def val_scores(model, sal_net, loader, idx):
    pix, blk, _ = predict(model, sal_net, loader)
    pix_iou = []; blk_iou = []; ceil_ = []
    for k, j in enumerate(idx):
        pm = np.unpackbits(P[j])[:SIZE*SIZE].reshape(SIZE, SIZE).astype(bool)
        bm = np.load(MASK_DIR / f"{j:06d}.npy").astype(bool)
        pix_iou.append(metrics.iou(pix[k] > 0.5, pm))
        blk_iou.append(metrics.iou(blk[k] > 0.5, bm))
        ceil_.append(metrics.block_ceiling(pm))
    return float(np.mean(pix_iou)), float(np.mean(blk_iou)), float(np.mean(ceil_))


t0 = time.time()
for ep in range(CFG["epochs"]):
    model.train(); sal_net.train(); tot = 0.0; nseen = 0; parts = {}
    for img, ev, d, bm, pm, c in tr:
        img, ev, d = img.to(DEVICE), ev.to(DEVICE), d.to(DEVICE)
        bm, pm, c = bm.to(DEVICE), pm.to(DEVICE), c.to(DEVICE)
        s = sal_net(img)
        out = model(ev, d, s, img)
        # Separate imbalance weights per resolution: a few percent of blocks are
        # positive, under one percent of pixels. One shared weight drove the
        # pixel head to predict tampering everywhere.
        loss, comp = models.total_loss(out, bm, pm, c,
                                       pos_weight=CFG.get("pos_weight", 12.0),
                                       pos_weight_pixel=CFG.get("pos_weight_pixel", 4.0))
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(params, 5.0)
        opt.step()
        tot += float(loss.detach()) * img.shape[0]; nseen += img.shape[0]
        for k, v in comp.items():
            parts[k] = parts.get(k, 0.0) + v * img.shape[0]
    sched.step()
    if ep % 5 == 0 or ep == CFG["epochs"] - 1:
        vp, vb, vc = val_scores(model, sal_net, va, va_idx)
        flag = ""
        if vp > best["val_pixel_iou"]:
            best = {"val_pixel_iou": vp, "epoch": ep}
            torch.save({"model": model.state_dict(), "sal": sal_net.state_dict()}, WORK / "best.pt")
            flag = " *"
        print(f"  ep {ep:3d} loss {tot/nseen:.4f} "
              f"(blk {parts['block']/nseen:.3f} px {parts['pixel']/nseen:.3f} "
              f"con {parts['consistency']/nseen:.3f} cls {parts['cls']/nseen:.3f})  "
              f"val px-IoU {vp:.4f} blk-IoU {vb:.4f} | {time.time()-t0:.0f}s{flag}")

print("best val pixel IoU", round(best["val_pixel_iou"], 4), "at epoch", best["epoch"])
print("selection used VALIDATION only. Test remains closed.")

## 5. Ablations

C0 is the control. C9/C10 ask what the key costs and whether the network has learned it.

In [ ]:
print("=== C0: classical floor, two versions ===")
print()
print("C0a raw threshold, no abstention. NOT our classical system: it flags")
print("    every block of a JPEG-75 image, because routine handling destroys the")
print("    mark everywhere. Scoring this as the baseline inflates the learned half.")
raw_blk, raw_px = [], []
for j in va_idx:
    pm = np.unpackbits(P[j])[:SIZE*SIZE].reshape(SIZE, SIZE).astype(bool)
    bm = np.load(MASK_DIR / f"{j:06d}.npy").astype(bool)
    raw_blk.append(metrics.iou(E[j][0].astype(np.float32) > 0.5, bm))
    raw_px.append(metrics.iou(metrics.upsample_to(E[j][0].astype(np.float32) > 0.5, (SIZE, SIZE)), pm))
print(f"    val block IoU {np.mean(raw_blk):.4f} | pixel IoU {np.mean(raw_px):.4f}")
print()
print("C0b threshold + the abstention rule the classical system actually has.")
print("    Compare C8 against this at equal coverage, never against C0a.")
for floor in (0.60, 0.70, 0.85):
    keep = [j for j in va_idx if E[j][1].astype(np.float32).mean() >= floor]
    if not keep:
        print(f"    survival floor {floor:.2f}  coverage 0.000  (nothing survives)")
        continue
    ious = []
    for j in keep:
        bm = np.load(MASK_DIR / f"{j:06d}.npy").astype(bool)
        ious.append(metrics.iou(E[j][0].astype(np.float32) > 0.5, bm))
    print(f"    survival floor {floor:.2f}  coverage {len(keep)/len(va_idx):.3f}  "
          f"block IoU(covered) {np.mean(ious):.4f}")

print()
print("=== C9 / C10: the key ===")
print("Recomputing evidence under a different key on held-out images.")
import cv2 as _cv
ladder = []
for k, j in enumerate(te_idx[:200]):
    im = _cv.cvtColor(_cv.imread(str(IMG_DIR / f"{j:06d}.png")), _cv.COLOR_BGR2RGB)
    a = evidence.read(im, CFG["key"], image_id=int(j))["E"]
    b = evidence.read(im, b"isic2019-synthesis-v1-WRONG", image_id=int(j))["E"]
    ladder.append((float(a.mean()), float(b.mean())))
if ladder:
    la, lb = np.array(ladder).mean(axis=0)
    print(f"  same key  mean E {la:.4f}")
    print(f"  wrong key mean E {lb:.4f}")
    print("  A wrong key must look like noise. If it looked like a weaker tamper")
    print("  signal the mark would not be keyed at all.")

## 6. Test set — opened once

Selection happened on validation. This cell runs exactly once per genuine experiment.

In [ ]:
# ================= TEST SET — OPENED ONCE =================
ck = torch.load(WORK / "best.pt", map_location=DEVICE)
model.load_state_dict(ck["model"]); sal_net.load_state_dict(ck["sal"])

pix, blk, cls_logits = predict(model, sal_net, te)
SIZE = CFG["size"]

rows = []
for k, j in enumerate(te_idx):
    fam = FAMILY_PLAN[F[j]]
    pm_true = np.unpackbits(P[j])[:SIZE*SIZE].reshape(SIZE, SIZE).astype(bool)
    bm_true = np.load(MASK_DIR / f"{j:06d}.npy").astype(bool)
    rows.append({
        "kind": "benign" if fam in BENIGN else "tamper",
        "family": fam,
        "pred": blk[k] > 0.5,
        "gt": bm_true,
        "pred_pixels": pix[k] > 0.5,
        "gt_pixels": pm_true,
    })

print("TEST RESULTS")
print()
print("A note on the two IoU columns. The block number is what the verdict layer")
print("decides on. The pixel number is what a localisation paper reports. They are")
print("not interchangeable, and the pixel number is bounded from above by the")
print("capability of the output grid -- see the ceiling column below.")
print()
det = metrics.detection_accuracy(rows)
print(f"  n images              {len(rows)}")
print(f"  detection accuracy    {det['detection_acc']:.4f}   (any-block-flagged; n tamper {det['n_tamper']})")
print(f"  false positives       {det['false_positive_rate']:.4f}   (n benign {det['n_benign']})")
print(f"  benefit @ pixel IoU .5 {metrics.benefit_at(rows, 0.5):.4f}")

strat = metrics.stratified_iou(rows)
print()
print("stratified by tampered area -- the only IoU table that means anything:")
print(f"  {'bucket':22s} {'n':>4s} {'blk IoU':>8s} {'px IoU':>8s} {'ceiling':>8s}")
for b, v in strat.items():
    print(f"  {b:22s} {v['n']:4d} {v['iou_block']:8.4f} "
          f"{v.get('iou_pixel', float('nan')):8.4f} {v.get('ceiling', float('nan')):8.4f}")

print()
print("per family:")
for fam in FAMILY_PLAN:
    m = [r for r in rows if r["family"] == fam]
    if not m:
        continue
    print(f"  {fam:5s} n={len(m):4d}  blk-IoU={np.mean([metrics.iou(r['pred'], r['gt']) for r in m]):.4f}"
          f"  px-IoU={np.mean([metrics.iou(r['pred_pixels'], r['gt_pixels']) for r in m]):.4f}"
          f"  flagged={np.mean([r['pred'].any() for r in m]):.3f}")

# ---- per-diagnosis strata. This is the dimension an ISIC-only study can add
# that a multi-corpus headline cannot: does localisation survive across lesion
# types, or does it only work on the textured ones?
print()
if METAOBJ.has_diagnosis:
    print("by ISIC diagnosis (tampered images only):")
    items = [{"dx": DX[j]} for j in te_idx]
    vals = [metrics.iou(r["pred"], r["gt"]) if r["kind"] == "tamper" else float("nan")
            for r in rows]
    strata = isic_meta.stratify_by(items, "dx")
    print(f"  {'dx':6s} {'n':>5s} {'blk IoU':>8s}  flag")
    for row in isic_meta.report(strata, [0.0 if v != v else v for v in vals]):
        star = "  small n" if row["small_n"] else ""
        print(f"  {row['stratum']:6s} {row['n']:5d} {row['mean']:8.4f}{star}")
    print("  A bucket with n<10 is not evidence and is marked as such.")
else:
    print("by ISIC diagnosis: UNAVAILABLE - the corpus carries no diagnosis column.")
    print("  Reported as unavailable rather than as one silent 'unknown' bucket.")

# ---- abstention: does answering less mean answering better?
print()
print("selective risk against coverage (tampered images, block level):")
confs, correct = [], []
for k, j in enumerate(te_idx):
    if rows[k]["kind"] != "tamper":
        continue
    p_ = blk[k]
    conf = np.abs(p_ - 0.5) * 2.0          # confidence in whichever side it chose
    confs.append(conf.ravel())
    correct.append((p_ > 0.5).ravel() == rows[k]["gt"].ravel())
if confs:
    confs = np.concatenate(confs); correct = np.concatenate(correct)
    for c in (1.0, 0.9, 0.75, 0.5, 0.25):
        k_ = max(1, int(round(c * confs.size)))
        sel = np.argsort(-confs)[:k_]
        print(f"  coverage {c:.2f}  risk {1.0 - correct[sel].mean():.4f}")
    print(f"  AURC {metrics.aurc(confs, correct):.4f}  (raw error {1.0 - correct.mean():.4f})")
    print("  If AURC is close to the raw error rate, the confidence carries no")
    print("  information and the abstention is decoration.")

CLS_NAMES = ["INTACT", "TAMPERED", "CANNOT_SAY"]
cls_pred = cls_logits.argmax(axis=1); cls_true = CLS_OF[te_idx]
cls_acc = float((cls_pred == cls_true).mean())
print()
print(f"three-way verdict accuracy {cls_acc:.4f}")
for a, name in enumerate(CLS_NAMES):
    m = cls_true == a
    if m.sum():
        print(f"  true {name:11s} n={int(m.sum()):4d} recall={float((cls_pred[m]==a).mean()):.3f}")

conf = np.zeros((3, 3), int)
for t_, p_ in zip(cls_true, cls_pred):
    conf[t_, p_] += 1

receipt = {
    "protocol": "isic2019-synthesis-v1",
    "mode": MODE,
    "architecture": "PixelRefineNet + SaliencyNet + keyed evidence (10ch)",
    "config": {k: (v if not isinstance(v, bytes) else v.decode()) for k, v in CFG.items()},
    "n_images": int(len(rows)) + int(len(tr_idx)) + int(len(va_idx)),
    "splits": {s: int((SPLIT_OF == s).sum()) for s in ("train", "val", "test")},
    "detection_definition": "an image is detected if any block is flagged; FPR is the fraction of benign images with any flag",
    "detection": det,
    "stratified": strat,
    "three_way_accuracy": cls_acc,
    "confusion": conf.tolist(),
    "comparable_to_published": False,
    "why_not_comparable": (
        "The 0.832 IoU bar is LiTS + lesion insertion with pixel ground truth. This run is "
        "ISIC 2019 with an eight-family protocol. Even at matched corpus the pixel IoU is "
        "bounded by the tamper size; report the ceiling column beside it."
    ),
    "claimable": MODE == "real",
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}
if MODE != "real":
    receipt["results"] = {"withheld": "smoke mode - no dataset, numbers are not results"}
OUT_F = WORK / "receipt.json"
OUT_F.write_text(json.dumps(receipt, indent=2))
print()
print("receipt:", OUT_F, "| claimable:", receipt["claimable"])

## What to do with these numbers

1. **If `claimable` is false**, the run found no dataset. Do not quote anything.
2. **ISIC 2019 numbers are not comparable to the 0.832 IoU bar.** That bar is
   LiTS with lesion insertion. A side-by-side table must carry the protocol
   label or it is misleading.
3. **The comparison that *is* fair** is inside this notebook: C0 vs C8 on the
   same split. The gap between them is the contribution of the learned half, and
   it is measurable without any external dataset.
4. **To clear the published bar on its own terms**, the LiTS run is required —
   same corpus, same tamper family, same metric. That is the next notebook.